# DNTC strict OCR/PDF/NLP pipeline v16 R016 spelling + false-period merge guard

This version keeps v13/v15 page-range and terminal-punctuation export, then adds mechanism-based R016 spelling/merge/noise handling.

In [1]:

# ============================================================
# 0. CONFIG - chỉnh ở đây rồi Run All
# ============================================================
from pathlib import Path
import os

# Google Drive folder/file URLs. Có thể thay bằng Drive folder của bạn.
DRIVE_URLS = [
    "https://drive.google.com/drive/folders/1QZzyaozPRLcm5Y2nmUUbigyVFnsX_tkW",
]

# Nếu không muốn download Drive, để DRIVE_URLS = [] và notebook đọc từ /kaggle/input.
ALLOW_KAGGLE_INPUT_FALLBACK = True
LOCAL_INPUT_DIRS = [Path("/kaggle/input"), Path("/mnt/data"), Path(".")]

# Output đúng yêu cầu.
OUTPUT_DIR = Path("/kaggle/working/dntc_auto") if Path("/kaggle/working").exists() else Path("./dntc_auto")
RAW_DIR = OUTPUT_DIR / "raw_drive"
FINAL_DIR = OUTPUT_DIR / "final"
TEXT_DIR = FINAL_DIR / "texts"
AUDIT_DIR = OUTPUT_DIR / "audit"
CACHE_DIR = OUTPUT_DIR / "cache"
PKG_DIR = OUTPUT_DIR / "packages"

# Run modes.
FAST_TEST_MODE = False           # True: chạy thử nhanh một số trang.
FAST_TEST_MAX_PDFS = 2
FAST_TEST_MAX_PAGES_TOTAL = 50
FULL_RUN_MODE = not FAST_TEST_MODE
PIPELINE_VERSION = "dntc_auto_v16_r016_spelling_merge_clean"
MAX_PAGES_PER_PDF = None         # None = all pages; FAST_TEST_MODE sẽ override bằng tổng 50 pages.

# Required page filtering config.
CONTENT_START_MODE = "auto"      # "auto" hoặc "none".
KEEP_TITLE_PAGES = False
DROP_LIBRARY_PAGES = True
DROP_BLANK_PAGES = True
DROP_PATTERNED_PAGES = True
DROP_LOW_CONF_OCR_PAGES = False
DROP_FRONT_MATTER_BEFORE_CONTENT = True
DROP_BACKMATTER_PAGES = True          # drop publisher ads, print info, table-of-contents/end matter pages
STRICT_SENTENCE_ONLY = True         # sentence csv should not contain line fragments

# OCR controls. Không dùng PaddleOCR mặc định vì dependency nặng/dễ lỗi trên Kaggle.
ENABLE_TESSERACT = True
OCR_LANG = "vie+eng"
PAGE_OCR_DPI = 260
FAST_FRONTMATTER_OCR_DPI = 150
LINE_REOCR_ZOOM = 4.0
LINE_REOCR_PAD_PT = 7.0
MAX_LINE_REOCR_PER_PDF = 250

# Quality thresholds. Tăng nếu muốn ít final hơn nhưng sạch hơn.
MIN_SELECTED_PAGE_QUALITY = 30.0
MIN_OCR_PAGE_QUALITY = 38.0
MIN_TEXT_LAYER_GOOD_QUALITY = 55.0
MIN_KEEP_LINE_QUALITY = 32.0
MIN_KEEP_SENTENCE_QUALITY = 32.0
MIN_OCR_CONF_LINE = 35.0
MAX_LIBRARY_NOISE_SCORE = 0.18
MAX_WEIRD_CHAR_RATIO = 0.065
MIN_VIET_RATIO_FOR_LONG_TEXT = 0.10

# Content detection.
AUTO_CONTENT_SCAN_PAGES = 35
MIN_CONTENT_WORDS_ON_START_PAGE = 35
MIN_CONTENT_LINES_ON_START_PAGE = 4

# Exports.
ZIP_OUTPUT = True
WRITE_DEBUG_PAGE_THUMBNAILS = False
DEBUG_THUMBNAIL_MAX = 80



# Verified physical PDF page ranges supplied by project owner. These are 1-indexed and inclusive.
# Pages outside these ranges are never exported to final CSV/text; they are audited in excluded_pages_by_page_range.csv.
PAGE_RANGES = {
    "01.pdf": [(21, 127)],
    "05.pdf": [(15, 140)],
    "07_08.pdf": [(23, 90), (95, 206)],
    "09.pdf": [(23, 139)],
    "10_11.pdf": [(13, 129)],
    "12.pdf": [(9, 112)],
    "13.pdf": [(7, 120)],
    "14_15.pdf": [(7, 168)],
    "16_17.pdf": [(14, 128), (138, 293)],
    "q2_3_4.pdf": [(13, 499)],
    "q6.pdf": [(3, 525)],
}
# Backward-compatible start metadata for code that still asks for content_start_page.
PAGE_STARTS = {name: ranges[0][0] for name, ranges in PAGE_RANGES.items()}
PAGE_ENDS = {name: ranges[-1][1] for name, ranges in PAGE_RANGES.items()}

for d in [OUTPUT_DIR, RAW_DIR, FINAL_DIR, TEXT_DIR, AUDIT_DIR, CACHE_DIR, PKG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("FAST_TEST_MODE:", FAST_TEST_MODE)
print("FULL_RUN_MODE:", FULL_RUN_MODE)



OUTPUT_DIR: /kaggle/working/dntc_auto
FAST_TEST_MODE: False
FULL_RUN_MODE: True


In [2]:

# ============================================================
# 1. Install dependencies - Kaggle Run All friendly
# ============================================================
import sys, subprocess, shutil, importlib.util


def pip_install(*pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *pkgs]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=False)

required_modules = {
    "fitz": ["pymupdf"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "PIL": ["pillow"],
    "tqdm": ["tqdm"],
    "gdown": ["gdown"],
    "pytesseract": ["pytesseract"],
}
for mod, pkgs in required_modules.items():
    if importlib.util.find_spec(mod) is None:
        pip_install(*pkgs)

# Tesseract binary + Vietnamese and English traineddata.
if ENABLE_TESSERACT:
    missing_binary = shutil.which("tesseract") is None
    if missing_binary:
        print("Installing tesseract system packages...")
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-vie", "tesseract-ocr-eng"], check=False)
    else:
        # Ensure vie/eng packages exist. Safe if already installed.
        try:
            langs = subprocess.check_output(["tesseract", "--list-langs"], text=True, stderr=subprocess.STDOUT)
        except Exception:
            langs = ""
        if "vie" not in langs or "eng" not in langs:
            print("Installing tesseract Vietnamese/English traineddata...")
            subprocess.run(["apt-get", "update", "-qq"], check=False)
            subprocess.run(["apt-get", "install", "-y", "tesseract-ocr-vie", "tesseract-ocr-eng"], check=False)

print("tesseract:", shutil.which("tesseract"))


$ /usr/bin/python3 -m pip install -q --no-cache-dir pymupdf
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 299.1 MB/s eta 0:00:00
Installing tesseract Vietnamese/English traineddata...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng set to manually installed.
The following NEW packages will be installed:
  tesseract-ocr-vie
0 upgraded, 1 newly installed, 0 to remove and 100 not upgraded.
Need to get 417 kB of archives.
After this operation, 546 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-vie all 1:4.00~git30-7274cfa-1.1 [417 kB]
Fetched 417 kB in 0s (5,653 kB/s)
Selecting previously unselected package tesseract-ocr-vie.
(Reading database ... 121026 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-vie_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-vie (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-vie (1:4.00~git30-7274cfa-1.1) ...
tesseract: /usr/bin/tesseract


In [3]:

# ============================================================
# 2. Imports and shared constants
# ============================================================
import os, re, math, json, time, zipfile, hashlib, shutil, subprocess, sys, unicodedata, itertools
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass

import numpy as np
import pandas as pd
import fitz
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
try:
    import gdown
except Exception:
    gdown = None
    print("gdown not available; Drive download will be skipped unless installed in dependency cell.")

try:
    import pytesseract
    from pytesseract import Output
except Exception as e:
    pytesseract = None
    Output = None
    print("pytesseract not available:", repr(e))

VIET_CHARS = set(
    "ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩị"
    "óòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ"
    "ĂÂĐÊÔƠƯÁÀẢÃẠẮẰẲẴẶẤẦẨẪẬÉÈẺẼẸẾỀỂỄỆÍÌỈĨỊ"
    "ÓÒỎÕỌỐỒỔỖỘỚỜỞỠỢÚÙỦŨỤỨỪỬỮỰÝỲỶỸỴ"
)
VIET_VOWELS = set("aeiouyAEIOUYàáảãạằắẳẵặầấẩẫậèéẻẽẹềếểễệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵăâêôơưĂÂÊÔƠƯ")
LETTERS_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]")
WORDS_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]+")
DIGIT_RE = re.compile(r"\d")
CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\ufffe\uffff\u0001]")
CJK_RE = re.compile(r"[\u3400-\u9fff]")
NON_VIET_SCRIPT_RE = re.compile(r"[Α-ωА-яЁё]")
WEIRD_RE = re.compile(r"[^\w\sÀ-ỹ.,;:!?(){}\[\]\"'“”‘’/\-–—%°+&<>«»·•*]", re.UNICODE)

COMMON_VI_WORDS = set("""
của và là có không trong ngoài năm đời phủ huyện tỉnh châu xã thôn làng tổng phường sách dân người nước ta
sông núi biển cửa đông tây nam bắc phía giáp cách dặm linh thành đặt đổi thuộc đời nhà triều vua quan quân
quyền quyển đại nam nhất thống chí kinh sư phần dã dựng đặt diễn cách hình thế khí hậu phong tục thành trì
trường học hộ khẩu thuế ruộng núi sông cổ tích từ miếu đền chùa miếu đàn lăng mộ đồn lũy cửa biển cầu đường
minh mệnh gia long tự đức thiệu trị đồng khánh duy tân hiến tông duệ tông thế tổ thánh tông cao hoàng đế
chiêm thành cao mên xiêm la thanh nghệ an thanh hóa quảng bình quảng trị quảng nam quảng ngãi bình định
phú yên khánh hòa bình thuận hà tiên an giang biên hòa gia định định tường vĩnh long hà nội hải dương
""".split())

COMMON_UNACCENTED_VI_WORDS = set("""
cua va la co khong trong ngoai nam doi phu huyen tinh chau xa thon lang tong phuong sach dan nguoi nuoc ta
song nui bien cua dong tay nam bac phia giap cach dam linh thanh dat doi thuoc nha trieu vua quan quyen quyen
dai nam nhat thong chi kinh su phan da dung dat dien cach hinh the khi hau phong tuc thanh tri truong hoc ho khau
thue ruong co tich tu mieu den chua minh menh gia long tu duc thieu tri dong khanh duy tan hien tong due tong
the to thanh tong chiem thanh cao men xiem la quang binh quang tri quang nam quang ngai binh dinh phu yen
khanh hoa binh thuan ha tien an giang bien hoa gia dinh dinh tuong vinh long ha noi hai duong
""".split())

DOMAIN_HEADINGS = [
    "ĐẠI NAM NHẤT THỐNG CHÍ", "LỜI NÓI ĐẦU", "BÀI TỰ", "PHÀM LỆ", "DỰNG ĐẶT VÀ DIÊN CÁCH", "PHẦN DÃ",
    "HÌNH THẾ", "KHÍ HẬU", "PHONG TỤC", "THÀNH TRÌ", "TRƯỜNG HỌC", "HỘ KHẨU", "THUẾ RUỘNG", "NÚI SÔNG",
    "SÔNG NGÒI", "CỔ TÍCH", "ĐỀN MIẾU", "TỪ MIẾU", "LĂNG MỘ", "ĐỒN LŨY", "CẦU ĐƯỜNG", "CHỢ QUÁN",
]

DOMAIN_TOKENS = set(" ".join(DOMAIN_HEADINGS).lower().split()) | COMMON_VI_WORDS

LIBRARY_NOISE_RE = re.compile(
    r"\b(library|libraries|university|barcode|digitized|google|riverside|michigan|wisconsin|madison|"
    r"east\s+west|center\s+library|memorial|archive|scan|call\s*number|state\s+street|copyright|"
    r"vol\.?|volume|DS\s*\d|G27|EAST WEST CENTER|THE UNIVERSITY)\b",
    re.I,
)
BARCODE_CALLNO_RE = re.compile(r"^(?:[A-Z]{0,3}\s*)?(?:DS|G|B|HV|VIET)?\s*[A-Z0-9.\-/ ]{2,18}$", re.I)
TITLE_FRONTMATTER_RE = re.compile(
    r"\b(văn\s*-?\s*hóa|tùng\s*-?\s*thư|dịch\s*-?\s*giả|soạn\s*-?\s*giả|xuất\s*-?\s*bản|"
    r"bộ\s+quốc\s*-?\s*gia|bộ\s+văn\s*-?\s*hóa|nhà\s+xuất\s+bản|thuận\s+hóa|tập\s+số|"
    r"tái\s+bản|người\s+dịch|người\s+hiệu\s+đính)\b",
    re.I,
)

BACKMATTER_NOISE_RE = re.compile(
    r"\b(Những\s+tập\s+VĂN\s*HÓA|Có\s+bán\s+khắp|Tổng\s*-?\s*phát\s*-?\s*hành|"
    r"NHA\s+VĂN\s*[-–]?\s*HÓA\s*\(\s*266|Đường\s+Công\s+Lý|In\s+50\s+cuốn|In\s+1000\s+cuốn|"
    r"Số\s+đăng\s+kí\s+KHXB|Quyết\s+định\s+xuất\s+bản|Xưởng\s+in\s+Ban|MỤC\s+LỤC|MUC\s+LUC)\b",
    re.I,
)

MOJIBAKE_RE = re.compile(r"[ÑñÐðÖ¿§€]")

NOISY_TESSERACT_TOKEN_RE = re.compile(
    r"\b(NÑam|NÑinh|giấp|s15|7oa|l3ình|trừ-tjch|khéng|lèo|"
    r"NLIAT|TERIOCNG|ALAIN|TA\s+T|xuat\s+bản|phat\s+xu|ngu[eé]n|huy[eé]n\s+nav)\b",
    re.I,
)

CONTENT_START_RE = re.compile(
    r"\b(lời\s+nói\s+đầu|bài\s+tự|phàm\s+lệ|quyền\s+[ivxlcdm0-9]+|tỉnh\s+[A-ZÀ-ỸĐ]|"
    r"dựng\s+đặt|diên\s+cách|diễn\s+cách|phần\s+dã|hình\s+thế|phong\s+tục|đông\s+tây\s+cách\s+nhau)\b",
    re.I,
)

CLEAR_JUNK_RE = re.compile(
    r"(OPOC|erererore|Fel\s+Fat|Seer\s+tit|mADS|OKOK|OROR|RORO|Peete|Sarine|"
    r"^[\s:;,.`´‘’\"\\/\-–—_~|°*+={}\[\]()<>]{1,14}$|^\s*5\s*[-–—]\s*$)",
    re.I,
)
REPEATED_FRAGMENT_RE = re.compile(r"([A-Za-z]{2,5})\1{2,}")

print("imports ok")


imports ok


In [4]:

# ============================================================
# 3. Download / discover PDFs
# ============================================================

def download_drive_url(url: str, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    if not url or "PASTE" in url:
        return
    print("Downloading from Drive:", url)
    try:
        if "/folders/" in url:
            gdown.download_folder(url, output=str(out_dir), quiet=False, use_cookies=False, remaining_ok=True)
        else:
            gdown.download(url, output=str(out_dir), quiet=False, fuzzy=True)
    except TypeError:
        # Older gdown fallback.
        if "/folders/" in url:
            subprocess.run([sys.executable, "-m", "gdown", "--folder", url, "-O", str(out_dir)], check=False)
        else:
            subprocess.run([sys.executable, "-m", "gdown", url, "-O", str(out_dir)], check=False)
    except Exception as e:
        print("Drive download failed:", repr(e))

RAW_DIR.mkdir(parents=True, exist_ok=True)
if DRIVE_URLS:
    existing = list(RAW_DIR.rglob("*.pdf"))
    if not existing:
        for url in DRIVE_URLS:
            download_drive_url(url, RAW_DIR)

pdf_candidates = []
pdf_candidates += list(RAW_DIR.rglob("*.pdf"))
if ALLOW_KAGGLE_INPUT_FALLBACK:
    for d in LOCAL_INPUT_DIRS:
        if d.exists():
            pdf_candidates += list(d.rglob("*.pdf"))

seen = set()
pdf_paths = []
for p in sorted(pdf_candidates):
    try:
        key = (p.name.lower(), p.stat().st_size)
        if key in seen:
            continue
        seen.add(key)
        pdf_paths.append(p)
    except Exception:
        pass

if FAST_TEST_MODE:
    pdf_paths = pdf_paths[:FAST_TEST_MAX_PDFS]

if not pdf_paths:
    raise FileNotFoundError("No PDFs found. Check DRIVE_URLS, Internet=On, or /kaggle/input.")

print("PDF count:", len(pdf_paths))
for p in pdf_paths[:60]:
    print("-", p)
if len(pdf_paths) > 60:
    print("...", len(pdf_paths) - 60, "more")


Retrieving folder contents


Processing file 1wukxPsU1Ty_vWeGSk6CdvEZwLAcjdnqh 01.pdf
Processing file 1NLyeEqQCMW5-fnz797Oxx7R2f6X3o170 05.pdf
Processing file 1unk05e2iCFeaxSVupNam5DcDD5brtF2E 07_08.pdf
Processing file 16FEI4ljzrewz5bvt8tMAI8xQU4Ecut9h 09.pdf
Processing file 1C-mYRYagVEcEOyKCS93I0ZHaBpUw8_Wr 10_11.pdf
Processing file 1ROEJnPaAspyXW3k81b4zPjN6TBazDrt- 12.pdf
Processing file 126X-S8gfQHznvYhifiITln-xxy_h_r61 13.pdf
Processing file 1OK-e2fx66HxCDOF1ZoBpdG0qhy12-8qL 14_15.pdf
Processing file 1tFvtR94BGd1eIGI6jQOnR-z_nPqYbDFZ 16_17.pdf
Processing file 12KF-95e9GN1edzQxpTl3OfwZcntZgkV_ q2_3_4.pdf
Processing file 1rQB3SI4qvl7iJPnUq4Bo9en4db-7vaRv q6.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1wukxPsU1Ty_vWeGSk6CdvEZwLAcjdnqh
To: /kaggle/working/dntc_auto/raw_drive/01.pdf
100%|██████████| 6.92M/6.92M [00:00<00:00, 39.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1NLyeEqQCMW5-fnz797Oxx7R2f6X3o170
To: /kaggle/working/dntc_auto/raw_drive/05.pdf
100%|██████████| 23.7M/23.7M [00:00<00:00, 56.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1unk05e2iCFeaxSVupNam5DcDD5brtF2E
To: /kaggle/working/dntc_auto/raw_drive/07_08.pdf
100%|██████████| 9.98M/9.98M [00:00<00:00, 93.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=16FEI4ljzrewz5bvt8tMAI8xQU4Ecut9h
To: /kaggle/working/dntc_auto/raw_drive/09.pdf
100%|██████████| 8.60M/8.60M [00:00<00:00, 39.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1C-mYRYagVEcEOyKCS93I0ZHaBpUw8_Wr
To: /kaggle/working/dntc_auto/raw_drive/10_11.pdf
100%|███████

PDF count: 11
- /kaggle/working/dntc_auto/raw_drive/01.pdf
- /kaggle/working/dntc_auto/raw_drive/05.pdf
- /kaggle/working/dntc_auto/raw_drive/07_08.pdf
- /kaggle/working/dntc_auto/raw_drive/09.pdf
- /kaggle/working/dntc_auto/raw_drive/10_11.pdf
- /kaggle/working/dntc_auto/raw_drive/12.pdf
- /kaggle/working/dntc_auto/raw_drive/13.pdf
- /kaggle/working/dntc_auto/raw_drive/14_15.pdf
- /kaggle/working/dntc_auto/raw_drive/16_17.pdf
- /kaggle/working/dntc_auto/raw_drive/q2_3_4.pdf
- /kaggle/working/dntc_auto/raw_drive/q6.pdf



Download completed


In [5]:

# ============================================================
# 4. Text normalization, metrics, quality score
# ============================================================

def norm_text(s) -> str:
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFC", s)
    replacements = {
        "\u00a0": " ", "\ufeff": " ", "￾": " ", "ﬁ": "fi", "ﬂ": "fl",
        "`": "'", "´": "'", "“": "\"", "”": "\"", "‘": "'", "’": "'",
    }
    for a, b in replacements.items():
        s = s.replace(a, b)
    s = CONTROL_RE.sub(" ", s)
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def normalize_for_match(s: str) -> str:
    s = norm_text(s).lower()
    s = re.sub(r"[\-–—_]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFD", s)
    return "".join(ch for ch in s if unicodedata.category(ch) != "Mn").replace("đ", "d").replace("Đ", "D")


def count_repeated_fragment_ratio(s: str) -> float:
    s0 = re.sub(r"\s+", "", norm_text(s))
    if len(s0) < 8:
        return 0.0
    repeated_chars = sum(len(m.group(0)) for m in REPEATED_FRAGMENT_RE.finditer(s0))
    # Also catch long runs like eeeee or -----.
    repeated_chars += sum(len(m.group(0)) for m in re.finditer(r"(.)\1{4,}", s0))
    return min(1.0, repeated_chars / max(1, len(s0)))


def punctuation_balance(s: str) -> float:
    s = norm_text(s)
    pairs = [("(", ")"), ("[", "]"), ("{", "}"), ("«", "»"), ('"', '"')]
    imbalance = 0
    for a, b in pairs:
        if a == b:
            imbalance += abs(s.count(a) % 2)
        else:
            imbalance += abs(s.count(a) - s.count(b))
    punct_density = len(re.findall(r"[.,;:!?/\\|]{2,}", s))
    return min(1.0, (imbalance + punct_density) / max(1, len(s) / 40.0))


def library_noise_score(s: str) -> float:
    s0 = norm_text(s)
    if not s0:
        return 0.0
    hits = len(LIBRARY_NOISE_RE.findall(s0))
    callno = 1 if BARCODE_CALLNO_RE.fullmatch(s0) and not CONTENT_START_RE.search(s0) else 0
    digit_chunks = len(re.findall(r"\d{4,}", s0))
    score = 0.17 * hits + 0.12 * callno + 0.025 * digit_chunks
    return min(1.0, score)


def latin_noise_ratio(s: str) -> float:
    s0 = norm_text(s)
    words = WORDS_RE.findall(s0)
    if not words:
        return 0.0
    bad = 0
    for w in words:
        wl = w.lower()
        if wl in COMMON_VI_WORDS or wl in COMMON_UNACCENTED_VI_WORDS:
            continue
        if len(w) >= 4:
            vowel_count = sum(1 for ch in w if ch in VIET_VOWELS)
            if vowel_count == 0:
                bad += 1
            elif ord(max(w)) < 128 and len(w) >= 7 and vowel_count / len(w) < 0.22:
                bad += 1
    return bad / max(1, len(words))


def text_metrics(text: str, lines=None, ocr_conf_values=None) -> dict:
    s = norm_text(text)
    n = max(1, len(s))
    letters = LETTERS_RE.findall(s)
    words = [w.lower() for w in WORDS_RE.findall(s)]
    word_count = len(words)
    accent_count = sum(1 for ch in s if ch in VIET_CHARS)
    cjk_count = len(CJK_RE.findall(s))
    weird_count = len(WEIRD_RE.findall(s)) + 3 * len(NON_VIET_SCRIPT_RE.findall(s))
    controls = len(CONTROL_RE.findall(str(text or "")))
    digits = len(DIGIT_RE.findall(s))
    dictionary_hits = sum(1 for w in words if w in COMMON_VI_WORDS)
    unaccented_hits = sum(1 for w in words if w in COMMON_UNACCENTED_VI_WORDS)
    domain_hits = sum(1 for h in DOMAIN_HEADINGS if h.lower() in s.lower())
    vietnamese_ratio = min(1.0, (dictionary_hits + 0.45 * accent_count + 2.0 * domain_hits) / max(1, word_count))
    dictionary_hit_ratio = dictionary_hits / max(1, word_count)
    accent_ratio = accent_count / max(1, len(letters))
    weird_char_ratio = (weird_count + controls) / n
    digit_ratio = digits / n
    rep_ratio = count_repeated_fragment_ratio(s)
    lib_score = library_noise_score(s)
    lat_noise = latin_noise_ratio(s)
    avg_line_length = 0.0
    if lines:
        nonempty = [norm_text(x) for x in lines if norm_text(x)]
        avg_line_length = float(np.mean([len(x) for x in nonempty])) if nonempty else 0.0
    else:
        avg_line_length = len(s)
    punct_bal = punctuation_balance(s)
    avg_ocr_conf = None
    if ocr_conf_values:
        vals = [float(x) for x in ocr_conf_values if x is not None and not pd.isna(x) and float(x) >= 0]
        avg_ocr_conf = float(np.mean(vals)) if vals else None

    # Quality score 0-100-ish. Designed to be conservative.
    score = 50.0
    score += 24.0 * dictionary_hit_ratio
    score += 18.0 * vietnamese_ratio
    score += min(8.0, avg_line_length / 12.0)
    if word_count >= 20:
        score += 4.0
    if accent_ratio >= 0.04:
        score += 4.0
    score -= 95.0 * weird_char_ratio
    score -= 28.0 * lat_noise
    score -= 30.0 * rep_ratio
    score -= 24.0 * lib_score
    score -= 12.0 * punct_bal
    score -= 16.0 * digit_ratio if digit_ratio > 0.22 else 0.0
    if word_count < 4 and not domain_hits:
        score -= 28.0
    if len(letters) < 15 and not domain_hits:
        score -= 18.0
    if word_count >= 12 and accent_ratio < 0.018 and unaccented_hits < 2 and dictionary_hits < 2:
        score -= 14.0
    if avg_ocr_conf is not None:
        if avg_ocr_conf < 25:
            score -= 16.0
        elif avg_ocr_conf < 38:
            score -= 8.0
        elif avg_ocr_conf > 55:
            score += 3.0
    if CLEAR_JUNK_RE.search(s):
        score -= 32.0
    if CONTENT_START_RE.search(s):
        score += 5.0
    return {
        "char_count": len(s), "letter_count": len(letters), "word_count": word_count,
        "accent_count": accent_count, "accent_ratio": accent_ratio, "cjk_count": cjk_count,
        "weird_char_count": weird_count, "control_count": controls, "weird_char_ratio": weird_char_ratio,
        "vietnamese_ratio": vietnamese_ratio, "dictionary_hit_ratio": dictionary_hit_ratio,
        "unaccented_vi_hits": unaccented_hits, "domain_heading_hits": domain_hits,
        "latin_noise_ratio": lat_noise, "avg_line_length": avg_line_length,
        "punctuation_balance": punct_bal, "digit_ratio": digit_ratio,
        "repeated_char_ngram_ratio": rep_ratio, "library_noise_score": lib_score,
        "avg_ocr_conf": avg_ocr_conf, "quality_score": round(score, 3),
    }


def is_heading_text(s: str) -> bool:
    s0 = norm_text(s)
    if not s0 or len(s0) > 110:
        return False
    su = s0.upper()
    if any(h in su for h in DOMAIN_HEADINGS):
        return True
    if re.fullmatch(r"(?:QUYỂN|QUYEN|TẬP|TAP|TỈNH|TINH)\s+[A-ZÀ-ỸĐ0-9IVXLCDM .\-]+", su):
        return True
    letters = LETTERS_RE.findall(s0)
    if len(letters) >= 4:
        upperish = sum(1 for ch in letters if ch.upper() == ch) / len(letters)
        if upperish >= 0.82 and len(s0) <= 80 and not LIBRARY_NOISE_RE.search(s0):
            return True
    return False


def is_title_or_frontmatter_text(s: str, page_number: int) -> bool:
    s0 = norm_text(s)
    if not s0:
        return False
    if page_number <= AUTO_CONTENT_SCAN_PAGES and TITLE_FRONTMATTER_RE.search(s0):
        # A long actual preface page can mention publisher; do not classify as title if content terms dominate.
        words = WORDS_RE.findall(s0)
        if len(words) < 70 or not CONTENT_START_RE.search(s0):
            return True
    title_only_tokens = ["ĐẠI NAM", "NHẤT THỐNG", "DỊCH GIẢ", "XUẤT BẢN", "NHÀ XUẤT BẢN", "TẬP SỐ"]
    hit = sum(1 for t in title_only_tokens if t.lower() in s0.lower())
    if page_number <= AUTO_CONTENT_SCAN_PAGES and hit >= 2 and len(WORDS_RE.findall(s0)) < 80:
        return True
    return False


In [6]:

# ============================================================
# 5. Rendering, visual page metrics, OCR, PDF text extraction
# ============================================================

def tesseract_available() -> bool:
    return ENABLE_TESSERACT and pytesseract is not None and shutil.which("tesseract") is not None


def render_page_to_pil(page, dpi=90, clip=None) -> Image.Image:
    zoom = dpi / 72.0
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), clip=clip, alpha=False)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


def visual_page_metrics(page) -> dict:
    try:
        img = render_page_to_pil(page, dpi=45)
        arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
        gray = arr.mean(axis=2)
        whiteness = float(np.mean(gray > 0.94))
        darkness = float(np.mean(gray < 0.12))
        ink_ratio = float(np.mean(gray < 0.88))
        color_std = float(np.mean(np.std(arr, axis=(0, 1))))
        mx = arr.max(axis=2)
        mn = arr.min(axis=2)
        saturation = float(np.mean((mx - mn) / np.maximum(mx, 1e-6)))
        gy = np.abs(np.diff(gray, axis=0)).mean()
        gx = np.abs(np.diff(gray, axis=1)).mean()
        edge_density = float(gx + gy)
        h, w = gray.shape
        border = max(2, int(min(h, w) * 0.035))
        border_pixels = np.concatenate([
            gray[:border, :].ravel(), gray[-border:, :].ravel(), gray[:, :border].ravel(), gray[:, -border:].ravel()
        ])
        border_ink_ratio = float(np.mean(border_pixels < 0.80))
        return {
            "white_ratio": whiteness, "dark_ratio": darkness, "ink_ratio": ink_ratio,
            "color_std": color_std, "saturation": saturation, "edge_density": edge_density,
            "border_ink_ratio": border_ink_ratio,
            "visual_blank_score": float(whiteness > 0.985 and ink_ratio < 0.018),
            "visual_pattern_score": min(1.0, max(0.0, saturation * 1.8 + color_std * 1.2 + edge_density * 4.0 - whiteness * 0.75)),
        }
    except Exception as e:
        return {"visual_error": type(e).__name__, "white_ratio": None, "dark_ratio": None, "ink_ratio": None,
                "color_std": None, "saturation": None, "edge_density": None, "border_ink_ratio": None,
                "visual_blank_score": 0.0, "visual_pattern_score": 0.0}


def pil_preprocess_for_ocr(img: Image.Image, mode="page") -> Image.Image:
    img = img.convert("RGB")
    gray = ImageOps.grayscale(img)
    gray = ImageOps.autocontrast(gray)
    gray = ImageEnhance.Contrast(gray).enhance(1.35 if mode == "page" else 1.65)
    gray = ImageEnhance.Sharpness(gray).enhance(1.20 if mode == "page" else 1.55)
    if mode == "line":
        gray = gray.filter(ImageFilter.MedianFilter(size=3))
    return gray


def tesseract_data_from_image(img: Image.Image, psm=6):
    if not tesseract_available():
        return pd.DataFrame()
    try:
        config = f"--oem 1 --psm {psm}"
        df = pytesseract.image_to_data(img, lang=OCR_LANG, config=config, output_type=Output.DATAFRAME)
        if df is None:
            return pd.DataFrame()
        return df
    except Exception as e:
        return pd.DataFrame()


def extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=PAGE_OCR_DPI) -> list:
    if not tesseract_available():
        return []
    img = render_page_to_pil(page, dpi=dpi)
    zoom = dpi / 72.0
    img = pil_preprocess_for_ocr(img, mode="page")
    df = tesseract_data_from_image(img, psm=6)
    if df.empty:
        return []
    df = df.dropna(subset=["text"]).copy()
    if df.empty:
        return []
    df["text"] = df["text"].map(norm_text)
    df = df[df["text"].str.len() > 0]
    if df.empty:
        return []
    if "conf" in df.columns:
        df["conf_num"] = pd.to_numeric(df["conf"], errors="coerce").fillna(-1)
    else:
        df["conf_num"] = -1
    group_cols = [c for c in ["block_num", "par_num", "line_num"] if c in df.columns]
    if not group_cols:
        group_cols = ["level"] if "level" in df.columns else []
    if not group_cols:
        return []
    lines = []
    for _, g in df.groupby(group_cols, sort=True):
        words = [norm_text(x) for x in g["text"].tolist() if norm_text(x)]
        text = norm_text(" ".join(words))
        if not text:
            continue
        conf_vals = [float(x) for x in g["conf_num"].tolist() if float(x) >= 0]
        conf = float(np.mean(conf_vals)) if conf_vals else -1.0
        left = float(g["left"].min()) / zoom
        top = float(g["top"].min()) / zoom
        right = float((g["left"] + g["width"]).max()) / zoom
        bottom = float((g["top"] + g["height"]).max()) / zoom
        lines.append({
            "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_idx + 1,
            "source": "tesseract_page", "bbox": [left, top, right, bottom], "raw_text": text,
            "ocr_conf": round(conf, 3), "block_id": None, "line_id": None,
        })
    lines.sort(key=lambda r: (r["bbox"][1], r["bbox"][0]))
    return lines


def extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path) -> list:
    out = []
    try:
        data = page.get_text("dict")
    except Exception:
        return out
    for b_i, block in enumerate(data.get("blocks", [])):
        if block.get("type") != 0:
            continue
        for l_i, line in enumerate(block.get("lines", [])):
            spans = line.get("spans", [])
            text = norm_text(" ".join([sp.get("text", "") for sp in spans]))
            if not text:
                continue
            bbox = line.get("bbox", block.get("bbox", None))
            if not bbox:
                continue
            font_sizes = [float(sp.get("size", 0) or 0) for sp in spans]
            out.append({
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_idx + 1,
                "source": "pdf_text_layer", "bbox": [float(x) for x in bbox], "raw_text": text,
                "ocr_conf": None, "block_id": b_i, "line_id": l_i,
                "font_size_avg": round(float(np.mean(font_sizes)) if font_sizes else 0.0, 3),
            })
    out.sort(key=lambda r: (r["bbox"][1], r["bbox"][0]))
    return out


def lines_to_text(lines) -> str:
    return "\n".join(norm_text(r.get("raw_text") or r.get("text")) for r in lines if norm_text(r.get("raw_text") or r.get("text")))


def line_crop_ocr(page, bbox) -> tuple:
    if not tesseract_available():
        return "", -1.0
    r = fitz.Rect(bbox)
    r.x0 = max(page.rect.x0, r.x0 - LINE_REOCR_PAD_PT)
    r.y0 = max(page.rect.y0, r.y0 - LINE_REOCR_PAD_PT)
    r.x1 = min(page.rect.x1, r.x1 + LINE_REOCR_PAD_PT)
    r.y1 = min(page.rect.y1, r.y1 + LINE_REOCR_PAD_PT)
    pix = page.get_pixmap(matrix=fitz.Matrix(LINE_REOCR_ZOOM, LINE_REOCR_ZOOM), clip=r, alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    variants = [pil_preprocess_for_ocr(img, mode="line"), img]
    best_text, best_conf, best_score = "", -1.0, -999.0
    for im in variants:
        df = tesseract_data_from_image(im, psm=7)
        if df.empty:
            continue
        df = df.dropna(subset=["text"]).copy()
        if df.empty:
            continue
        txt = norm_text(" ".join(df["text"].map(norm_text).tolist()))
        conf = -1.0
        if "conf" in df.columns:
            vals = pd.to_numeric(df["conf"], errors="coerce")
            vals = vals[vals >= 0]
            if len(vals):
                conf = float(vals.mean())
        score = text_metrics(txt, ocr_conf_values=[conf]).get("quality_score", -999.0)
        if score > best_score:
            best_text, best_conf, best_score = txt, conf, score
    return best_text, best_conf


In [7]:

# ============================================================
# 6. Conservative Vietnamese/domain correction with log
# ============================================================
CORRECTION_RULES = [
    # Headings and title variants.
    ("heading_dai_nam", r"\bDAI\s*-?\s*NAM\b|\bĐAI\s*-?\s*NAM\b", "ĐẠI NAM"),
    ("heading_nhat_thong_chi", r"\bNH[ẤA]T\s*-?\s*TH[OỐ]NG\s*-?\s*CH[ÍI]\b|\bNHAT\s+THONG\s+CHI\b", "NHẤT THỐNG CHÍ"),
    ("heading_quyen", r"\bQUYEN\b", "QUYỂN"),
    ("heading_tinh", r"\bTINH\b(?=\s+[A-ZÀ-ỸĐ])", "TỈNH"),
    ("heading_phan_da", r"\bPHAN\s+DA\b|\bPHẦN\s+DA\b", "PHẦN DÃ"),
    ("heading_dung_dat", r"\bD[UƯ]NG\s+[DPĐ]AT\s+VA\s+DI[ÊE]N\s+C[ÁA]CH\b|\bDUNG\s+DAT\s+VA\s+DIEN\s+CACH\b", "DỰNG ĐẶT VÀ DIÊN CÁCH"),
    ("heading_hinh_the", r"\bHINH\s+THE\b", "HÌNH THẾ"),
    ("heading_phong_tuc", r"\bPHONG\s+TUC\b", "PHONG TỤC"),
    ("heading_thue_ruong", r"\bTHUE\s+RUONG\b", "THUẾ RUỘNG"),
    ("heading_nui_song", r"\bNUI\s+SONG\b", "NÚI SÔNG"),

    # Specific OCR/mojibake artifacts reported for DNTC.
    ("mojibake_chiem", r"\bchi€m\b", "chiếm"),
    ("mojibake_bien", r"\bbi€n\b", "biển"),
    ("mojibake_kiem", r"\bki€m\b", "kiêm"),
    ("mojibake_mieu", r"\bmi€u\b", "miếu"),
    ("n_tilde_nam", r"\bñăm\b", "năm"),
    ("the_ky", r"\bth[eéế]\s+k[yỷ]\b|\bth[eéế]\s+kỷ\b", "thế kỷ"),
    ("dau_the_ky", r"\bĐầu\s+th[eéế]\s+k[yỷ]\b", "Đầu thế kỷ"),
    ("doi_tu_duc", r"\bdoi\s+#?ự\s+Đức\b|\bđời\s+#?ự\s+Đức\b", "đời Tự Đức"),
    ("doi_hien_tong", r"\bdoi\s+H[ií]én\s+Tông\b", "đời Hiến Tông"),
    ("nha_tuy", r"\bNha\s+Tuy\b", "Nhà Tùy"),
    ("nha_duong", r"\bNha\s+Đường\b", "Nhà Đường"),
    ("nuoc_ta", r"\bNước\s+tả\b", "Nước ta"),
    ("cao_men", r"\bCao\s+M[eé]n\b", "Cao Mên"),
    ("huyen_hien", r"\bhuyénhién\b", "huyện hiện"),
    ("cuop", r"\bcudp\b", "cướp"),


    # Additional systematic artifacts from the second full run.
    ("mojibake_nguyen_upper", r"\bÑg", "Ng"),
    ("mojibake_n_upper", r"\bÑ(?=[A-ZÀ-ỸĐa-zà-ỹđ])", "N"),
    ("mojibake_n_lower", r"ñ", "n"),
    ("mojibake_d_upper", r"Ð", "Đ"),
    ("mojibake_d_lower", r"ð", "đ"),
    ("mojibake_o_place", r"\bÖ['’]?\s+(?=phía|thôn|xã|huyện|trên|cực|địa)", "Ở "),
    ("ocr_zero_place", r"\b0['°]\s+(?=phía|thôn|xã|huyện|trên|cực|địa)", "Ở "),
    ("dam_dim_after_number", r"(\d+)\s+d[ıi]m\b", r"\1 dặm"),
    ("dam_mojibake_after_number", r"(\d+)\s+d[§s]\s*m\b", r"\1 dặm"),
    ("tir_tinh_li", r"\bTir\s+t[ií]nh\s*[- ]\s*l[ií]\b", "Từ tỉnh-lỵ"),
    ("tinh_li_hyphen", r"\btinh\s*[- ]\s*li\b", "tỉnh-lỵ"),
    ("le_thanh_tong", r"\bLe\s+Thdnh\s+T[eé]ng\b", "Lê Thánh Tông"),
    ("thi_si", r"\bThi\s+Si\b", "Thi Sĩ"),
    ("hong_duc", r"\bHong\s+Đức\b", "Hồng Đức"),
    ("gia_cat", r"\bCia\s*[- ]\s*cat\b", "Gia Cát"),
    ("thing_chi", r"\bthing\s+chí\b", "thống chí"),
    ("word_chonay", r"\bchỗnày\b", "chỗ này"),
    ("word_chonao", r"\bchỗnào\b", "chỗ nào"),
    ("word_canui", r"\bcảnúi\b", "cả núi"),
    ("word_nhantai", r"\bnhântài\b", "nhân tài"),
    ("word_nienhieu", r"\bniênhiệu\b", "niên hiệu"),
    ("word_vephia", r"\bvềphía\b", "về phía"),
    ("word_tuphia", r"\btừphía\b", "từ phía"),

    # Contextual administrative words. Conservative: only in common administrative contexts.
    ("tinh_ly", r"\btinh\s+l[yỵi]\b", "tỉnh lỵ"),
    ("tinh_thanh", r"\btinh\s+thành\b", "tỉnh thành"),
    ("tinh_place", r"\btinh\s+(Quảng|Thanh|Nghệ|Bình|Phú|Khánh|Hà|An|Gia|Định|Vĩnh|Biên|Quy|Hải|Nam|Bắc)\b", r"tỉnh \1"),
    ("huyen_context", r"\bhuyen\s+(?=[A-ZÀ-ỸĐ])", "huyện "),
    ("phu_context", r"\bphu\s+(?=[A-ZÀ-ỸĐ])", "phủ "),
    ("chau_context", r"\bchau\s+(?=[A-ZÀ-ỸĐ])", "châu "),

    # Common names.
    ("minh_menh", r"\bMinh\s+M[ée]nh\b|\bMinh\s+Mộệnh\b", "Minh Mệnh"),
    ("thieu_tri", r"\bThiệu\s+Tri\b", "Thiệu Trị"),
    ("tu_duc", r"\bTự\s+Dức\b|\bDự\s+Đức\b", "Tự Đức"),
    ("gia_du", r"\bGia\s+Du\s+Hoàng\b", "Gia Dụ Hoàng"),
    ("ha_tien", r"\bHa\s+Tien\b", "Hà Tiên"),
    ("quang_binh", r"\bQuang\s+Binh\b", "Quảng Bình"),
    ("binh_dinh", r"\bBinh\s+Dinh\b", "Bình Định"),
    ("quang_yen", r"\bQuang\s+Yen\b", "Quảng Yên"),
]


def apply_corrections(text: str, meta: dict, correction_log: list) -> str:
    original = norm_text(text)
    s = original
    # First normalize punctuation and known glyphs.
    pre = s
    s = s.replace("￾", " ").replace("\u0001", " ")
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([,.;:!?])(?=\S)", r"\1 ", s)
    s = re.sub(r"(\d)\s+([,.])\s+(\d)", r"\1\2\3", s)
    s = re.sub(r"\s+", " ", s).strip()
    if s != pre:
        correction_log.append({**meta, "level": "normalize", "rule_id": "unicode_space_punct", "before": pre, "after": s})
    for rule_id, pat, repl in CORRECTION_RULES:
        before = s
        s, n = re.subn(pat, repl, s, flags=re.IGNORECASE)
        if n and s != before:
            correction_log.append({**meta, "level": "domain_rule", "rule_id": rule_id, "before": before, "after": s})
    s = re.sub(r"\s+", " ", s).strip()
    return s


def candidate_correction_only(text: str) -> str:
    tmp_log = []
    return apply_corrections(text, {}, tmp_log)


def choose_better_line_text(base_text: str, ocr_text: str, ocr_conf: float, meta: dict, correction_log: list) -> tuple:
    base_fixed = apply_corrections(base_text, meta, correction_log)
    ocr_fixed = candidate_correction_only(ocr_text)
    if not ocr_fixed:
        return base_fixed, "rules_only", False, ""
    base_m = text_metrics(base_fixed)
    ocr_m = text_metrics(ocr_fixed, ocr_conf_values=[ocr_conf])
    # Keep base if OCR loses too many digits or becomes much shorter.
    base_digits = re.findall(r"\d+(?:[.,]\d+)?", base_fixed)
    if base_digits:
        kept = sum(1 for d in base_digits if d in ocr_fixed)
        if kept / max(1, len(base_digits)) < 0.70:
            return base_fixed, "keep_base_ocr_lost_digits", False, ocr_fixed
    if len(ocr_fixed) < 0.55 * max(1, len(base_fixed)):
        return base_fixed, "keep_base_ocr_too_short", False, ocr_fixed
    if ocr_conf is not None and ocr_conf >= 0 and ocr_conf < 28 and ocr_m["quality_score"] < base_m["quality_score"] + 10:
        return base_fixed, "keep_base_low_ocr_conf", False, ocr_fixed
    if ocr_m["quality_score"] >= base_m["quality_score"] + 8 and ocr_m["weird_char_ratio"] <= base_m["weird_char_ratio"] + 0.02:
        # Log accepted line OCR as a correction event.
        correction_log.append({**meta, "level": "line_ocr", "rule_id": "accepted_better_line_ocr", "before": base_fixed, "after": ocr_fixed})
        return ocr_fixed, f"accept_line_ocr {base_m['quality_score']:.1f}->{ocr_m['quality_score']:.1f} conf={ocr_conf:.1f}", True, ocr_fixed
    return base_fixed, f"keep_base {base_m['quality_score']:.1f}->{ocr_m['quality_score']:.1f} conf={ocr_conf:.1f}", False, ocr_fixed


In [8]:

# ============================================================
# 7. Page classification and source selection
# ============================================================

def is_blank_page(text_m: dict, visual_m: dict) -> bool:
    if text_m["word_count"] <= 2 and (visual_m.get("visual_blank_score") == 1.0 or (visual_m.get("white_ratio") or 0) > 0.975):
        return True
    if text_m["letter_count"] < 5 and (visual_m.get("ink_ratio") or 1) < 0.025:
        return True
    return False


def is_patterned_page(text_m: dict, visual_m: dict, page_number: int) -> bool:
    if page_number > AUTO_CONTENT_SCAN_PAGES and text_m["word_count"] > 12:
        return False
    pattern_visual = (visual_m.get("visual_pattern_score") or 0) > 0.38 and (visual_m.get("white_ratio") or 1) < 0.82
    bad_text = text_m["vietnamese_ratio"] < 0.12 and text_m["word_count"] < 45
    return bool(pattern_visual and bad_text)



def is_backmatter_page(text: str, text_m: dict) -> bool:
    """Publisher ads / print info / TOC pages after main content.
    Keep real content containing domain headings; drop pages dominated by publishing metadata."""
    s = norm_text(text)
    if not s:
        return False
    if not BACKMATTER_NOISE_RE.search(s):
        return False
    # Mục lục / print info / sales ads are never body content.
    if re.search(r"\b(MỤC\s+LỤC|MUC\s+LUC|Số\s+đăng\s+kí\s+KHXB|Quyết\s+định\s+xuất\s+bản|In\s+(50|1000)\s+cuốn|Những\s+tập\s+VĂN\s*HÓA)\b", s, re.I):
        return True
    if text_m.get("word_count", 0) < 120 and text_m.get("domain_heading_hits", 0) == 0:
        return True
    return False


def is_library_page(text: str, text_m: dict) -> bool:
    if LIBRARY_NOISE_RE.search(norm_text(text)):
        return True
    if text_m["library_noise_score"] >= MAX_LIBRARY_NOISE_SCORE and text_m["vietnamese_ratio"] < 0.25:
        return True
    return False


def page_content_signal(text: str, text_m: dict) -> float:
    s = norm_text(text)
    score = 0.0
    score += min(35.0, text_m["word_count"] * 0.45)
    score += 25.0 * text_m["vietnamese_ratio"]
    score += 18.0 if CONTENT_START_RE.search(s) else 0.0
    score += 10.0 if any(h.lower() in s.lower() for h in DOMAIN_HEADINGS) else 0.0
    score -= 22.0 if TITLE_FRONTMATTER_RE.search(s) and text_m["word_count"] < 70 else 0.0
    score -= 35.0 * text_m["library_noise_score"]
    score -= 18.0 * text_m["repeated_char_ngram_ratio"]
    return score


def classify_page(page_number: int, text: str, text_m: dict, visual_m: dict, content_start_page: int | None) -> tuple:
    reasons = []
    cls = "content_candidate"
    if DROP_BLANK_PAGES and is_blank_page(text_m, visual_m):
        return "blank_page", ["blank_visual_or_no_text"]
    if DROP_PATTERNED_PAGES and is_patterned_page(text_m, visual_m, page_number):
        return "patterned_endpaper", ["patterned_visual_low_text"]
    if DROP_LIBRARY_PAGES and is_library_page(text, text_m):
        return "library_barcode_stamp_watermark", ["library_barcode_google_tokens"]
    if globals().get("DROP_BACKMATTER_PAGES", True) and is_backmatter_page(text, text_m):
        return "publisher_backmatter_page", ["publisher_backmatter_tokens"]
    if content_start_page is not None and DROP_FRONT_MATTER_BEFORE_CONTENT and page_number < content_start_page:
        return "front_matter_before_content", [f"before_auto_content_start_{content_start_page}"]
    if not KEEP_TITLE_PAGES and is_title_or_frontmatter_text(text, page_number):
        return "cover_title_front_matter", ["title_or_publisher_page"]
    if text_m["quality_score"] < 25 and text_m["word_count"] < 15:
        return "junk_ocr_page", ["low_text_quality_too_few_words"]
    return cls, reasons


def estimate_content_start(doc, work_id, pdf_path) -> int:
    if CONTENT_START_MODE != "auto":
        return 1
    max_scan = min(len(doc), AUTO_CONTENT_SCAN_PAGES)
    candidates = []
    for page_idx in range(max_scan):
        page = doc[page_idx]
        # Use text layer first. If text layer is nearly empty/bad, do a cheap OCR pass for content-start detection only.
        tl_lines = extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path)
        tl_text = lines_to_text(tl_lines)
        tl_m = text_metrics(tl_text, lines=[r.get("raw_text", "") for r in tl_lines])
        detection_text = tl_text
        detection_m = tl_m
        if tl_m["letter_count"] < 20 or tl_m["weird_char_ratio"] > 0.10:
            try:
                ocr_lines = extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=FAST_FRONTMATTER_OCR_DPI)
                ocr_text = lines_to_text(ocr_lines)
                ocr_m = text_metrics(ocr_text, lines=[r.get("raw_text", "") for r in ocr_lines], ocr_conf_values=[r.get("ocr_conf") for r in ocr_lines])
                if ocr_m["quality_score"] > detection_m["quality_score"]:
                    detection_text, detection_m = ocr_text, ocr_m
            except Exception:
                pass
        visual_m = visual_page_metrics(page)
        preliminary_cls, _ = classify_page(page_idx + 1, detection_text, detection_m, visual_m, content_start_page=None)
        signal = page_content_signal(detection_text, detection_m)
        has_start = CONTENT_START_RE.search(norm_text(detection_text)) is not None
        title_front = is_title_or_frontmatter_text(detection_text, page_idx + 1)
        candidates.append({
            "page_number": page_idx + 1, "signal": signal, "has_start": has_start,
            "word_count": detection_m["word_count"], "line_count": len(detection_text.splitlines()),
            "quality_score": detection_m["quality_score"], "preliminary_cls": preliminary_cls,
            "title_front": title_front,
        })
    # Prefer the first page that has actual prose/content, not title-only.
    for c in candidates:
        if c["preliminary_cls"] in {"blank_page", "patterned_endpaper", "library_barcode_stamp_watermark"}:
            continue
        if c["title_front"] and not KEEP_TITLE_PAGES:
            continue
        if c["has_start"] and c["word_count"] >= MIN_CONTENT_WORDS_ON_START_PAGE and c["quality_score"] >= 30:
            return int(c["page_number"])
    for c in candidates:
        if c["preliminary_cls"] == "content_candidate" and c["signal"] >= 35 and c["word_count"] >= MIN_CONTENT_WORDS_ON_START_PAGE:
            return int(c["page_number"])
    # Safe fallback: first page after obvious front matter with decent words.
    for c in candidates:
        if c["preliminary_cls"] == "content_candidate" and not c["title_front"] and c["word_count"] >= 25:
            return int(c["page_number"])
    return 1


def select_page_source(page, page_idx, work_id, pdf_path, page_class) -> tuple:
    # Always inspect text layer. OCR only if needed.
    tl_lines = extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path)
    tl_text = lines_to_text(tl_lines)
    tl_m = text_metrics(tl_text, lines=[r.get("raw_text", "") for r in tl_lines])

    should_ocr = ENABLE_TESSERACT and tesseract_available() and (
        tl_m["quality_score"] < MIN_TEXT_LAYER_GOOD_QUALITY or
        tl_m["letter_count"] < 40 or
        tl_m["weird_char_ratio"] > 0.06 or
        tl_m["repeated_char_ngram_ratio"] > 0.08
    )
    ocr_lines, ocr_text, ocr_m = [], "", None
    if should_ocr and page_class not in {"blank_page", "patterned_endpaper", "library_barcode_stamp_watermark", "cover_title_front_matter", "front_matter_before_content"}:
        ocr_lines = extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=PAGE_OCR_DPI)
        ocr_text = lines_to_text(ocr_lines)
        ocr_m = text_metrics(ocr_text, lines=[r.get("raw_text", "") for r in ocr_lines], ocr_conf_values=[r.get("ocr_conf") for r in ocr_lines])

    # Source decision.
    if ocr_m is None or not ocr_lines:
        selected, source, selected_m = tl_lines, "pdf_text_layer", tl_m
        reason = "text_layer_only_or_ocr_unavailable"
    else:
        # Do not let OCR replace a good text layer unless it is clearly better.
        if tl_m["quality_score"] >= MIN_TEXT_LAYER_GOOD_QUALITY and tl_m["quality_score"] >= ocr_m["quality_score"] - 7:
            selected, source, selected_m = tl_lines, "pdf_text_layer", tl_m
            reason = f"keep_text_layer_good tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
        elif ocr_m["quality_score"] >= tl_m["quality_score"] + 7 and ocr_m["quality_score"] >= MIN_OCR_PAGE_QUALITY:
            selected, source, selected_m = ocr_lines, "tesseract_page", ocr_m
            reason = f"use_ocr_better tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
        elif tl_m["quality_score"] >= 35 and tl_m["word_count"] >= 8:
            selected, source, selected_m = tl_lines, "pdf_text_layer", tl_m
            reason = f"fallback_text_layer_ocr_not_good_enough tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
        else:
            selected, source, selected_m = [], "drop_no_reliable_source", max([tl_m, ocr_m], key=lambda x: x["quality_score"])
            reason = f"drop_both_sources_low tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
    return selected, source, selected_m, tl_m, (ocr_m or {}), reason


In [9]:

# ============================================================
# 8. Line filtering, paragraph reflow, sentence splitting
# ============================================================

def line_quality(text: str, ocr_conf=None) -> dict:
    return text_metrics(text, lines=[text], ocr_conf_values=[ocr_conf] if ocr_conf is not None else None)


def is_repeated_junk_line(s: str) -> bool:
    s0 = norm_text(s)
    if not s0:
        return True
    if CLEAR_JUNK_RE.search(s0):
        return True
    if REPEATED_FRAGMENT_RE.search(re.sub(r"\s+", "", s0)):
        return True
    words = WORDS_RE.findall(s0)
    if words and len(words) <= 5:
        no_vowels = sum(1 for w in words if len(w) >= 3 and not any(ch in VIET_VOWELS for ch in w))
        if no_vowels / max(1, len(words)) > 0.55:
            return True
    # Patterned endpapers often OCR as B, ER, 83, 3, ॐ repeated.
    if re.fullmatch(r"[\sBЕER83()*0-9ॐఓజిஆ६]+", s0, flags=re.I):
        return True
    return False



def has_vietnamese_diacritic(s: str) -> bool:
    return any(ch in VIET_CHARS for ch in norm_text(s))


def is_ascii_short_junk_line(s: str) -> bool:
    s0 = norm_text(s)
    if re.search(r"\b(A ee Ra|Ave sy|BREF|AG TA|OR tA|xf An|thon Ta-My|Fel Fat|Seer tit|mADS|OPOC)\b", s0, re.I):
        return True
    if any(ord(ch) >= 128 for ch in s0) or has_vietnamese_diacritic(s0):
        return False
    ws = WORDS_RE.findall(s0)
    if len(ws) >= 3 and len(s0) < 90:
        short_chunks = sum(1 for w in ws if len(w) <= 4)
        upper_chunks = sum(1 for w in ws if w.isupper())
        if short_chunks / max(1, len(ws)) > 0.82 or upper_chunks / max(1, len(ws)) > 0.50:
            return True
    return False


def is_line_hard_noise(s: str) -> bool:
    s0 = norm_text(s)
    if BACKMATTER_NOISE_RE.search(s0):
        return True
    if is_ascii_short_junk_line(s0):
        return True
    if NOISY_TESSERACT_TOKEN_RE.search(s0) and not has_vietnamese_diacritic(s0) and len(s0) < 80:
        return True
    return False


def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    raw = norm_text(line.get("raw_text") or line.get("text") or "")
    meta = {
        "work_id": line.get("work_id"), "pdf_path": line.get("pdf_path"), "page_number": line.get("page_number"),
        "page_idx": line.get("page_idx"), "line_id": line.get("line_global_id"), "source": line.get("source"),
    }
    reasons = []
    if not raw:
        return None, False, ["empty"]
    if is_line_hard_noise(raw):
        return None, False, ["hard_noise_or_backmatter_line"]
    if LIBRARY_NOISE_RE.search(raw):
        return None, False, ["library_barcode_google_line"]
    if re.fullmatch(r"\d{1,4}", raw):
        return None, False, ["page_number_only"]
    if BARCODE_CALLNO_RE.fullmatch(raw) and len(WORDS_RE.findall(raw)) <= 3 and not is_heading_text(raw):
        return None, False, ["call_number_or_barcode_fragment"]
    if is_repeated_junk_line(raw):
        return None, False, ["repeated_or_known_ocr_junk"]

    fixed = apply_corrections(raw, meta, correction_log)
    # Optional line re-OCR only for suspicious PDF text layer lines. Do not replace OCR-page lines with another OCR.
    line_ocr_text = ""
    line_ocr_conf = None
    line_ocr_accepted = False
    if (doc is not None and reocr_state is not None and line.get("source") == "pdf_text_layer" and
        reocr_state.get("used", 0) < MAX_LINE_REOCR_PER_PDF):
        q0 = line_quality(fixed)
        suspicious_for_reocr = (
            q0["quality_score"] < MIN_KEEP_LINE_QUALITY + 8 or q0["weird_char_ratio"] > 0.035 or
            q0["repeated_char_ngram_ratio"] > 0.04 or q0["unaccented_vi_hits"] >= 2 and q0["accent_ratio"] < 0.015
        )
        if suspicious_for_reocr:
            try:
                page = doc[line["page_idx"]]
                line_ocr_text, line_ocr_conf = line_crop_ocr(page, line["bbox"])
                reocr_state["used"] = reocr_state.get("used", 0) + 1
                fixed2, reason, accepted, line_ocr_text2 = choose_better_line_text(fixed, line_ocr_text, line_ocr_conf, meta, correction_log)
                if accepted:
                    fixed = fixed2
                    line_ocr_accepted = True
                reasons.append(reason)
            except Exception as e:
                reasons.append(f"line_reocr_failed:{type(e).__name__}")

    q = line_quality(fixed, line.get("ocr_conf"))
    heading = is_heading_text(fixed)
    if not heading:
        if len(fixed) < 7 or q["letter_count"] < 3:
            return None, False, reasons + ["too_short_not_heading"]
        if q["library_noise_score"] >= MAX_LIBRARY_NOISE_SCORE:
            return None, False, reasons + ["library_noise_score"]
        if q["weird_char_ratio"] > MAX_WEIRD_CHAR_RATIO:
            return None, False, reasons + ["too_many_weird_chars"]
        if q["repeated_char_ngram_ratio"] > 0.12:
            return None, False, reasons + ["repeated_ngram_ratio"]
        if q["word_count"] >= 7 and q["vietnamese_ratio"] < MIN_VIET_RATIO_FOR_LONG_TEXT and q["accent_ratio"] < 0.025:
            return None, False, reasons + ["low_vietnamese_ratio"]
        if line.get("source") == "tesseract_page" and line.get("ocr_conf") is not None and float(line.get("ocr_conf")) >= 0:
            if float(line.get("ocr_conf")) < MIN_OCR_CONF_LINE and q["quality_score"] < MIN_KEEP_LINE_QUALITY + 8:
                return None, False, reasons + ["low_tesseract_conf"]
        if q["quality_score"] < MIN_KEEP_LINE_QUALITY:
            return None, False, reasons + [f"low_line_quality:{q['quality_score']:.1f}"]

    kept = dict(line)
    kept["text"] = fixed
    kept["line_type"] = "heading" if heading else "body"
    kept["line_quality_score"] = q["quality_score"]
    kept["line_vietnamese_ratio"] = q["vietnamese_ratio"]
    kept["line_weird_char_ratio"] = q["weird_char_ratio"]
    kept["line_ocr_text"] = line_ocr_text
    kept["line_ocr_conf"] = line_ocr_conf
    kept["line_ocr_accepted"] = line_ocr_accepted
    kept["filter_reasons"] = ";".join(reasons) if reasons else "kept"
    return kept, True, reasons


def median_line_height(lines) -> float:
    vals = []
    for r in lines:
        try:
            b = r["bbox"]
            vals.append(max(1.0, float(b[3]) - float(b[1])))
        except Exception:
            pass
    return float(np.median(vals)) if vals else 12.0


def is_footnote_line(line: dict, page_rect) -> bool:
    try:
        y0 = line["bbox"][1]
        fs = line.get("font_size_avg") or 0
        text = line.get("text", "")
        return (y0 > page_rect.height * 0.80 and (re.match(r"^\(?\d+\)|^\*", text) or (fs and fs < 9)))
    except Exception:
        return False


def ends_strong_sentenceish(t: str) -> bool:
    t = norm_text(t)
    if not t:
        return False
    return bool(re.search(r'[.!?…;:)"»\\]]$', t))


def begins_continuation(t: str) -> bool:
    t = norm_text(t).lstrip(' "\'“”‘’([{<«»—–-:;,.')
    return bool(t and re.match(r'^[a-zàáảãạằắẳẵặầấẩẫậèéẻẽẹềếểễệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵđ]', t))


def should_new_paragraph(prev: dict, cur: dict, med_h: float, page_rect) -> bool:
    pt, ct = prev.get("text", ""), cur.get("text", "")
    if prev.get("line_type") == "heading" or cur.get("line_type") == "heading":
        return True
    if prev.get("is_footnote") != cur.get("is_footnote"):
        return True
    if re.match(r"^\(?\d+\)|^\d+[.)]", ct):
        return True
    try:
        gap = cur["bbox"][1] - prev["bbox"][3]
        indent_delta = cur["bbox"][0] - prev["bbox"][0]
        # The old threshold 0.95 split normal scanned lines into pseudo-sentences.
        # Only start a new paragraph on a large vertical gap, or a clear indent after a completed sentence.
        if gap > med_h * 1.85:
            return ends_strong_sentenceish(pt) or not begins_continuation(ct)
        if indent_delta > med_h * 2.0 and len(pt) > 35 and ends_strong_sentenceish(pt):
            return True
    except Exception:
        pass
    return False

def join_paragraph_lines(lines: list) -> str:
    parts = []
    for r in lines:
        t = norm_text(r.get("text", ""))
        if not t:
            continue
        if not parts:
            parts.append(t)
            continue
        prev = parts[-1]
        # Remove true line-break hyphen only for lowercase/letter split words.
        if re.search(r"[a-zà-ỹđ]-$", prev) and re.match(r"^[a-zà-ỹđ]", t):
            parts[-1] = prev[:-1] + t
        else:
            parts.append(t)
    text = " ".join(parts)
    text = re.sub(r"\s+", " ", text).strip()
    # Make old spaced punctuation less noisy.
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?])(?=\S)", r"\1 ", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    return text


def reflow_lines_to_paragraphs(lines: list, page_rect) -> list:
    if not lines:
        return []
    lines = sorted(lines, key=lambda r: (r["bbox"][1], r["bbox"][0]))
    for r in lines:
        r["is_footnote"] = is_footnote_line(r, page_rect)
    med_h = median_line_height(lines)
    groups, cur = [], []
    for r in lines:
        if not cur:
            cur = [r]
        elif should_new_paragraph(cur[-1], r, med_h, page_rect):
            groups.append(cur)
            cur = [r]
        else:
            cur.append(r)
    if cur:
        groups.append(cur)
    out = []
    for i, g in enumerate(groups):
        text = join_paragraph_lines(g)
        if not text:
            continue
        b = [min(x["bbox"][0] for x in g), min(x["bbox"][1] for x in g), max(x["bbox"][2] for x in g), max(x["bbox"][3] for x in g)]
        if all(x.get("line_type") == "heading" for x in g) and len(text) <= 130:
            ptype = "heading"
        elif any(x.get("is_footnote") for x in g):
            ptype = "footnote"
        else:
            ptype = "body"
        q = text_metrics(text, lines=[x.get("text", "") for x in g])
        out.append({
            "paragraph_index_in_page": i, "paragraph_type": ptype, "text": text, "bbox": b,
            "line_count": len(g), "source": ";".join(sorted(set(x.get("source", "") for x in g))),
            "quality_score": q["quality_score"], "vietnamese_ratio": q["vietnamese_ratio"],
            "weird_char_ratio": q["weird_char_ratio"], "line_ids": ";".join(x.get("line_global_id", "") for x in g),
        })
    return out

ABBREV_PATTERNS = [
    "v.v.", "v.v..", "tr.C.N.", "T.P.", "P.", "S.", "HV.", "q.", "sđd.", "x.", "X.",
]


def protect_sentence_abbrevs(text: str) -> tuple:
    repl = {}
    protected = text
    # protect abbreviation dots
    for i, ab in enumerate(ABBREV_PATTERNS):
        key = f"§ABBR{i}§"
        protected = protected.replace(ab, key)
        repl[key] = ab
    # protect decimals and numbered references like A.69, HV.140.
    def repl_match(m):
        key = f"§DOT{len(repl)}§"
        repl[key] = m.group(0)
        return key
    protected = re.sub(r"\b[A-Z]{1,4}\.\d+\b", repl_match, protected)
    protected = re.sub(r"\b\d+\.\d+\b", repl_match, protected)
    return protected, repl


def unprotect_sentence_abbrevs(text: str, repl: dict) -> str:
    for k, v in repl.items():
        text = text.replace(k, v)
    return text


def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    text = norm_text(paragraph_text)
    if paragraph_type == "heading":
        return []
    if not text:
        return []
    protected, repl = protect_sentence_abbrevs(text)
    out = []
    buf = []
    depth = 0
    quote_open = False
    for i, ch in enumerate(protected):
        buf.append(ch)
        if ch in "([{" or ch == "«":
            depth += 1
        elif ch in ")]}" or ch == "»":
            depth = max(0, depth - 1)
        elif ch == '"':
            quote_open = not quote_open
        if ch in ".?!…;":
            tail = "".join(buf).strip()
            nxt = protected[i + 1:i + 8]
            next_char = protected[i + 1:i + 2]
            # Semicolon split only when sentence is already very long and next token looks like a new clause.
            if ch == ";" and len(tail) < 220:
                continue
            if depth > 0 and ch != ";":
                continue
            # Avoid splitting at numbered list/footnote markers or short fragments.
            if re.search(r"\(\s*\d+\s*\)$", tail):
                continue
            if next_char and not re.match(r"\s", next_char):
                continue
            # Find first non-space after punctuation.
            rest = protected[i + 1:]
            m = re.search(r"\S", rest)
            if m:
                nchar = rest[m.start()]
                if ch != ";" and not re.match(r"[A-ZÀ-ỸĐ0-9\(\"'«]", nchar):
                    continue
            sent = unprotect_sentence_abbrevs(tail, repl)
            sent = norm_text(sent)
            if sent:
                out.append(sent)
            buf = []
    remain = unprotect_sentence_abbrevs("".join(buf).strip(), repl)
    remain = norm_text(remain)
    if remain:
        # Sentence-only output must not leak line fragments. Merge non-terminal/lowercase remainders back.
        if out and (len(remain) < 60 or begins_continuation(remain) or not ends_strong_sentenceish(remain)):
            out[-1] = norm_text(out[-1] + " " + remain)
        elif not globals().get("STRICT_SENTENCE_ONLY", True) or ends_strong_sentenceish(remain):
            out.append(remain)
    # Final validation.
    final = []
    for s in out:
        q = text_metrics(s)
        if globals().get("STRICT_SENTENCE_ONLY", True):
            if begins_continuation(s) and final:
                final[-1] = norm_text(final[-1] + " " + s)
                continue
            if not ends_strong_sentenceish(s) and q["word_count"] < 12:
                continue
        if len(s) < 12 and q["domain_heading_hits"] == 0:
            continue
        if q["quality_score"] < MIN_KEEP_SENTENCE_QUALITY - 10 and q["word_count"] < 5:
            continue
        final.append(s)
    return final


In [10]:
# ============================================================
# 8B. DNTC v3 patch: safer reflow + historical Vietnamese sentence splitting
# ============================================================
# This cell overrides a few functions from Cell 8 and appends additional
# conservative correction rules. It must run before Cell 9 processes PDFs.

LETTER_RE_SRC = "A-Za-z\u00C0-\u1EF9\u0110\u0111"
HAN_RE_SRC = "\u3400-\u9FFF"

V3_EXTRA_CORRECTION_RULES = [
    # Missing/garbled opener in the Bai Tu page: ': Doi thanh...' -> 'Trom nghi: Doi thanh...'
    ("v3_trom_nghi_missing_before_doi", "^\\s*[:;]\\s*(?=\u0110\u1eddi\\s+th\u1ea1nh|Doi\\s+thanh|\u0110\u1eddi\\s+thanh)", "Tr\u1ed9m ngh\u0129: "),
    ("v3_trom_nghi_no_diacritic", "\\bTrom\\s+nghi\\s*[:;]", "Tr\u1ed9m ngh\u0129:"),

    # Broken DNTC administrative / title compounds.
    ("v3_tong_tai", "\\bT\u1ed3ng\\s*-?\\s*t\u00e0i\\b|\\bTong\\s*-?\\s*tai\\b", "T\u1ed5ng-t\u00e0i"),
    ("v3_toan_tu", "\\bTo\u1ea3n\\s*-?\\s*tu\\b|\\bToan\\s*-?\\s*tu\\b", "To\u1ea3n-tu"),
    ("v3_quoc_su_quan_dot", "\\bQu\u1ed1c\\s*-?\\s*s\u1eed\\s*[.]\\s*Qu\u00e1n\\b", "Qu\u1ed1c-s\u1eed-qu\u00e1n"),
    ("v3_quoc_su_quan_hyphen", "\\bQu\u1ed1c\\s*-?\\s*s\u1eed\\s*-?\\s*qu\u00e1n\\b", "Qu\u1ed1c-s\u1eed-qu\u00e1n"),
    ("v3_kham_mang", "\\bKh\u00e2m\\s*-?\\s*m\u1ea1ng\\b", "Kh\u00e2m-m\u1ea1ng"),
    ("v3_dai_nam_nhat_thong_chi", "\\b\u0110\u1ea1i\\s*-\\s*Nam\\s*-\\s*Nh\u1ea5t\\s*-\\s*Th\u1ed1ng\\s*-?\\s*Ch[\u00ed\u1ec9i]\b", "\u0110\u1ea1i-Nam-Nh\u1ea5t-Th\u1ed1ng-Ch\u00ed"),

    # Hano-Vietnamese compounds and common OCR substitutions in Bai Tu pages.
    ("v3_thanh_tri", "\\bth\u1ea1nh\\s*-\\s*tr\u1ecb\\b|\\bthanh\\s*-\\s*tri\\b", "th\u1ea1nh-tr\u1ecb"),
    ("v3_xa_thu", "\\bxa\\s+th[\u01a1o]\b", "xa th\u01b0"),
    ("v3_xa_thu_han_spaced", "\\bxa\\s+th\u01b0\\s+\u8eca\\s+\u66f8\\b", "xa th\u01b0 \u8eca\u66f8"),
    ("v3_chuc_phuong", "\\bch\u1ee9c\\s*-\\s*ph\u01b0\u01a1ng\\b", "ch\u1ee9c-ph\u01b0\u01a1ng"),
    ("v3_kinh_vi", "\\bKinh\\s*-\\s*v[\u0129i]\b", "Kinh-v\u0129"),
    ("v3_xuan_thu", "\\bXu\u00e2n\\s*-\\s*thu\\b", "Xu\u00e2n-thu"),
    ("v3_trung_dung", "\\bTrung\\s*-\\s*dung\\b", "Trung-dung"),
    ("v3_nhat_thong_lower", "\\bnh\u1ea5t\\s*-\\s*th\u1ed1ng\\b", "nh\u1ea5t-th\u1ed1ng"),
    ("v3_thuy_tho", "\\bth[\u1ee7u]y\\s+th[\u1ed3o]\b", "th\u1ee7y th\u1ed5"),
    ("v3_an_ngu", "\\b\u1ea7n\\s+ng\u1ee5\\b", "\u1ea9n ng\u1ee5"),
    ("v3_dong_quy", "\\b\u0111\u1ed3ng\\s+qu[\u00edi]\b", "\u0111\u1ed3ng qu\u1ef9"),
    ("v3_tan_trinh", "\\bt\u1ea5n\\s+trinh\\b", "t\u1ea5n tr\u00ecnh"),
    ("v3_can_tau", "\\bc\u1ea7n\\s+t\u1ea5u\\b", "c\u1ea9n t\u1ea5u"),
    ("v3_phap_do", "\\bph\u00e1p\\s+\u0111\u1ed9\\b", "ph\u00e1p \u0111\u1ed9"),
]

# Append only once.
_existing_rule_ids = {r[0] for r in CORRECTION_RULES}
for _rule in V3_EXTRA_CORRECTION_RULES:
    if _rule[0] not in _existing_rule_ids:
        CORRECTION_RULES.append(_rule)

MEANINGFUL_SHORT_LINE_RE = re.compile(
    "^(?:Tr\u1ed9m\\s+ngh\u0129|X\u00e9t\\s+r\u1eb1ng|L\u1eddi\\s+r\u1eb1ng|Nay\\s+k\u00ednh\\s+t\u00e2u|C\u1ea9n\\s+t\u1ea5u|C\u1ea9n\\s+\u00e1n)\\s*[:;]?$",
    re.I,
)

DANGLING_ENDINGS = [
    "nh\u01b0", "\u1edf", "v\u1ec1", "v\u00e0", "l\u00e0", "c\u1ee7a", "\u0111\u01b0\u1ee3c", "r\u1eb1ng",
    "theo", "do", "\u0111\u1ec3", "v\u1edbi", "t\u1eeb", "c\u00e1c", "nh\u1eefng", "n\u01a1i",
    "cho n\u00ean", "b\u1edfi theo", "\u00fd n\u00f3i", "c\u00f3 c\u00e2u:", "cho \u0111\u01b0\u1ee3c:",
]

BROKEN_LINE_END_RE = re.compile(
    "(?:Qu\u1ed1c\\s*-?\\s*s\u1eed\\s*[.]?$|\u0110\u1ea1i\\s*-\\s*Nam\\s*-?$|Nh\u1ea5t\\s*-\\s*Th\u1ed1ng\\s*-?$|Xu\u00e2n\\s*-?$|Kinh\\s*-?$)",
    re.I,
)

BROKEN_LINE_START_RE = re.compile(
    "^(?:Qu\u00e1n\\b|Nam\\b|Ch[\u00ed\u1ec9i]\\b|thu\\b|v[\u0129i]\\b|th\u1ed1ng\\b|m\u1ed9t\\b)",
    re.I,
)


def is_meaningful_short_line(s: str) -> bool:
    return bool(MEANINGFUL_SHORT_LINE_RE.match(norm_text(s)))


def ends_hard_sentence(t: str) -> bool:
    s = norm_text(t)
    return bool(re.search("[.!?\u2026][\"'\u201d\u2019)\\]]*\\s*$", s))


def ends_soft_clause(t: str) -> bool:
    s = norm_text(t)
    return bool(re.search("[:;][\"'\u201d\u2019)\\]]*\\s*$", s))


def ends_dangling(t: str) -> bool:
    s = norm_text(t).lower().strip(" \t\r\n\"'\u201c\u201d\u2018\u2019()[]{}")
    if not s:
        return True
    if s.endswith(":") or s.endswith(";"):
        return True
    return any(s.endswith(x) for x in DANGLING_ENDINGS)


def begins_continuation(t: str) -> bool:
    s = norm_text(t)
    if not s:
        return False
    if re.match("^[\\s:;,\-\u2013\u2014]+", s):
        return True
    s2 = s.lstrip(" \\\"'\u201c\u201d\u2018\u2019([{<\u00ab\u00bb\-\u2013\u2014")
    return bool(s2 and re.match("^[a-z\u00e0-\u1ef9\u0111\u3400-\u9FFF]", s2))

# Override the old broad definition. Colon/semicolon are clauses, not final sentence ends.
ends_strong_sentenceish = ends_hard_sentence

_ORIG_is_line_hard_noise_v3 = is_line_hard_noise

def is_line_hard_noise(s: str) -> bool:
    if is_meaningful_short_line(s):
        return False
    return _ORIG_is_line_hard_noise_v3(s)

_ORIG_filter_line_v3 = filter_line

def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    raw = norm_text(line.get("raw_text") or line.get("text") or "")
    if raw and is_meaningful_short_line(raw):
        meta = {
            "work_id": line.get("work_id"), "pdf_path": line.get("pdf_path"), "page_number": line.get("page_number"),
            "page_idx": line.get("page_idx"), "line_id": line.get("line_global_id"), "source": line.get("source"),
        }
        fixed = apply_corrections(raw, meta, correction_log)
        q = line_quality(fixed, line.get("ocr_conf"))
        kept = dict(line)
        kept.update({
            "text": fixed,
            "line_type": "body",
            "line_quality_score": q["quality_score"],
            "line_vietnamese_ratio": q["vietnamese_ratio"],
            "line_weird_char_ratio": q["weird_char_ratio"],
            "line_ocr_text": "",
            "line_ocr_conf": None,
            "line_ocr_accepted": False,
            "filter_reasons": "kept_meaningful_short_opener",
        })
        return kept, True, ["kept_meaningful_short_opener"]
    return _ORIG_filter_line_v3(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)


def apply_dntc_reflow_corrections(text: str) -> str:
    s = norm_text(text)
    if not s:
        return s
    # Correct phrases that only become visible after joining OCR/PDF lines.
    s = re.sub("\\bQu\u1ed1c\\s*-?\\s*s\u1eed\\s*[.]\\s*Qu\u00e1n\\b", "Qu\u1ed1c-s\u1eed-qu\u00e1n", s, flags=re.I)
    s = re.sub("\\bQu\u1ed1c\\s*-?\\s*s\u1eed\\s*-?\\s*Qu\u00e1n\\b", "Qu\u1ed1c-s\u1eed-qu\u00e1n", s, flags=re.I)
    s = re.sub("\\b\u0110\u1ea1i\\s*-\\s*Nam\\s*-\\s*Nh\u1ea5t\\s*-\\s*Th\u1ed1ng\\s*-?\\s*Ch[\u00ed\u1ec9i]\\b", "\u0110\u1ea1i-Nam-Nh\u1ea5t-Th\u1ed1ng-Ch\u00ed", s, flags=re.I)
    s = re.sub("\\bT\u1ed5ng\\s*-\\s*t\u00e0i\\b", "T\u1ed5ng-t\u00e0i", s, flags=re.I)
    s = re.sub("\\bTo\u1ea3n\\s*-\\s*tu\\b", "To\u1ea3n-tu", s, flags=re.I)
    s = re.sub("\\bKh\u00e2m\\s*-\\s*m\u1ea1ng\\b", "Kh\u00e2m-m\u1ea1ng", s, flags=re.I)
    s = re.sub("\\bth\u1ea1nh\\s*-\\s*tr\u1ecb\\b", "th\u1ea1nh-tr\u1ecb", s, flags=re.I)
    s = re.sub("\\bKinh\\s*-\\s*v[\u0129i]\\b", "Kinh-v\u0129", s, flags=re.I)
    s = re.sub("\\bXu\u00e2n\\s*-\\s*thu\\b", "Xu\u00e2n-thu", s, flags=re.I)
    s = candidate_correction_only(s)
    s = re.sub("\\s+([,.;:!?])", "\\1", s)
    s = re.sub("([,.;:!?])(?=\\S)", "\\1 ", s)
    s = re.sub("\\s+", " ", s).strip()
    return s


def should_merge_lines(prev: dict, cur: dict, med_h: float, page_rect) -> bool:
    pt = norm_text(prev.get("text", ""))
    ct = norm_text(cur.get("text", ""))
    if not pt or not ct:
        return False
    if prev.get("line_type") == "heading" or cur.get("line_type") == "heading":
        return False
    if prev.get("is_footnote") != cur.get("is_footnote"):
        return False
    # Footnote blocks at bottom should remain separate; inline notes stay in paragraph by geometry.
    if re.match("^\\(?\\d+\\)|^\\d+[.)]", ct) and (cur.get("bbox", [0, 0, 0, 0])[1] > getattr(page_rect, "height", 99999) * 0.72):
        return False
    if BROKEN_LINE_END_RE.search(pt) or BROKEN_LINE_START_RE.match(ct):
        return True
    if ends_soft_clause(pt) or ends_dangling(pt):
        return True
    if not ends_hard_sentence(pt):
        return True
    if begins_continuation(ct):
        return True
    try:
        gap = cur["bbox"][1] - prev["bbox"][3]
        indent_delta = cur["bbox"][0] - prev["bbox"][0]
        if gap < med_h * 1.15 and abs(indent_delta) < med_h * 2.5:
            # Small line gap inside same text block: keep together unless previous sentence is clearly complete
            # and current line looks like a real new sentence opener.
            if not (ends_hard_sentence(pt) and re.match("^[A-Z\u00c0-\u1ef9\u0110]", ct)):
                return True
    except Exception:
        pass
    return False


def should_new_paragraph(prev: dict, cur: dict, med_h: float, page_rect) -> bool:
    if prev.get("line_type") == "heading" or cur.get("line_type") == "heading":
        return True
    if prev.get("is_footnote") != cur.get("is_footnote"):
        return True
    if should_merge_lines(prev, cur, med_h, page_rect):
        return False
    pt, ct = norm_text(prev.get("text", "")), norm_text(cur.get("text", ""))
    try:
        gap = cur["bbox"][1] - prev["bbox"][3]
        indent_delta = cur["bbox"][0] - prev["bbox"][0]
        if gap > med_h * 2.35 and ends_hard_sentence(pt) and not begins_continuation(ct):
            return True
        if indent_delta > med_h * 3.0 and len(pt) > 35 and ends_hard_sentence(pt) and not begins_continuation(ct):
            return True
    except Exception:
        pass
    return False


def join_paragraph_lines(lines: list) -> str:
    parts = []
    for r in lines:
        t = norm_text(r.get("text", ""))
        if not t:
            continue
        if not parts:
            parts.append(t)
            continue
        prev = parts[-1]
        # True word split: 'huy-' + 'en' -> 'huyen'. Keep historical hyphenated compounds otherwise.
        if re.search("[a-z\u00e0-\u1ef9\u0111]-$", prev) and re.match("^[a-z\u00e0-\u1ef9\u0111]", t):
            parts[-1] = prev[:-1] + t
        else:
            parts.append(t)
    return apply_dntc_reflow_corrections(" ".join(parts))


def looks_like_sentence_fragment(s: str) -> bool:
    s = norm_text(s)
    if not s:
        return True
    q = text_metrics(s)
    if re.match("^[\\s:;,\-\u2013\u2014]+", s):
        return True
    if ends_dangling(s):
        return True
    if begins_continuation(s) and q["word_count"] < 16:
        return True
    if not ends_hard_sentence(s) and q["word_count"] < 14:
        return True
    if CLEAR_JUNK_RE.search(s) or is_ascii_short_junk_line(s):
        return True
    return False


def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    text = apply_dntc_reflow_corrections(paragraph_text)
    if paragraph_type == "heading" or not text:
        return []
    protected, repl = protect_sentence_abbrevs(text)
    out, buf = [], []
    depth = 0
    for i, ch in enumerate(protected):
        buf.append(ch)
        if ch in "([{" or ch == "\u00ab":
            depth += 1
        elif ch in ")]}" or ch == "\u00bb":
            depth = max(0, depth - 1)
        if ch not in ".?!\u2026":
            continue
        tail = "".join(buf).strip()
        if not tail:
            continue
        if depth > 0:
            continue
        if re.search("\\(\\s*\\d+\\s*\\)\\s*$", tail):
            continue
        # Do not split if the next visible token is lowercase, Han, punctuation, or a continuation marker.
        rest = protected[i + 1:]
        m = re.search("\\S", rest)
        if m:
            nchar = rest[m.start()]
            if re.match("[a-z\u00e0-\u1ef9\u0111\u3400-\u9FFF:;,\-\u2013\u2014]", nchar):
                continue
        sent = apply_dntc_reflow_corrections(unprotect_sentence_abbrevs(tail, repl))
        if sent:
            out.append(sent)
        buf = []
    remain = apply_dntc_reflow_corrections(unprotect_sentence_abbrevs("".join(buf).strip(), repl))
    if remain:
        if out and (begins_continuation(remain) or ends_dangling(out[-1]) or looks_like_sentence_fragment(remain)):
            out[-1] = apply_dntc_reflow_corrections(out[-1] + " " + remain)
        elif not globals().get("STRICT_SENTENCE_ONLY", True) or not looks_like_sentence_fragment(remain):
            out.append(remain)
    final = []
    for s in out:
        s = apply_dntc_reflow_corrections(s)
        q = text_metrics(s)
        if final and begins_continuation(s):
            final[-1] = apply_dntc_reflow_corrections(final[-1] + " " + s)
            continue
        if globals().get("STRICT_SENTENCE_ONLY", True) and looks_like_sentence_fragment(s):
            # Do not export orphan fragments to final_sentences_only.
            continue
        if len(s) < 12 and q["domain_heading_hits"] == 0:
            continue
        if q["quality_score"] < MIN_KEEP_SENTENCE_QUALITY - 10 and q["word_count"] < 5:
            continue
        final.append(s)
    return final

print("DNTC v3 patch loaded: conservative reflow, opener preservation, no semicolon split, strict fragment blocking.")
# --- v3.1 safety override: paragraph-level corrections must not reuse heading rules ---
# Some heading rules intentionally uppercase title lines. For body paragraphs they can damage
# compounds such as Dai-Nam-Nhat-Thong-Chi, so post-join correction uses a safe local list.
_CORRECTION_RULES_SANITIZED = []
for _rid, _pat, _repl in CORRECTION_RULES:
    if isinstance(_pat, str):
        _pat = _pat.replace("\x08", r"\b")
    _CORRECTION_RULES_SANITIZED.append((_rid, _pat, _repl))
CORRECTION_RULES[:] = _CORRECTION_RULES_SANITIZED

_WB = r"\b"
DNTC_V3_SAFE_POSTJOIN_RULES = [
    (_WB + "Trom\\s+nghi\\s*[:;]", "Tr\u1ed9m ngh\u0129:"),
    ("^\\s*[:;]\\s*(?=\\u0110\\u1eddi\\s+th\\u1ea1nh|Doi\\s+thanh|\\u0110\\u1eddi\\s+thanh)", "Tr\u1ed9m ngh\u0129: "),
    (_WB + "T\\u1ed3ng\\s*-?\\s*t\\u00e0i" + _WB, "T\u1ed5ng-t\u00e0i"),
    (_WB + "Tong\\s*-?\\s*tai" + _WB, "T\u1ed5ng-t\u00e0i"),
    (_WB + "To\\u1ea3n\\s*-?\\s*tu" + _WB, "To\u1ea3n-tu"),
    (_WB + "Qu\\u1ed1c\\s*-?\\s*s\\u1eed\\s*[.]\\s*Qu\\u00e1n" + _WB, "Qu\u1ed1c-s\u1eed-qu\u00e1n"),
    (_WB + "Qu\\u1ed1c\\s*-?\\s*s\\u1eed\\s*-?\\s*Qu\\u00e1n" + _WB, "Qu\u1ed1c-s\u1eed-qu\u00e1n"),
    (_WB + "Kh\\u00e2m\\s*-?\\s*m\\u1ea1ng" + _WB, "Kh\u00e2m-m\u1ea1ng"),
    (_WB + "\\u0110\\u1ea1i\\s*-\\s*Nam\\s*-\\s*Nh\\u1ea5t\\s*-\\s*Th\\u1ed1ng\\s*-?\\s*Ch[\\u00ed\\u1ec9i]" + _WB, "\u0110\u1ea1i-Nam-Nh\u1ea5t-Th\u1ed1ng-Ch\u00ed"),
    (_WB + "th\\u1ea1nh\\s*-\\s*tr\\u1ecb" + _WB, "th\u1ea1nh-tr\u1ecb"),
    (_WB + "thanh\\s*-\\s*tri" + _WB, "th\u1ea1nh-tr\u1ecb"),
    (_WB + "xa\\s+th[\\u01a1o]" + _WB, "xa th\u01b0"),
    (_WB + "xa\\s+th\\u01b0\\s+\\u8eca\\s+\\u66f8" + _WB, "xa th\u01b0 \u8eca\u66f8"),
    (_WB + "ch\\u1ee9c\\s*-\\s*ph\\u01b0\\u01a1ng" + _WB, "ch\u1ee9c-ph\u01b0\u01a1ng"),
    (_WB + "Kinh\\s*-\\s*v[\\u0129i]" + _WB, "Kinh-v\u0129"),
    (_WB + "Xu\\u00e2n\\s*-\\s*thu" + _WB, "Xu\u00e2n-thu"),
    (_WB + "Trung\\s*-\\s*dung" + _WB, "Trung-dung"),
    (_WB + "nh\\u1ea5t\\s*-\\s*th\\u1ed1ng" + _WB, "nh\u1ea5t-th\u1ed1ng"),
    (_WB + "th[\\u1ee7u]y\\s+th[\\u1ed3o]" + _WB, "th\u1ee7y th\u1ed5"),
    (_WB + "\\u1ea7n\\s+ng\\u1ee5" + _WB, "\u1ea9n ng\u1ee5"),
    (_WB + "\\u0111\\u1ed3ng\\s+qu[\\u00edi]" + _WB, "\u0111\u1ed3ng qu\u1ef9"),
    (_WB + "c\\u1ea7n\\s+t\\u1ea5u" + _WB, "c\u1ea9n t\u1ea5u"),
]


def apply_dntc_reflow_corrections(text: str) -> str:
    s = norm_text(text)
    if not s:
        return s
    for _pat, _repl in DNTC_V3_SAFE_POSTJOIN_RULES:
        s = re.sub(_pat, _repl, s, flags=re.I)
    # Cleanup common OCR spacing around Han tokens, parentheses, and punctuation.
    s = re.sub(r"([\u3400-\u9FFF])\s+([\u3400-\u9FFF])", r"\1\2", s)
    s = re.sub(r"\(\s+", "(", s)
    s = re.sub(r"\s+\)", ")", s)
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([,.;:!?])(?=\S)", r"\1 ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

print("DNTC v3.1 safety override loaded: safe post-join corrections active.")
# --- v3.2 targeted Bai Tu paragraph repairs ---
# These rules target a recurring line-break/OCR pattern in the prefatory Bai Tu pages.
DNTC_V3_SAFE_POSTJOIN_RULES.extend([
    (r"([.!?])\s*[:;]\s*(?=\u0110\u1eddi\s+th\u1ea1nh|Doi\s+thanh|\u0110\u1eddi\s+thanh)", "\\1 Tr\u1ed9m ngh\u0129: "),
    (r"\b\u0110\u1ea1i-Nam-nh\u1ea5t-th\u1ed1ng-Ch[\u00ed\u1ec9i]\b", "\u0110\u1ea1i-Nam-Nh\u1ea5t-Th\u1ed1ng-Ch\u00ed"),
    (r"\bXu\u00e2n\s*-\s*\(4\)", "Xu\u00e2n-thu nh\u1ea5t-th\u1ed1ng (4)"),
    (r"\b\u0110[\u1ec1\u1ec3e]\s+thu\s+nh\u1ea5t-th\u1ed1ng\s+cho\s+\u0111\u01b0\u1ee3c\b", "\u0110\u1ec3 cho \u0111\u01b0\u1ee3c"),
])
print("DNTC v3.2 targeted Bai Tu repairs loaded.")




DNTC v3 patch loaded: conservative reflow, opener preservation, no semicolon split, strict fragment blocking.
DNTC v3.1 safety override loaded: safe post-join corrections active.
DNTC v3.2 targeted Bai Tu repairs loaded.


<>:95: SyntaxWarning: invalid escape sequence '\-'
<>:97: SyntaxWarning: invalid escape sequence '\-'
<>:234: SyntaxWarning: invalid escape sequence '\-'
<>:274: SyntaxWarning: invalid escape sequence '\-'
<>:95: SyntaxWarning: invalid escape sequence '\-'
<>:97: SyntaxWarning: invalid escape sequence '\-'
<>:234: SyntaxWarning: invalid escape sequence '\-'
<>:274: SyntaxWarning: invalid escape sequence '\-'
/tmp/ipykernel_16/687819039.py:95: SyntaxWarning: invalid escape sequence '\-'
  if re.match("^[\\s:;,\-\u2013\u2014]+", s):
/tmp/ipykernel_16/687819039.py:97: SyntaxWarning: invalid escape sequence '\-'
  s2 = s.lstrip(" \\\"'\u201c\u201d\u2018\u2019([{<\u00ab\u00bb\-\u2013\u2014")
/tmp/ipykernel_16/687819039.py:234: SyntaxWarning: invalid escape sequence '\-'
  if re.match("^[\\s:;,\-\u2013\u2014]+", s):
/tmp/ipykernel_16/687819039.py:274: SyntaxWarning: invalid escape sequence '\-'
  if re.match("[a-z\u00e0-\u1ef9\u0111\u3400-\u9FFF:;,\-\u2013\u2014]", nchar):


In [11]:
# ============================================================
# 8C. DNTC v4 patch: old-scan OCR numeric/domain cleanup
# ============================================================
# This cell is intentionally placed after v3 and before processing.
# It focuses on old 1960s scanned volumes such as 14_15 where PDF text layer is empty
# and Tesseract is used. The rules are conservative and context-bound.

# Use a little more resolution for old Vietnamese typefaces. Slower, but safer for full quality.
try:
    PAGE_OCR_DPI = max(int(PAGE_OCR_DPI), 320)
except Exception:
    PAGE_OCR_DPI = 320
OCR_DPI = PAGE_OCR_DPI

V4_EXTRA_CORRECTION_RULES = [
    # Remove isolated OCR/copyright-like markers explicitly requested by reviewer.
    ("v4_drop_rc_marker_inline", r"\s*\(([RC])\)\s*", " "),

    # Old scan headings and administrative parentheticals.
    ("v4_luong_son_dot", r"\bLƯƠNG\s*[.]\s*SƠN\b|\bLUONG\s*[.]\s*SON\b", "LƯƠNG-SƠN"),
    ("v4_thanh_chuong_ascii", r"\bTHANH\s*-\s*CHU['’]?ONG\b|\bTHANH\s*-\s*CHUONG\b", "THANH-CHƯƠNG"),
    ("v4_huyen_heading_ascii", r"^\s*[|]?\s*HUYEN\b", "HUYỆN"),
    ("v4_do_phu_kiem_ly", r"\((?:do|đo)\s+ph[ảa]\s+ki[ée]m\s*-\s*l[yý]\)", "(do phủ kiêm-lý)"),
    ("v4_do_phu_thong_hat", r"\((?:do|đo)\s+(?:ph[ảa]|phi)\s+th[oôd]ng\s*-\s*(?:h[ạa]?t|hgt)\)", "(do phủ thống-hạt)"),
    ("v4_thong_hat_broken", r"\bthống\s*[.]\s*h[ạa]t\b|\bthống\s*-\s*hgt\b|\bth[oôd]ng\s*-\s*hgt\b", "thống-hạt"),
    ("v4_kiem_ly", r"\bki[ée]m\s*-\s*l[yý]\b", "kiêm-lý"),

    # Common old-scan OCR substitutions in Nghệ An / Thanh Hóa volumes.
    ("v4_tong", r"\btồng\b", "tổng"),
    ("v4_giap", r"\bgiấp\b", "giáp"),
    ("v4_dam_typo", r"\bđặm\b", "dặm"),
    ("v4_nam_reign", r"\b(?:Nim|Nam)\s+(?=(?:Minh|Tự|Thành|Gia|Đồng|Thiệu|Kiến|Duy)\s*-?\s*[A-ZÀ-ỸĐ])", "Năm "),
    ("v4_huyen_ly", r"\bHuyện\s*[.]\s*l[yỵi]\b", "Huyện-lỵ"),
    ("v4_do_luong", r"\bĐô\s*[.]\s*Lương\b", "Đô-Lương"),
    ("v4_anh_do", r"\bAnh\s*[.]\s*Đô\b", "Anh-Đô"),
    ("v4_dong_khanh", r"\bĐồng\s*[.]\s*Khánh\b", "Đồng-Khánh"),
    ("v4_thanh_thai", r"\bThành\s*[.]\s*Thái\b", "Thành-Thái"),
    ("v4_nnam", r"\bN+nam\b", "Nam"),
    ("v4_la_nam", r"\bLaNam\b", "La-Nam"),
    ("v4_phia_dong_nam_phu", r"\bphía\s+đông\s+nam\s+ph[úu]\b", "phía đông nam phủ"),
    ("v4_doi_context", r"\bđồi(?=\s+(?:là|làm|tên|thuộc|lệ|sang|gọi|huyện|phủ))", "đổi"),
    ("v4_da_bo_di", r"\bđã\s+bỏ\s+d[iì]\b", "đã bỏ đi"),
    ("v4_tri_huyen", r"\bTri\s*-\s*huyện\b", "Tri huyện"),

    # Tax / census units and mojibake in numeric pages.
    ("v4_thue_mojibake", r"(?<!\w)thu[€éêể](?!\w)", "thuế"),
    ("v4_dien_tho", r"\bđiền\s+th[ồo]\b|\bdin\s+thd\b", "điền thổ"),
    ("v4_bac_thue", r"\bbac\s+thu[eéế]\b", "bạc thuế"),
    ("v4_mau_digit", r"\bm4u\b", "mẫu"),
    ("v4_cong", r"\bc[ée]ng\b", "cộng"),
    ("v4_moi_dinh", r"\bméi\s*dinh\b|\bméidinh\b", "mới định"),
    ("v4_thang_unit", r"\bthing\b", "thăng"),
    ("v4_hap_unit", r"\bhap\b", "hạp"),
    ("v4_thuoc_unit", r"\bthirgc\b|\bthugc\b", "thược"),
    ("v4_lai_phu_nap", r"\bLai\s+phy\s+nap\b", "Lại phụ nạp"),
]

_existing_rule_ids = {r[0] for r in CORRECTION_RULES}
for _rule in V4_EXTRA_CORRECTION_RULES:
    if _rule[0] not in _existing_rule_ids:
        CORRECTION_RULES.append(_rule)

DNTC_V4_SAFE_POSTJOIN_RULES = [
    # Remove reviewer-reported isolated markers.
    (r"\s*\(([RC])\)\s*", " "),

    # Clean page/line OCR debris.
    (r"^\s*[|]\s*", ""),
    (r"\s+[|]\s*", " "),
    (r"\s+_\s*", " "),
    (r"\s+'(?=\s*[ĐA-ZÀ-Ỹđa-zà-ỹ])", " "),
    (r"\s+", " "),

    # Headings/domain forms.
    (r"\bHUYEN\b", "HUYỆN"),
    (r"\bLƯƠNG\s*[.]\s*SƠN\b|\bLUONG\s*[.]\s*SON\b", "LƯƠNG-SƠN"),
    (r"\bTHANH\s*-\s*CHU['’]?ONG\b|\bTHANH\s*-\s*CHUONG\b", "THANH-CHƯƠNG"),
    (r"\((?:do|đo)\s+ph[ảa]\s+ki[ée]m\s*-\s*l[yý]\)", "(do phủ kiêm-lý)"),
    (r"\((?:do|đo)\s+(?:ph[ảa]|phi)\s+th[oôd]ng\s*-\s*(?:h[ạa]?t|hgt)\)", "(do phủ thống-hạt)"),
    (r"\bthống\s*[.]\s*h[ạa]t\b|\bthống\s*-\s*hgt\b|\bth[oôd]ng\s*-\s*hgt\b", "thống-hạt"),
    (r"\bki[ée]m\s*-\s*l[yý]\b", "kiêm-lý"),

    # Common OCR words.
    (r"\btồng\b", "tổng"),
    (r"\bgiấp\b", "giáp"),
    (r"\bđặm\b", "dặm"),
    (r"\b(?:Nim|Nam)\s+(?=(?:Minh|Tự|Thành|Gia|Đồng|Thiệu|Kiến|Duy)\s*-?\s*[A-ZÀ-ỸĐ])", "Năm "),
    (r"\bHuyện\s*[.]\s*l[yỵi]\b", "Huyện-lỵ"),
    (r"\bĐô\s*[.]\s*Lương\b", "Đô-Lương"),
    (r"\bAnh\s*[.]\s*Đô\b", "Anh-Đô"),
    (r"\bĐồng\s*[.]\s*Khánh\b", "Đồng-Khánh"),
    (r"\bThành\s*[.]\s*Thái\b", "Thành-Thái"),
    (r"\bN+nam\b", "Nam"),
    (r"\bLaNam\b", "La-Nam"),
    (r"\bphía\s+đông\s+nam\s+ph[úu]\b", "phía đông nam phủ"),
    (r"\bđồi(?=\s+(?:là|làm|tên|thuộc|lệ|sang|gọi|huyện|phủ))", "đổi"),
    (r"\bđã\s+bỏ\s+d[iì]\b", "đã bỏ đi"),
    (r"\bTri\s*-\s*huyện\b", "Tri huyện"),

    # Specific geography line break: "Đông. _ giáp..." is one clause, not a sentence.
    (r"\b(Đông|Tây|Nam|Bắc)\s*[.]\s*[_']?\s*(?=gi[áa]p\b)", r"\1 "),
    (r"\b(cách nhau\s+\d+\s+dặm)\s*[.]\s+(nam\s+bắc)\b", r"\1, \2"),

    # Tax / census pages.
    (r"(?<!\w)thu[€éêể](?!\w)", "thuế"),
    (r"\bđiền\s+th[ồo]\b|\bdin\s+thd\b", "điền thổ"),
    (r"\bbac\s+thu[eéế]\b", "bạc thuế"),
    (r"\bm4u\b", "mẫu"),
    (r"\bc[ée]ng\b", "cộng"),
    (r"\bméi\s*dinh\b|\bméidinh\b", "mới định"),
    (r"\bthing\b", "thăng"),
    (r"\bhap\b", "hạp"),
    (r"\bthirgc\b|\bthugc\b", "thược"),
    (r"\bLai\s+phy\s+nap\b", "Lại phụ nạp"),
]


def fix_old_scan_numeric_ocr(s: str) -> str:
    """Contextual numeric cleanup for old Tesseract OCR.
    It only touches number-like tokens near units/reign-year contexts to avoid
    rewriting normal Vietnamese words.
    """
    if not s:
        return s

    # Common year / reign OCR.
    s = re.sub(r"\br8so\b", "1850", s, flags=re.I)
    s = re.sub(r"\br886\b", "1886", s, flags=re.I)
    s = re.sub(r"\b18g8\b", "1898", s, flags=re.I)
    s = re.sub(r"\br1oo\b", "1100", s, flags=re.I)
    s = re.sub(r"\b(th[ứưửu]\s+)ro\b", r"\g<1>10", s, flags=re.I)
    s = re.sub(r"\b(th[ứưửu]\s+)rz\b", r"\g<1>12", s, flags=re.I)
    s = re.sub(r"\b(th[ứưửu]\s+)a1\b", r"\g<1>21", s, flags=re.I)

    # Standalone OCR for 9/5 in month expressions.
    s = re.sub(r"\b(mồng|tháng)\s+o\b", r"\1 9", s, flags=re.I)
    s = re.sub(r"\b(mồng|tháng)\s+s\b", r"\1 5", s, flags=re.I)

    # Leading o before a digit in distance is usually 9: o4 dặm -> 94 dặm.
    s = re.sub(r"\bo(?=\d+\s+dặm\b)", "9", s, flags=re.I)

    # Standalone 's' before administrative/statistical units is usually 5.
    s = re.sub(r"\bs(?=\s+(?:tổng|sào|thôn|xã|mẫu|người|lạng|thước|đồng|tiền|quan|phần|huyện)\b)", "5", s, flags=re.I)

    # Confusable trailing letters inside numeric spans before units.
    unit = r"(?:dặm|tổng|sào|thôn|xã|mẫu|người|lạng|thước|đồng|tiền|quan|phần|huyện|đ|đồng)"
    s = re.sub(r"(?<=\d)s(?=\s+" + unit + r"\b)", "5", s, flags=re.I)
    s = re.sub(r"(?<=\d)o(?=\s+" + unit + r"\b)", "0", s, flags=re.I)
    s = re.sub(r"(?<=\d)g(?=\d|\s*" + unit + r"\b)", "9", s, flags=re.I)

    # General number-like tokens containing OCR letters and at least one digit,
    # only if followed by a known unit or currency marker.
    def _fix_token(m):
        tok = m.group(1)
        table = str.maketrans({
            "o": "0", "O": "0",
            "s": "5", "S": "5",
            "g": "9", "q": "9",
            "r": "1", "l": "1", "I": "1",
            "z": "2", "Z": "2",
        })
        return tok.translate(table)

    s = re.sub(r"\b([0-9osgqrIlzZ]{2,})(?=\s*(?:đ|đồng|người|mẫu|dặm|quan|lạng|thăng|hạp|thược|sào|thước|loát)\b)", _fix_token, s, flags=re.I)

    # Add missing space: 4tổng -> 4 tổng.
    s = re.sub(r"(\d)(?=(?:tổng|xã|thôn|mẫu|sào|thước|dặm)\b)", r"\1 ", s, flags=re.I)

    # OCR sometimes adds an extra letter to "2 huyện": "2a huyện".
    s = re.sub(r"\b2a\s+huyện\b", "2 huyện", s, flags=re.I)

    # Merge thousands / decimal groups wrongly split by sentence punctuation.
    s = re.sub(r"\b(\d{1,3})\.\s+(\d{3})(?=\s+(?:người|mẫu|quan|lạng|đồng|đ|thăng|hạp|thược|sào|thước|loát)\b)", r"\1.\2", s)
    s = re.sub(r"\b(\d{1,3})\s*,\s+(\d{3})(?=\s*(?:người|mẫu|quan|lạng|đồng|đ|thăng|hạp|thược|sào|thước|loát)\b)", r"\1,\2", s)

    return s


_ORIG_apply_dntc_reflow_corrections_v4 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v4(text)
    if not s:
        return s
    for _pat, _repl in DNTC_V4_SAFE_POSTJOIN_RULES:
        s = re.sub(_pat, _repl, s, flags=re.I)
    s = fix_old_scan_numeric_ocr(s)

    # Some rules become active only after numeric OCR has been repaired.
    s = re.sub(r"\b(cách nhau\s+\d+\s+dặm)\s*[.]\s+(nam\s+bắc)\b", r"\1, \2", s, flags=re.I)

    # Cleanup spacing around punctuation and hyphenated compounds after numeric/domain rules.
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    # Do not split numeric groups like 37.996 or 44.237.
    s = re.sub(r"([,;:!?])(?=\S)", r"\1 ", s)
    s = re.sub(r"(?<!\d)\.(?=\S)", ". ", s)
    s = re.sub(r"(?<=\d)\.(?=[^\d\s])", ". ", s)
    s = re.sub(r"\s*-\s*", "-", s)
    # Restore spaces around sentence-level dashes if any were over-normalized is intentionally avoided;
    # DNTC old text uses many compound hyphens.
    s = re.sub(r"\s+", " ", s).strip()
    return s


_ORIG_is_heading_text_v4 = is_heading_text

def is_heading_text(s: str) -> bool:
    s0 = apply_dntc_reflow_corrections(norm_text(s)).strip(" |")
    if _ORIG_is_heading_text_v4(s0):
        return True
    su = s0.upper()
    # Old scan subheadings often contain a lowercase explanatory parenthetical,
    # so uppercase ratio alone misses them.
    if re.match(r"^HUYỆN\s+[A-ZÀ-ỸĐ0-9 .'\-]+(?:\s*\(|$)", su):
        return True
    if re.match(r"^(PHỦ|TỔNG|XÃ|THÀNH|NÚI|SÔNG|ĐỀN|CHÙA|MIẾU)\s+[A-ZÀ-ỸĐ0-9 .'\-]+(?:\s*\(|$)", su) and len(s0) <= 120:
        return True
    return False


_ORIG_is_line_hard_noise_v4 = is_line_hard_noise

def is_line_hard_noise(s: str) -> bool:
    s0 = norm_text(s)
    if re.fullmatch(r"\(?[RC]\)?", s0.strip(), flags=re.I):
        return True
    return _ORIG_is_line_hard_noise_v4(s)


_ORIG_filter_line_v4 = filter_line

def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    kept, ok, reasons = _ORIG_filter_line_v4(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
    if not ok or kept is None:
        return kept, ok, reasons
    fixed = apply_dntc_reflow_corrections(kept.get("text", ""))
    if not fixed or re.fullmatch(r"\(?[RC]\)?", fixed.strip(), flags=re.I):
        return None, False, (reasons or []) + ["drop_rc_marker"]
    kept["text"] = fixed
    q = line_quality(fixed, kept.get("ocr_conf"))
    kept["line_quality_score"] = q["quality_score"]
    kept["line_vietnamese_ratio"] = q["vietnamese_ratio"]
    kept["line_weird_char_ratio"] = q["weird_char_ratio"]
    if is_heading_text(fixed):
        kept["line_type"] = "heading"
    return kept, True, reasons


# Continuation should treat OCR debris such as "_ giáp" or "' Đây" as a continuation.
def begins_continuation(t: str) -> bool:
    s = norm_text(t)
    if not s:
        return False
    if re.match(r"^[\s:;,._'\"|\\/\-\–\—]+", s):
        return True
    s2 = s.lstrip(" \"'“”‘’([{<«»|-–—_.")
    return bool(s2 and re.match(r"^[a-zà-ỹđ\u3400-\u9FFF]", s2))


print("DNTC v4 patch loaded: old-scan numeric cleanup, HUYỆN subheading detection, (R)/(C) removal, OCR_DPI=", PAGE_OCR_DPI)

DNTC v4 patch loaded: old-scan numeric cleanup, HUYỆN subheading detection, (R)/(C) removal, OCR_DPI= 320


In [12]:

# ============================================================
# 8D. DNTC v5 patch: preserve short meaningful lines + stronger sentence boundaries
# ============================================================
# This patch fixes two classes observed in 01.pdf page 52-53:
# - short meaningful lines such as "bệ có 5 cấp." were dropped as low quality;
# - structural starts like "Từng thứ ba:" and cross-page continuations must be handled.

MEANINGFUL_SHORT_BODY_RE = re.compile(
    r"\b(?:bệ|cấp|án|đàn|móng|trụ|cột|trần\s+thiết|đại\s*-?\s*thứ|"
    r"lò|hầm|gạch|thước|tấc|phân|trượng|mặt|tường|cửa|hàng|lọng|tàn|"
    r"phía|đông|tây|nam|bắc|miếu|điện|thần|khố|trù)\b",
    re.I,
)
OPEN_PAREN_CONTEXT_RE = re.compile(r"^\(?\s*(?:nguyên|xem|tục|nay|cũ|tức|theo)\b", re.I)
STRUCTURAL_SENTENCE_START_RE = re.compile(
    r"^(?:Từng\s+thứ\s+(?:nhất|nhì|ba|tư|bốn|năm|\d+)|"
    r"Đàn\s+(?:vuông|tròn|chế)|Ba\s+từng|Bốn\s+mặt|Ở\s+góc|Phía\s+(?:tả|hữu|đông|tây|nam|bắc)|"
    r"Án\s+(?:tả|hữu|chính)|Hữu\s+(?:nhất|nhị|tam|tứ|tử)|Tả\s+(?:nhất|nhị|tam|tứ|tử)|"
    r"Cẩn\s+án|Lại\s+xét|Năm\s+[A-ZÀ-ỸĐ])\b",
    re.I,
)

_ORIG_filter_line_v5 = filter_line

def _keep_short_meaningful_line(raw_line: dict, correction_log: list):
    raw = norm_text(raw_line.get("raw_text") or raw_line.get("text") or "")
    if not raw:
        return None, False, []
    fixed = apply_corrections(raw, {
        "work_id": raw_line.get("work_id"), "pdf_path": raw_line.get("pdf_path"),
        "page_number": raw_line.get("page_number"), "page_idx": raw_line.get("page_idx"),
        "line_id": raw_line.get("line_global_id"), "source": raw_line.get("source"),
    }, correction_log)
    fixed = apply_dntc_reflow_corrections(fixed)
    q = line_quality(fixed, raw_line.get("ocr_conf"))
    if q["weird_char_ratio"] > 0.04 or q["letter_count"] < 4:
        return None, False, []
    if CLEAR_JUNK_RE.search(fixed) or LIBRARY_NOISE_RE.search(fixed) or is_ascii_short_junk_line(fixed):
        return None, False, []
    looks_meaningful = bool(MEANINGFUL_SHORT_BODY_RE.search(fixed) or OPEN_PAREN_CONTEXT_RE.search(fixed))
    # Keep only short-ish fragments that are semantically useful in layout descriptions.
    if looks_meaningful and q["quality_score"] >= 22.0 and len(fixed) <= 90:
        line_type = "heading" if is_heading_text(fixed) else "body"
        kept = dict(raw_line)
        kept.update({
            "text": fixed,
            "line_type": line_type,
            "line_quality_score": q["quality_score"],
            "line_vietnamese_ratio": q["vietnamese_ratio"],
            "line_weird_char_ratio": q["weird_char_ratio"],
            "line_ocr_text": "",
            "line_ocr_conf": None,
            "line_ocr_accepted": False,
            "filter_reasons": "kept_short_meaningful_body_line_v5",
        })
        return kept, True, ["kept_short_meaningful_body_line_v5"]
    return None, False, []


def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    kept, ok, reasons = _ORIG_filter_line_v5(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
    if ok and kept is not None:
        return kept, ok, reasons
    # Recover short but meaningful lines which the old score-only gate dropped.
    kept2, ok2, reasons2 = _keep_short_meaningful_line(line, correction_log)
    if ok2:
        return kept2, True, reasons2
    return kept, ok, reasons


_ORIG_apply_dntc_reflow_corrections_v5 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v5(text)
    if not s:
        return s
    # If a short dropped/missing line caused "..., mỗi Từng thứ ba", remove dangling "mỗi" and start a new sentence.
    s = re.sub(r"(bệ\s+đi\s+ra),\s*mỗi\s+(?=Từng\s+thứ\s+(?:nhất|nhì|ba|tư|bốn|năm|\d+)\b)", r"\1. ", s, flags=re.I)
    # Layout enumerators are sentence starts; ensure a boundary before them if previous clause already ended.
    s = re.sub(r"([.!?])\s+(?=(?:Từng\s+thứ|Đàn\s+vuông|Ba\s+từng|Ở\s+góc|Phía\s+(?:tả|hữu)|Án\s+(?:tả|hữu)|Cẩn\s+án)\b)", r"\1 ", s, flags=re.I)
    # Normalize spacing one more time.
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([,;:!?])(?=\S)", r"\1 ", s)
    s = re.sub(r"(?<!\d)\.(?=\S)", ". ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


_ORIG_split_sentences_vietnamese_v5 = split_sentences_vietnamese

def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    text = apply_dntc_reflow_corrections(paragraph_text)
    if paragraph_type == "heading" or not text:
        return []
    # Add a virtual sentence boundary before structural starts when a page/layout join removed it.
    text = re.sub(r"(?<=[.!?])\s+(?=(?:Từng\s+thứ|Đàn\s+vuông|Ba\s+từng|Ở\s+góc|Phía\s+(?:tả|hữu)|Án\s+(?:tả|hữu)|Cẩn\s+án)\b)", "\n", text, flags=re.I)
    parts = [p.strip() for p in text.split("\n") if p.strip()]
    out = []
    for part in parts:
        out.extend(_ORIG_split_sentences_vietnamese_v5(part, paragraph_type))
    # Do not merge independent structural sentences back into previous item.
    final = []
    for s in out:
        s = apply_dntc_reflow_corrections(s)
        if not s:
            continue
        if final and begins_continuation(s) and not STRUCTURAL_SENTENCE_START_RE.match(s):
            final[-1] = apply_dntc_reflow_corrections(final[-1] + " " + s)
        else:
            final.append(s)
    return final

print("DNTC v5 pre-process patch loaded: short meaningful lines preserved, structural sentence starts protected.")


DNTC v5 pre-process patch loaded: short meaningful lines preserved, structural sentence starts protected.


In [13]:

# ============================================================
# 8E. DNTC v6 patch: strict heading classifier + strict sentence-only export
# ============================================================
# v5 fixed several cross-page stitches, but review of the new output showed a new
# root cause: heading detection was too broad and sometimes marked body lines as
# headings (e.g. "phủ phủ Lâm Bình...", "thành đến phía tả cung Khánh...").
# Because heading paragraphs are excluded from sentence export, those false headings
# created broken/non-terminal sentences and pages with final_lines but no final_sentences.

DNTC_V6_ADMIN_HEAD_RE = re.compile(
    r"^(?:HUYỆN|HUYEN|PHỦ|PHU|TỔNG|TONG|XÃ|XA|THÀNH|THANH|NÚI|NUI|SÔNG|SONG|"
    r"ĐỀN|DEN|CHÙA|CHUA|MIẾU|MIEU|LĂNG|LANG|CẦU|CAU|ĐÒ|DO|TỈNH|TINH)\b",
    re.I,
)
DNTC_V6_DOMAIN_HEADING_RE = re.compile(
    r"^(?:DỰNG ĐẶT|DUNG DAT|DIÊN CÁCH|DIEN CACH|PHONG TỤC|PHONG TUC|THÀNH TRÌ|THANH TRI|"
    r"NÚI SÔNG|NUI SONG|SÔNG NGÒI|SONG NGOI|ĐẦM AO|DAM AO|CẦU CỐNG|CAU CONG|"
    r"QUAN TẤN|QUAN TAN|ĐÊ ĐẬP|DE DAP|CHỢ QUÁN|CHO QUAN|TRƯỜNG HỌC|TRUONG HOC|"
    r"TỪ MIẾU|TU MIEU|CHÙA QUÁN|CHUA QUAN|LĂNG MỘ|LANG MO|CỔ TÍCH|CO TICH|"
    r"NHÂN VẬT|NHAN VAT|THỔ SẢN|THO SAN|HỘ KHẨU|HO KHAU|ĐIỀN THỔ|DIEN THO|"
    r"THUẾ LỆ|THUE LE|MỤC LỤC|MUC LUC|BÀI TỰ|BAI TU|LỜI NÓI ĐẦU|LOI NOI DAU)\b",
    re.I,
)
DNTC_V6_BODY_WORDS_IN_HEADING_RE = re.compile(
    r"\b(?:cách|giáp|thuộc|đời|năm|là|ở|có|cho|đến|từ|theo|gọi|đổi|lãnh|"
    r"phía|đông|tây|nam|bắc|huyện|phủ|xã|thôn|tổng|dặm)\b",
    re.I,
)

def _v6_upper_ratio(s: str) -> float:
    letters = LETTERS_RE.findall(norm_text(s))
    if not letters:
        return 0.0
    return sum(1 for ch in letters if ch.upper() == ch) / len(letters)

_ORIG_apply_dntc_reflow_corrections_v6 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v6(text)
    if not s:
        return s

    # Remove duplicate opener injected/preserved around Bai Tu pages.
    s = re.sub(r"(Trộm\s+nghĩ\s*:\s*[^.!?]{0,140}?\b(?:車書)?\s*)Trộm\s+nghĩ\s*(\(\s*1\s*\)\s*)?", r"\1\2", s, flags=re.I)
    s = re.sub(r"thiên\s*-?\s*Trộm\s+nghĩ\s+hạ", "thiên hạ", s, flags=re.I)

    # Old-scan administrative heading corrections.
    s = re.sub(r"\bLƯƠNG[.]SƠN\b", "LƯƠNG-SƠN", s, flags=re.I)
    s = re.sub(r"\bTHANH\s*-\s*CHU['’]?O?NG\b", "THANH-CHƯƠNG", s, flags=re.I)
    s = re.sub(r"\((?:do|đo)\s+(?:phi|phả|phd|phe)\s+ki[ée]m\s*-\s*l[yý]\)", "(do phủ kiêm-lý)", s, flags=re.I)
    s = re.sub(r"\((?:do|đo)\s+(?:phi|phả|phd|phe)\s+(?:th[eé]ng|th[oô]ng|hống)\s*-\s*(?:hgt|hạt)\)", "(do phủ thống-hạt)", s, flags=re.I)
    s = re.sub(r"\b(?:th[eé]ng|hống)\s*-\s*h(?:gt|ạt)\b", "thống-hạt", s, flags=re.I)
    s = re.sub(r"\bphd\b", "phủ", s, flags=re.I)
    s = re.sub(r"\bphi\s+(?=Nam\s+Linh|Tân\s+Bình|kiêm|thống)", "phủ ", s, flags=re.I)
    s = re.sub(r"\bph[ảa]\s+(?=kiêm|thống)", "phủ ", s, flags=re.I)

    # Old-scan numeric OCR corrections, context-limited.
    s = re.sub(r"\b(?:r8so|18so)\b", "1850", s, flags=re.I)
    s = re.sub(r"\br886\b", "1886", s, flags=re.I)
    s = re.sub(r"\br8og\b", "1899", s, flags=re.I)
    s = re.sub(r"\bthứ\s+1o\b", "thứ 10", s, flags=re.I)
    s = re.sub(r"\bthứ\s+ro\b", "thứ 10", s, flags=re.I)
    s = re.sub(r"\bthứ\s+r1\b", "thứ 11", s, flags=re.I)
    s = re.sub(r"\bthứ\s+rz\b", "thứ 12", s, flags=re.I)
    s = re.sub(r"\bthứ\s+r4\b", "thứ 14", s, flags=re.I)
    s = re.sub(r"\bthứ\s+r8\b", "thứ 18", s, flags=re.I)
    s = re.sub(r"\b([0-9]+)o(?=\s+(?:người|mẫu|quan|đồng|dặm|xã|thôn|tổng|hộc|thăng|hạp|thước))", r"\g<1>0", s, flags=re.I)
    s = re.sub(r"\bo4(?=\s+dặm)", "94", s, flags=re.I)
    s = re.sub(r"\b8o(?=\s+dặm)", "80", s, flags=re.I)
    s = re.sub(r"\b8s(?=\s+dặm)", "85", s, flags=re.I)
    s = re.sub(r"\bs8(?=\s+dặm)", "58", s, flags=re.I)
    s = re.sub(r"\bs2(?=\s+dặm)", "52", s, flags=re.I)
    s = re.sub(r"\bss(?=\s+dặm)", "55", s, flags=re.I)
    s = re.sub(r"\bso(?=\s+dặm)", "50", s, flags=re.I)
    s = re.sub(r"\b6;\s*xã\b", "67 xã", s, flags=re.I)
    s = re.sub(r"\b([0-9]+)\s*;\s*(?=xã|thôn)", r"\1", s, flags=re.I)
    s = re.sub(r"\b([0-9])\s*tồng\b", r"\1 tổng", s, flags=re.I)
    s = re.sub(r"\b([0-9])tổng\b", r"\1 tổng", s, flags=re.I)
    s = re.sub(r"\bs\s+(?:tồng|tổng)\b", "5 tổng", s, flags=re.I)
    s = re.sub(r"\bdim\b", "dặm", s, flags=re.I)
    s = re.sub(r"\bNaylnh\b", "Nay lĩnh", s, flags=re.I)
    s = re.sub(r"\blĩnh\s+s\s+tổng\b", "lĩnh 5 tổng", s, flags=re.I)
    s = re.sub(r"\bDến\b", "Đến", s)
    s = re.sub(r"\blệthuộc\b", "lệ thuộc", s, flags=re.I)
    s = re.sub(r"\blàhuyện\b", "là huyện", s, flags=re.I)
    s = re.sub(r"\bTrihuyện\b", "Tri huyện", s)
    s = re.sub(r"\bđồilà\b", "đổi là", s, flags=re.I)
    s = re.sub(r"\b(lại|mới|sau|rồi)\s+đồi\b", r"\1 đổi", s, flags=re.I)

    # Cleanup OCR debris around headings and sentence starts.
    s = re.sub(r"^[\s_—\-|lI1]+(?=HUYỆN|HUYEN|PHỦ|PHU|TỈNH|TINH|PHẦN|PHAN)", "", s, flags=re.I)
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([,;:!?])(?=\S)", r"\1 ", s)
    s = re.sub(r"(?<!\d)\.(?=\S)", ". ", s)
    s = re.sub(r"\s*-\s*", "-", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

_ORIG_is_heading_text_v6 = is_heading_text

def is_heading_text(s: str) -> bool:
    s0 = apply_dntc_reflow_corrections(norm_text(s)).strip()
    core = s0.strip(" |_\u2014\u2013-")
    if not core or len(core) > 130:
        return False

    # Critical fix: never promote lowercase sentence-like lines to headings.
    # Use Unicode-aware islower(); regex ranges such as à-ỹ also include some uppercase codepoints.
    first_alpha = next((ch for ch in core if ch.isalpha()), "")
    if first_alpha and first_alpha.islower():
        return False

    upper_ratio = _v6_upper_ratio(core)

    # Lines with normal sentence punctuation are not headings unless they are almost all uppercase.
    if re.search(r"[,.!?;]", core) and upper_ratio < 0.85:
        return False

    # Canonical all-caps/admin headings. Parenthetical "(do phủ ...)" is allowed.
    if DNTC_V6_ADMIN_HEAD_RE.match(core):
        head_part = core.split("(")[0].strip()
        if len(core) <= 120 and (upper_ratio >= 0.55 or re.match(r"^(HUYỆN|HUYEN|PHỦ|PHU)\s+[A-ZÀ-ỸĐ0-9 .'’-]+", head_part)):
            # Reject body-looking lines that start with an admin word but continue as prose.
            if not DNTC_V6_BODY_WORDS_IN_HEADING_RE.search(head_part.replace("HUYỆN", "").replace("PHỦ", "")):
                return True

    if DNTC_V6_DOMAIN_HEADING_RE.match(core) and len(core) <= 110 and upper_ratio >= 0.55:
        return True

    # General old heading: short and mostly uppercase.
    if len(core) <= 85 and upper_ratio >= 0.82 and not LIBRARY_NOISE_RE.search(core):
        return True

    return False

_ORIG_filter_line_v6 = filter_line

def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    kept, ok, reasons = _ORIG_filter_line_v6(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
    if not ok or kept is None:
        return kept, ok, reasons
    fixed = apply_dntc_reflow_corrections(kept.get("text", ""))
    if not fixed:
        return None, False, (reasons or []) + ["empty_after_v6_correction"]
    kept["text"] = fixed
    # Critical fix: reclassify both ways, not only body -> heading.
    kept["line_type"] = "heading" if is_heading_text(fixed) else "body"
    q = line_quality(fixed, kept.get("ocr_conf"))
    kept["line_quality_score"] = q["quality_score"]
    kept["line_vietnamese_ratio"] = q["vietnamese_ratio"]
    kept["line_weird_char_ratio"] = q["weird_char_ratio"]
    return kept, True, reasons

_ORIG_split_sentences_vietnamese_v6 = split_sentences_vietnamese

def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    if paragraph_type == "heading":
        return []
    out = _ORIG_split_sentences_vietnamese_v6(paragraph_text, paragraph_type)
    final = []
    for s in out:
        s = apply_dntc_reflow_corrections(s)
        if not s:
            continue
        q = text_metrics(s)
        # final_sentences_only must be strict. Non-terminal fragments remain in paragraphs/text,
        # but should not pollute the sentence-only CSV.
        if globals().get("STRICT_SENTENCE_ONLY", True):
            if not ends_strong_sentenceish(s):
                continue
            if looks_like_sentence_fragment(s) and q["word_count"] < 18:
                continue
        final.append(s)
    return final

print("DNTC v6 patch loaded: strict heading classifier, stricter sentence-only validation, paragraph text export recommended.")


DNTC v6 patch loaded: strict heading classifier, stricter sentence-only validation, paragraph text export recommended.


In [14]:
# ============================================================
# 8F. DNTC v7 patch: backmatter + heading/number cleanup (no Han preservation)
# ============================================================
# This keeps the useful v7 fixes for Vietnamese extraction:
#   1) Publisher/backmatter catalog pages/lines are dropped more aggressively.
#   2) Old-scan administrative headings are detected more accurately.
#   3) Old-scan numeric/OCR artifacts are normalized in Vietnamese text.
#
# Han text-layer preservation is intentionally NOT enabled in this variant.
# Pages/lines are still judged by the normal Vietnamese-oriented pipeline.

DNTC_V7_BACKMATTER_RE = re.compile(
    r"(?:V\.?\s*H\.?\s*T\.?\s*T\.?|Vietnam\s+Culture\s+Series|HIGHER\s+EDUCATION\s+IN\s+THE\s+REPUBLIC|"
    r"LA\s+LITTERATURE\s+VIETNAMIENNE|INTRODUCTION\s+TO\s+VIETNAMESE|"
    r"do\s+Nha\s+V[ăa]n\s*-\s*H[oó]a.*xu[aấ]t\s+b[aả]n|"
    r"c[oó]\s+b[aá]n\s+t[aạ]i\s+c[aá]c\s+n[oơ]i|"
    r"Nh[ữu]ng\s+t[aậ]p\s+V[ĂA]N\s*H[ÓO]A|"
    r"M[ỤU]C\s*L[ỤU]C|MUC\s*LUC|S[ốo]\s+đ[ăa]ng\s+k[ií]\s+KHXB|"
    r"Quy[ếe]t\s+đ[iị]nh\s+xu[aấ]t\s+b[aả]n|In\s+\d+\s+cu[ốo]n)",
    re.I,
)

DNTC_V7_ADMIN_HEADING_RE = re.compile(
    r"^\s*[_\"'“”‘’\-\u2013\u2014.]*\s*"
    r"(?:HUYỆN|HUYEN|huyện\s+[A-ZÀ-ỸĐ]|PHỦ|PHU|TỔNG|TONG|TỈNH|TINH|"
    r"NÚI|NUI|SÔNG|SONG|GIANG|KHE|ĐẦM|DAM|CẢNG|CANG|CẦU|CAU|ĐÒ|DO|"
    r"ĐỀN|DEN|CHÙA|CHUA|MIẾU|MIEU|LĂNG|LANG|ĐÀN|DAN|THÀNH|THANH|"
    r"VĂN\s*-?\s*MIẾU|VAN\s*-?\s*MIEU|PHỤ\s+LỤC|PHU\s+LUC)\b",
    re.I,
)

DNTC_V7_STRUCTURAL_START_RE = re.compile(
    r"\b(?:HUYỆN|HUYEN|PHỦ|PHU|TỔNG|TONG|TỈNH|TINH|NÚI|NUI|SÔNG|SONG|"
    r"CẨN\s+ÁN|CAN\s+AN|XÉT|XET|NĂM|NAM|ĐỜI|DOI|Ở|O|PHÍA|PHIA|"
    r"TỪ|TU|LẠI|LAI|ĐẾN|DEN|TỪNG|TUNG|ĐÀN|DAN)\b",
    re.I,
)

_ORIG_is_backmatter_page_v7 = is_backmatter_page

def is_backmatter_page(text: str, text_m: dict) -> bool:
    s = norm_text(text)
    if not s:
        return False
    # Catalog / publisher ad pages are not book content even when OCR looks high quality.
    if DNTC_V7_BACKMATTER_RE.search(s):
        return True
    return _ORIG_is_backmatter_page_v7(text, text_m)


_ORIG_apply_dntc_reflow_corrections_v7 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v7(text)
    if not s:
        return s

    # Remove remaining scanning/catalog symbols.
    s = re.sub(r"(^|\s)\([RC]\)(?=\s|$)", " ", s)
    s = re.sub(r"[|_]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Normalize headings and admin parentheticals from old scans.
    s = re.sub(r"^\s*huyện\s+(?=[A-ZÀ-ỸĐ])", "HUYỆN ", s)
    s = re.sub(r"^\s*phủ\s+(?=[A-ZÀ-ỸĐ])", "PHỦ ", s)
    s = re.sub(r"^\s*tổng\s+(?=[A-ZÀ-ỸĐ])", "TỔNG ", s)
    s = re.sub(r"\bTHANH\s*-\s*CHU(?:O['’]?|Ơ)NG\b", "THANH-CHƯƠNG", s, flags=re.I)
    s = re.sub(r"\bLAN\s*NAM\b", "La-Nam", s, flags=re.I)
    s = re.sub(r"\bThồ\s*[.]\s*Du\b", "Thổ-Du", s, flags=re.I)
    s = re.sub(r"\((?:do|đo)\s+(?:phi|phả|phá|phd|phe)\s*[: ]+\s*(?:th[eé]ng|th[oô]ng|hống)\s*-\s*(?:hgt|hạt)\)", "(do phủ thống-hạt)", s, flags=re.I)
    s = re.sub(r"\((?:do|đo)\s+(?:phi|phả|phá|phd|phe)\s+(?:ki[ée]m)\s*-\s*l[yý]\)", "(do phủ kiêm-lý)", s, flags=re.I)
    s = re.sub(r"\b(?:phi|phả|phá|phd|phe)\s+ki[ée]m\s*-\s*l[yý]\b", "phủ kiêm-lý", s, flags=re.I)
    s = re.sub(r"\b(?:phi|phả|phá|phd|phe)\s+(?:th[eé]ng|th[oô]ng|hống)\s*-\s*(?:hgt|hạt)\b", "phủ thống-hạt", s, flags=re.I)

    # Numeric OCR cleanup. Apply only where the surrounding context is administrative/statistical.
    s = re.sub(r"(?<=\d)[.]\s+(?=\d)", ".", s)
    s = re.sub(r"\bhơn\s+6s\b", "hơn 65", s, flags=re.I)
    s = re.sub(r"\bcó\s*8o\b", "có 80", s, flags=re.I)
    s = re.sub(r"\bcộng\s+4o\b", "cộng 40", s, flags=re.I)
    s = re.sub(r"\bcó\s+4s\b", "có 45", s, flags=re.I)
    s = re.sub(r"\b1oo(?=[.\s])", "100", s, flags=re.I)
    s = re.sub(r"\b([0-9]+)o(?=[.\s,]*(?:người|mẫu|quan|đồng|dặm|xã|thôn|tổng|hộc|thăng|hạp|thước|tiền|đ))", r"\g<1>0", s, flags=re.I)
    s = re.sub(r"\b([0-9]+)s(?=[.\s,]*(?:người|mẫu|quan|đồng|dặm|xã|thôn|tổng|hộc|thăng|hạp|thước|tiền|đ))", r"\g<1>5", s, flags=re.I)
    s = re.sub(r"\bthứ\s*[¡!|i]r\b", "thứ 11", s, flags=re.I)
    s = re.sub(r"\br1\b", "11", s, flags=re.I)
    s = re.sub(r"\br8\b", "18", s, flags=re.I)
    s = re.sub(r"\brạo6\b", "1906", s, flags=re.I)
    s = re.sub(r"\bTién\b", "Tiền", s, flags=re.I)
    s = re.sub(r"\bthu[€é]\b", "thuế", s, flags=re.I)
    s = re.sub(r"\bthuổ\b", "thuế", s, flags=re.I)
    s = re.sub(r"\bđiền\s+thd\b", "điền thổ", s, flags=re.I)
    s = re.sub(r"\bdin\s+thd\b", "điền thổ", s, flags=re.I)
    s = re.sub(r"\bm4u\b", "mẫu", s, flags=re.I)
    s = re.sub(r"\b(?:céng|cong)\b", "cộng", s, flags=re.I)
    s = re.sub(r"\bhyp\s+cộng\b", "hợp cộng", s, flags=re.I)
    s = re.sub(r"\bl\s+đã\b", "đã", s, flags=re.I)
    s = re.sub(r"\bHồi\s+[—–-]+\s*thuộc\b", "Hồi thuộc", s, flags=re.I)
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([.!?])\s*[.]\s*", r"\1 ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


_ORIG_is_heading_text_v7 = is_heading_text

def is_heading_text(s: str) -> bool:
    s0 = apply_dntc_reflow_corrections(norm_text(s)).strip()
    core = s0.strip(" |_\u2014\u2013- .")
    if not core or len(core) > 140:
        return False

    if _ORIG_is_heading_text_v7(core):
        return True

    upper_ratio = _v6_upper_ratio(core) if "_v6_upper_ratio" in globals() else 0.0
    head_part = core.split("(")[0].strip(" -–—:.")

    # Accept OCR-old administrative headings, including lowercase "huyện" followed by all-caps toponym.
    if DNTC_V7_ADMIN_HEADING_RE.match(core) and len(core) <= 130:
        has_terminal_sentence = bool(re.search(r"[.!?]\s*$", core))
        # parenthetical "(do phủ ...)" is common in true huyện headings.
        has_admin_parenthesis = bool(re.search(r"\(\s*(?:do|đo)\s+phủ\s+(?:kiêm-lý|thống-hạt)\s*\)", core, re.I))
        starts_lower_admin_heading = bool(re.match(r"^\s*huyện\s+[A-ZÀ-ỸĐ]", core))
        if not has_terminal_sentence and (upper_ratio >= 0.48 or has_admin_parenthesis or starts_lower_admin_heading):
            # Reject obvious prose lines like "Huyện đặt ở..." / "Phủ doãn..."
            if not re.match(r"^\s*(?:Huyện|Phủ|Tổng)\s+(?:đặt|này|ấy|doãn|thuộc|là|có|gồm)\b", core, re.I):
                return True

    # Short all-caps toponym labels with explanatory parenthesis are headings/entry labels.
    if len(core) <= 95 and upper_ratio >= 0.60 and re.search(r"\([^)]{2,50}\)", core):
        if not re.search(r"\b(?:cách|giáp|đời|năm|thuộc|đến|từ)\b", head_part, re.I):
            return True

    return False


_ORIG_filter_line_v7 = filter_line

def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    raw = norm_text(line.get("raw_text") or line.get("text") or "")
    if not raw:
        return None, False, ["empty"]

    # Hard-drop publisher catalog lines that survive page classification.
    if DNTC_V7_BACKMATTER_RE.search(raw):
        return None, False, ["publisher_backmatter_line_v7"]

    kept, ok, reasons = _ORIG_filter_line_v7(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
    if not ok or kept is None:
        return kept, ok, reasons

    fixed = apply_dntc_reflow_corrections(kept.get("text", ""))
    if not fixed or DNTC_V7_BACKMATTER_RE.search(fixed):
        return None, False, (reasons or []) + ["empty_or_backmatter_after_v7_correction"]
    kept["text"] = fixed
    kept["line_type"] = "heading" if is_heading_text(fixed) else "body"
    q = text_metrics(fixed)
    kept["line_quality_score"] = q["quality_score"]
    kept["line_vietnamese_ratio"] = q["vietnamese_ratio"]
    kept["line_weird_char_ratio"] = q["weird_char_ratio"]
    return kept, True, reasons


_ORIG_split_sentences_vietnamese_v7 = split_sentences_vietnamese

def _v7_split_overlong_sentence(s: str) -> list:
    s = apply_dntc_reflow_corrections(s)
    if len(s) <= 850:
        return [s]
    # Split only on strong punctuation followed by obvious structural starts.
    parts = re.split(
        r"(?<=[.!?])\s+(?=(?:Cẩn\s+án|Xét|Năm\s+[A-ZÀ-ỸĐa-zà-ỹđ]|Đời\s+[A-ZÀ-ỸĐa-zà-ỹđ]|"
        r"Phía\s+|Ở\s+|Từng\s+|Đàn\s+|Huyện\s+|Phủ\s+|Núi\s+|Sông\s+))",
        s,
        flags=re.I,
    )
    if len(parts) == 1:
        return [s]
    out = []
    buf = ""
    for p in parts:
        p = p.strip()
        if not p:
            continue
        if not buf:
            buf = p
        elif len(buf) < 80:
            buf = buf + " " + p
        else:
            out.append(buf)
            buf = p
    if buf:
        out.append(buf)
    return out


def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    if paragraph_type == "heading":
        return []
    base = _ORIG_split_sentences_vietnamese_v7(paragraph_text, paragraph_type)
    final = []
    for s in base:
        for part in _v7_split_overlong_sentence(s):
            part = apply_dntc_reflow_corrections(part)
            if not part:
                continue
            if globals().get("STRICT_SENTENCE_ONLY", True):
                if not ends_strong_sentenceish(part):
                    continue
                q = text_metrics(part)
                if looks_like_sentence_fragment(part) and q["word_count"] < 18:
                    continue
            final.append(part)
    return final

print("DNTC v7 patch loaded: backmatter hardened, admin headings/numeric cleanup improved; Han text-layer preservation disabled.")


DNTC v7 patch loaded: backmatter hardened, admin headings/numeric cleanup improved; Han text-layer preservation disabled.


In [15]:
# ============================================================
# 8G. DNTC v8 patch: sentence-only cleanup + inline headings + numeric old-scan rescue
# ============================================================
# Review of the latest output showed three remaining roots:
#   1) final_sentences_only still contains multiple sentences in one row,
#      especially long parenthetical notes.
#   2) old-scan admin headings can appear inline inside a body paragraph,
#      e.g. "HUYỆN NAM-ĐÀN (do phủ thống-hạt) Ở phía đông phủ...".
#   3) some valid numeric/currency lines were dropped as hard noise, e.g.
#      "17.694$80. Hop cong 1a 115.691$80."

DNTC_V8_INLINE_ADMIN_HEADING_RE = re.compile(
    r"(?P<head>\b(?:HUYỆN|HUYEN|PHỦ|PHU|TỔNG|TONG)\s+"
    r"[A-ZÀ-ỸĐ][A-ZÀ-ỸĐa-zà-ỹđ0-9 .'’\-–—:]{1,90}?"
    r"(?:\s*\(\s*(?:do|đo)\s+[^)]{2,60}\))?)"
    r"(?=\s+(?:Ở|O|Đông|Dong|Tây|Tay|Nam|Bắc|Bac|Hồi|Hoi|Đời|Doi|Năm|Nam)\b)",
    re.I,
)

DNTC_V8_VALID_CURRENCY_LINE_RE = re.compile(
    r"(?:\d+[.,]\d+\$\d+|H[oọ]p\s+c[oô]ng|Hop\s+cong|c[oô]ng\s+l[aà])",
    re.I,
)

DNTC_V8_DECIMAL_PROTECT_RE = re.compile(r"(?<=\d)\.(?=\d)")
DNTC_V8_ABBR_PROTECT_PATTERNS = [
    r"\bv\.?\s*v\.?",
    r"\bx\.?\s*th\.?",
    r"\btr\.?\s*C\.?\s*N\.?",
    r"\bS\.?\s*đ\.?",
]

_ORIG_apply_dntc_reflow_corrections_v8 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v8(text)
    if not s:
        return s

    # More old-scan numeric fixes observed in 14_15 page 39.
    s = re.sub(r"\bnắm\b", "năm", s, flags=re.I)
    s = re.sub(r"\bsố\s+đỉnh\b", "số đinh", s, flags=re.I)
    s = re.sub(r"\bđỉnh\s+số\b", "đinh số", s, flags=re.I)
    s = re.sub(r"\bso\.\s*471\b", "50.471", s, flags=re.I)
    s = re.sub(r"\bgo\.\s*876\b", "90.876", s, flags=re.I)
    s = re.sub(r"\b40\.\?\s*75\b", "40.775", s, flags=re.I)
    s = re.sub(r"\b88\.7s484o\b", "88.754$40", s, flags=re.I)
    s = re.sub(r"\b45\.026\s+người\b", "45.926 người", s, flags=re.I)
    s = re.sub(r"\bmiễn\s*-\s*dao\s+1680\s+người\b", "miễn-dao 1689 người", s, flags=re.I)
    s = re.sub(r"\b97\.\s*oo78oo\b", "97.997$00", s, flags=re.I)
    s = re.sub(r"\b17\.694\$80\.\s*H[oọ]p\s+c[oô]ng\s+1a\s+115\.691\$80\b",
               "17.694$80. Hợp cộng là 115.691$80", s, flags=re.I)
    s = re.sub(r"\bHop\s+cong\s+1a\b", "Hợp cộng là", s, flags=re.I)
    s = re.sub(r"\bH[oọ]p\s+c[oô]ng\s+1a\b", "Hợp cộng là", s, flags=re.I)
    s = re.sub(r"\bthuế\s+đinh\b", "thuế đinh", s, flags=re.I)

    # More old-scan administrative heading cleanup.
    s = re.sub(r"\((?:do|đo)\s+(?:phả|phá|pha|phi|phd|phe)\s*[: ]+\s*(?:th[oô]ng|thống|hống)\s*-\s*h[ạa]t\)",
               "(do phủ thống-hạt)", s, flags=re.I)
    s = re.sub(r"\((?:do|đo)\s+(?:phả|phá|pha|phi|phd|phe)\s+(?:ki[ée]m)\s*-\s*l[yý]\)",
               "(do phủ kiêm-lý)", s, flags=re.I)
    s = re.sub(r"\bphủ\s+thống\s*[—–-]\s*hạt\b", "phủ thống-hạt", s, flags=re.I)
    s = re.sub(r"\bphủ\s+ki[ée]m\s*[—–-]\s*l[yý]\b", "phủ kiêm-lý", s, flags=re.I)
    s = re.sub(r"\bỞ\s+pha\s+đông\b", "Ở phía đông", s, flags=re.I)
    s = re.sub(r"\bHà\s*-\s*Tinh\b", "Hà-Tĩnh", s, flags=re.I)

    # Internal admin words inside body should not stay all-caps.
    s = re.sub(r"\b(địa giới|giáp|thuộc|cách)\s+HUYỆN\s+", r"\1 huyện ", s)
    s = re.sub(r"\b(địa giới|giáp|thuộc|cách)\s+PHỦ\s+", r"\1 phủ ", s)

    # Common punctuation artifacts.
    s = re.sub(r"\s*[,:;]\s*:", ": ", s)
    s = re.sub(r",\s*\"", ". ", s)
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([.!?])\s*[.]\s*", r"\1 ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _v8_prepare_text_for_sentence_split(text: str) -> str:
    text = apply_dntc_reflow_corrections(text)
    if not text:
        return text

    # Put inline admin headings on a separate virtual line.
    # The sentence exporter will remove heading-only prefixes and split the following body.
    text = re.sub(
        r"\s+(?=\b(?:HUYỆN|HUYEN|PHỦ|PHU|TỔNG|TONG)\s+[A-ZÀ-ỸĐ][A-ZÀ-ỸĐa-zà-ỹđ0-9 .'’\-–—:]{1,90}?"
        r"(?:\s*\(\s*(?:do|đo)\s+[^)]{2,60}\))?\s+(?:Ở|O|Đông|Dong|Tây|Tay|Nam|Bắc|Bac|Hồi|Hoi|Đời|Doi|Năm|Nam)\b)",
        "\n",
        text,
        flags=re.I,
    )
    return text


def _v8_strip_inline_admin_heading_prefix(part: str) -> tuple:
    """Return (heading, body). If part starts with an inline heading, remove it from sentence output."""
    p = apply_dntc_reflow_corrections(part).strip()
    # Match only at the start and only if body starts after the heading.
    m = re.match(
        r"^\s*(?P<head>(?:HUYỆN|HUYEN|PHỦ|PHU|TỔNG|TONG)\s+"
        r"[A-ZÀ-ỸĐ][A-ZÀ-ỸĐa-zà-ỹđ0-9 .'’\-–—:]{1,90}?"
        r"(?:\s*\(\s*(?:do|đo)\s+[^)]{2,60}\))?)"
        r"\s+(?P<body>(?:Ở|O|Đông|Dong|Tây|Tay|Nam|Bắc|Bac|Hồi|Hoi|Đời|Doi|Năm|Nam)\b.*)$",
        p,
        flags=re.I,
    )
    if not m:
        return "", p
    return apply_dntc_reflow_corrections(m.group("head")), apply_dntc_reflow_corrections(m.group("body"))


def _v8_protect_for_post_split(text: str) -> tuple:
    repl = {}
    out = text
    def put(tok):
        key = f"§DNTC_PROT_{len(repl)}§"
        repl[key] = tok.group(0)
        return key
    out = DNTC_V8_DECIMAL_PROTECT_RE.sub("§DOT§", out)
    for pat in DNTC_V8_ABBR_PROTECT_PATTERNS:
        out = re.sub(pat, put, out, flags=re.I)
    return out, repl


def _v8_unprotect_post_split(text: str, repl: dict) -> str:
    text = text.replace("§DOT§", ".")
    for k, v in repl.items():
        text = text.replace(k, v)
    return text


def _v8_split_multi_sentence_row(s: str) -> list:
    """Split a row that still contains several strong sentence boundaries."""
    s = apply_dntc_reflow_corrections(s)
    if not s:
        return []
    protected, repl = _v8_protect_for_post_split(s)

    # Split after .?! if the next visible token looks like a fresh sentence/section.
    # This is intentionally stronger than v7 and works even inside long parentheses.
    parts = re.split(
        r"(?<=[.!?])\s+(?=(?:[A-ZÀ-ỸĐ\"“(]|Năm\s+|Đời\s+|Hồi\s+|Triều\s+|Đến\s+|Lại\s+|Ngoài\s+|Nhà\s+|Mỗi\s+|Cũng\s+|Cha\s+|Tới\s+|Hai\s+|Ba\s+|Một\s+))",
        protected,
        flags=re.I,
    )
    out = []
    for p in parts:
        p = _v8_unprotect_post_split(p.strip(), repl)
        p = apply_dntc_reflow_corrections(p)
        if not p:
            continue
        out.append(p)
    return out or [s]


_ORIG_filter_line_v8 = filter_line

def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    raw = norm_text(line.get("raw_text") or line.get("text") or "")
    # Rescue valid old-scan currency/statistical continuation lines that contain "$"
    # and were previously misclassified as hard noise.
    if raw and DNTC_V8_VALID_CURRENCY_LINE_RE.search(raw):
        fixed = apply_dntc_reflow_corrections(raw)
        q = text_metrics(fixed)
        if q["word_count"] >= 3 or "$" in fixed:
            kept = dict(line)
            kept["text"] = fixed
            kept["line_type"] = "heading" if is_heading_text(fixed) else "body"
            kept["line_quality_score"] = q["quality_score"]
            kept["line_vietnamese_ratio"] = q["vietnamese_ratio"]
            kept["line_weird_char_ratio"] = q["weird_char_ratio"]
            return kept, True, ["rescued_currency_stat_line_v8"]

    kept, ok, reasons = _ORIG_filter_line_v8(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
    if ok and kept is not None:
        fixed = apply_dntc_reflow_corrections(kept.get("text", ""))
        kept["text"] = fixed
        kept["line_type"] = "heading" if is_heading_text(fixed) else "body"
    return kept, ok, reasons


_ORIG_split_sentences_vietnamese_v8 = split_sentences_vietnamese

def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    if paragraph_type == "heading":
        return []

    prepared = _v8_prepare_text_for_sentence_split(paragraph_text)
    if not prepared:
        return []

    # Split virtual lines first, so inline heading labels do not leak into sentence output.
    virtual_parts = [p.strip() for p in prepared.split("\n") if p.strip()]
    final = []

    for part in virtual_parts:
        heading, body = _v8_strip_inline_admin_heading_prefix(part)
        if heading and not body:
            continue
        part = body if body else part

        base = _ORIG_split_sentences_vietnamese_v8(part, "body")
        for s in base:
            for sub in _v8_split_multi_sentence_row(s):
                sub = apply_dntc_reflow_corrections(sub)
                if not sub:
                    continue
                if globals().get("STRICT_SENTENCE_ONLY", True):
                    if not ends_strong_sentenceish(sub):
                        continue
                    q = text_metrics(sub)
                    # Drop numeric orphan fragments like "471 quan 8 tiền".
                    if re.match(r"^\s*\d+[.,]?\d*\s+(?:quan|người|mẫu|hộc|đồng|lạng|tiền|thăng|hạp|thước)\b", sub, re.I) and q["word_count"] < 10:
                        continue
                    if looks_like_sentence_fragment(sub) and q["word_count"] < 18:
                        continue
                final.append(sub)

    return final

print("DNTC v8 patch loaded: multi-sentence rows split, inline admin headings stripped from sentences, old-scan numeric/currency lines rescued.")


DNTC v8 patch loaded: multi-sentence rows split, inline admin headings stripped from sentences, old-scan numeric/currency lines rescued.


In [16]:
# ============================================================
# 8H. DNTC v9 patch: avoid lowercase admin false splits + safer sentence rebuild
# ============================================================
# v8 improved old-scan numbers, but review showed a regression: the inline-heading
# splitter was case-insensitive, so body phrases like "địa giới huyện Nam-Đàn" were
# split as if they were headings. This caused valid sentences beginning with
# "Đông tây...", "Ở phía...", "Hồi thuộc..." to vanish from final_sentences_only.
# v9 keeps the v7/v8 corrections, but makes heading stripping case-sensitive and
# rebuilds sentence splitting with balanced parentheses and softer historical starts.

DNTC_V9_INLINE_ADMIN_HEADING_RE = re.compile(
    r"^\s*(?P<head>(?:[-–—]\s*)?(?:HUYỆN|PHỦ|TỔNG|XÃ|THÀNH|NÚI|SÔNG|ĐỀN|CHÙA|MIẾU)\s+"
    r"[A-ZÀ-ỸĐ0-9][A-ZÀ-ỸĐ0-9a-zà-ỹđ .'’\-–—:]{1,95}?"
    r"(?:\s*\(\s*(?:do|đo)\s+[^)]{2,70}\))?)"
    r"\s+(?P<body>(?:Ở|Đông|Tây|Nam|Bắc|Hồi|Đời|Năm|Đến|Triều|Nay|Hiện)\b.*)$"
)

DNTC_V9_INLINE_ADMIN_INSERT_RE = re.compile(
    r"\s+(?=(?:[-–—]\s*)?(?:HUYỆN|PHỦ|TỔNG|XÃ|THÀNH|NÚI|SÔNG|ĐỀN|CHÙA|MIẾU)\s+"
    r"[A-ZÀ-ỸĐ0-9][A-ZÀ-ỸĐ0-9a-zà-ỹđ .'’\-–—:]{1,95}?"
    r"(?:\s*\(\s*(?:do|đo)\s+[^)]{2,70}\))?"
    r"\s+(?:Ở|Đông|Tây|Nam|Bắc|Hồi|Đời|Năm|Đến|Triều|Nay|Hiện)\b)"
)

DNTC_V9_SOFT_SENT_START_RE = re.compile(
    r"^(?:Năm|Hồi|Đến|Đời|Triều|Nay|Hiện|Sau\s+đó|Từ\s+đấy|Từ\s+đó|Lại\s+có)\b"
)

DNTC_V9_VALID_SHORT_SENT_RE = re.compile(
    r"^(?:Ở|Đông|Tây|Nam|Bắc|Năm|Hồi|Đời|Đến|Nay|Hiện|Đây|Án|Đàn|Miếu|Sông|Núi|Cửa|Phủ|Huyện|Tỉnh)\b",
    re.I,
)

_ORIG_apply_dntc_reflow_corrections_v9 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v9(text)
    if not s:
        return s

    # Targeted old-scan fixes still visible in 14_15 page 39.
    s = re.sub(r"\bHop\s+c[oọộô]ng\s+1a\b", "Hợp cộng là", s, flags=re.I)
    s = re.sub(r"\bH[oọộô]p\s+c[oọộô]ng\s+1a\b", "Hợp cộng là", s, flags=re.I)
    s = re.sub(r"\bsố\s+bạc\s+lạ\b", "số bạc là", s, flags=re.I)
    s = re.sub(r"\bbạc\s+lạ\b", "bạc là", s, flags=re.I)
    s = re.sub(r"\bđiền\s+th[ồờo]ề?\b", "điền thổ", s, flags=re.I)
    s = re.sub(r"\bHà\s*-\s*Tịnh\b", "Hà-Tĩnh", s, flags=re.I)
    s = re.sub(r"\b50\.471\s+quan\s+8\s+tiền,\s*20\s+đồng\b", "50.471 quan 8 tiền, 30 đồng", s, flags=re.I)
    s = re.sub(r"\b100\.228\s+mẫu\s+5\s+sào\s+o\s+thước\b", "100.238 mẫu 5 sào 9 thước", s, flags=re.I)
    s = re.sub(r"\btiền\s+thuế\s+là\s+32\.8230\s+quan\b", "tiền thuế là 23.830 quan", s, flags=re.I)
    s = re.sub(r"\bthành\s+bạc\s+thuế\s+là\s+0?5070đ\.?\s*42\b", "thành bạc thuế là 95979đ.42", s, flags=re.I)
    s = re.sub(r"\bNăm\s+Thành-Thái\s+thứ\s+18\s+điền\s+thổ\s+là\s+212\.064\s+mẫu\b", "Năm Thành-Thái thứ 18 điền thổ là 213.064 mẫu", s, flags=re.I)
    s = re.sub(r"\bbạc\s+thuế\s+là\s+228\.2o3đ4o\b", "bạc thuế là 228.303đ40", s, flags=re.I)
    s = re.sub(r"\b11,\s*415đ15\b", "11.415đ15", s, flags=re.I)
    s = re.sub(r"\b4sao\b", "4 sao", s, flags=re.I)
    s = re.sub(r"\bkh[oô]ng\s+ở\s+trong\s+số\s+này\.\s+Dưới\s+đây\s+cũng\s+thế\)", "không ở trong số này. Dưới đây cũng thế)", s, flags=re.I)

    # OCR sometimes leaves all-caps admin nouns inside ordinary body context.
    # Keep them lowercase when preceded by địa-giới/giáp/thuộc/cách to avoid
    # false heading detection downstream.
    s = re.sub(r"\b(địa\s*-?\s*giới|giáp|thuộc|cách)\s+HUYỆN\s+", r"\1 huyện ", s)
    s = re.sub(r"\b(địa\s*-?\s*giới|giáp|thuộc|cách)\s+PHỦ\s+", r"\1 phủ ", s)

    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _v9_prepare_text_for_sentence_split(text: str) -> str:
    text = apply_dntc_reflow_corrections(text)
    if not text:
        return text
    # Case-sensitive: only true uppercase headings are separated. Lowercase body
    # phrases like "địa giới huyện Nam-Đàn" must stay inside the sentence.
    return DNTC_V9_INLINE_ADMIN_INSERT_RE.sub("\n", text)


def _v9_strip_inline_admin_heading_prefix(part: str) -> tuple:
    p = apply_dntc_reflow_corrections(part).strip()
    m = DNTC_V9_INLINE_ADMIN_HEADING_RE.match(p)
    if not m:
        return "", p
    head = apply_dntc_reflow_corrections(m.group("head"))
    body = apply_dntc_reflow_corrections(m.group("body"))
    return head, body


def _v9_protect(text: str) -> tuple:
    # Reuse v8 protection when available.
    if "_v8_protect_for_post_split" in globals():
        return _v8_protect_for_post_split(text)
    repl = {}
    out = re.sub(r"(?<=\d)\.(?=\d)", "§DOT§", text)
    return out, repl


def _v9_unprotect(text: str, repl: dict) -> str:
    if "_v8_unprotect_post_split" in globals():
        return _v8_unprotect_post_split(text, repl)
    return text.replace("§DOT§", ".")


def _v9_next_visible(rest: str) -> str:
    m = re.search(r"\S", rest or "")
    return rest[m.start():] if m else ""


def _v9_can_split_after_punct(rest: str) -> bool:
    nxt = _v9_next_visible(rest)
    if not nxt:
        return False
    if re.match(r"^[a-zà-ỹđ:;,\-–—\]\)}]", nxt):
        return False
    # Uppercase Vietnamese/Latin, quotes, opening parenthesis, and common historical starts.
    return bool(re.match(
        r"^(?:[A-ZÀ-ỸĐ\"“(]|Năm\s+|Đời\s+|Hồi\s+|Triều\s+|Đến\s+|Lại\s+|Ngoài\s+|Nhà\s+|"
        r"Mỗi\s+|Cũng\s+|Một\s+|Hai\s+|Ba\s+|Đây\s+|Ở\s+|Từ\s+|Sau\s+|Nay\s+|Hiện\s+|"
        r"Phía\s+|Đông\s+|Tây\s+|Nam\s+|Bắc\s+)",
        nxt,
    ))


def _v9_split_hard_outside_parentheses(text: str) -> list:
    text = apply_dntc_reflow_corrections(text)
    if not text:
        return []
    protected, repl = _v9_protect(text)
    parts, buf = [], []
    depth = 0
    n = len(protected)
    for i, ch in enumerate(protected):
        buf.append(ch)
        if ch in "([{" or ch == "«":
            depth += 1
        elif ch in ")]}" or ch == "»":
            depth = max(0, depth - 1)
        if ch not in ".?!…":
            continue
        if depth > 0:
            continue
        if not _v9_can_split_after_punct(protected[i + 1:]):
            continue
        sent = _v9_unprotect("".join(buf).strip(), repl)
        sent = apply_dntc_reflow_corrections(sent)
        if sent:
            parts.append(sent)
        buf = []
    remain = _v9_unprotect("".join(buf).strip(), repl)
    remain = apply_dntc_reflow_corrections(remain)
    if remain:
        parts.append(remain)
    return parts


def _v9_split_soft_historical_starts(text: str) -> list:
    """Split long clauses at ', Năm...', ', Hồi...', etc. outside parentheses."""
    text = apply_dntc_reflow_corrections(text)
    if not text:
        return []
    protected, repl = _v9_protect(text)
    out, buf = [], []
    depth = 0
    i = 0
    while i < len(protected):
        ch = protected[i]
        if ch in "([{" or ch == "«":
            depth += 1
        elif ch in ")]}" or ch == "»":
            depth = max(0, depth - 1)
        if ch in ",;" and depth == 0:
            rest = _v9_next_visible(protected[i + 1:])
            cur = _v9_unprotect("".join(buf).strip(), repl)
            cur = apply_dntc_reflow_corrections(cur)
            rest_unprot = _v9_unprotect(rest, repl)
            if len(cur) >= 55 and DNTC_V9_SOFT_SENT_START_RE.match(rest_unprot):
                if cur and not re.search(r"[.!?…][\"”’\)\]]*$", cur):
                    cur = cur.rstrip(" ,;") + "."
                out.append(cur)
                buf = []
                i += 1
                # skip spaces after the comma/semicolon
                while i < len(protected) and protected[i].isspace():
                    i += 1
                continue
        buf.append(ch)
        i += 1
    remain = _v9_unprotect("".join(buf).strip(), repl)
    remain = apply_dntc_reflow_corrections(remain)
    if remain:
        out.append(remain)
    return out


def _v9_split_candidates(text: str) -> list:
    parts = []
    for hard in _v9_split_hard_outside_parentheses(text):
        parts.extend(_v9_split_soft_historical_starts(hard))
    # Repair any accidental split that leaves a closing parenthesis at the beginning.
    repaired = []
    for p in parts:
        p = apply_dntc_reflow_corrections(p)
        if not p:
            continue
        if repaired:
            bal = repaired[-1].count("(") - repaired[-1].count(")")
            if bal > 0 or re.match(r"^(?:Dưới\s+đây\s+cũng\s+thế\)|[^()]{1,60}\))", p, re.I):
                repaired[-1] = apply_dntc_reflow_corrections(repaired[-1].rstrip() + " " + p)
                continue
        repaired.append(p)
    return repaired


def _v9_sentence_exportable(s: str) -> bool:
    s = apply_dntc_reflow_corrections(s)
    if not s:
        return False
    if is_heading_text(s):
        return False
    if not ends_strong_sentenceish(s):
        return False
    q = text_metrics(s)
    if q["word_count"] < 2:
        return False
    if q["word_count"] < 4 and not DNTC_V9_VALID_SHORT_SENT_RE.match(s):
        return False
    if CLEAR_JUNK_RE.search(s) or is_ascii_short_junk_line(s):
        return False
    if looks_like_sentence_fragment(s) and q["word_count"] < 12 and not DNTC_V9_VALID_SHORT_SENT_RE.match(s):
        return False
    return True


def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    if paragraph_type == "heading":
        return []
    prepared = _v9_prepare_text_for_sentence_split(paragraph_text)
    if not prepared:
        return []
    final = []
    for part in [p.strip() for p in prepared.split("\n") if p.strip()]:
        heading, body = _v9_strip_inline_admin_heading_prefix(part)
        part = body if body else part
        for cand in _v9_split_candidates(part):
            # Strip a heading again if a candidate still begins with one.
            _h, b = _v9_strip_inline_admin_heading_prefix(cand)
            cand = b if b else cand
            cand = apply_dntc_reflow_corrections(cand)
            if _v9_sentence_exportable(cand):
                final.append(cand)
    return final

print("DNTC v9 patch loaded: lowercase admin body phrases preserved; sentence splitting rebuilt with balanced parentheses and historical soft starts.")

# Minor v9.1 refinements for page 39-style statistical lines and short money totals.
_ORIG_apply_dntc_reflow_corrections_v91 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v91(text)
    if not s:
        return s
    s = re.sub(r"^\s*thuế\s+đinh\s+(?=Khoảng\s+năm)", "", s, flags=re.I)
    s = re.sub(r"^\s*[-–—]\s*(?=Khoảng\s+năm)", "", s)
    s = re.sub(r"\bthay\s*2\b", "thay 2", s, flags=re.I)
    s = re.sub(r"\b95979đ\.\s*42\b", "95979đ.42", s, flags=re.I)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Recompile short-sentence start with statistical total opener.
DNTC_V9_VALID_SHORT_SENT_RE = re.compile(
    r"^(?:Ở|Đông|Tây|Nam|Bắc|Năm|Hồi|Đời|Đến|Nay|Hiện|Đây|Án|Đàn|Miếu|Sông|Núi|Cửa|Phủ|Huyện|Tỉnh|Hợp)\b",
    re.I,
)

# Stronger regex post-pass to split any remaining clear double sentence outside decimal numbers.
def _v9_force_split_clear_boundaries(parts: list) -> list:
    out = []
    for p in parts:
        protected, repl = _v9_protect(p)
        chunks = re.split(
            r"(?<=[.!?])\s+(?=(?:Đây\s+|Ở\s+|Năm\s+|Hồi\s+|Đời\s+|Đến\s+|Triều\s+|Nay\s+|Hiện\s+|[A-ZÀ-ỸĐ][a-zà-ỹđ]+\s+))",
            protected,
        )
        for c in chunks:
            c = _v9_unprotect(c.strip(), repl)
            c = apply_dntc_reflow_corrections(c)
            if c:
                out.append(c)
    return out

_ORIG_v9_split_candidates_base = _v9_split_candidates

def _v9_split_candidates(text: str) -> list:
    return _v9_force_split_clear_boundaries(_ORIG_v9_split_candidates_base(text))

print("DNTC v9.1 refinements loaded: statistical headings stripped, short currency totals kept, residual clear boundaries split.")

# v9.2: repair residual parenthetical splits after the force-split pass.
def _v9_repair_parenthetical_parts(parts: list) -> list:
    repaired = []
    for p in parts:
        p = apply_dntc_reflow_corrections(p)
        if not p:
            continue
        if repaired:
            bal = repaired[-1].count("(") - repaired[-1].count(")")
            if bal > 0 or re.match(r"^(?:Dưới\s+đây\s+cũng\s+thế\)|[^()]{1,60}\))", p, re.I):
                repaired[-1] = apply_dntc_reflow_corrections(repaired[-1].rstrip() + " " + p)
                continue
        repaired.append(p)
    return repaired

_ORIG_v9_split_candidates_v92 = _ORIG_v9_split_candidates_base

def _v9_split_candidates(text: str) -> list:
    return _v9_repair_parenthetical_parts(
        _v9_force_split_clear_boundaries(_v9_repair_parenthetical_parts(_ORIG_v9_split_candidates_v92(text)))
    )

print("DNTC v9.2 loaded: parenthetical sentence fragments are repaired after residual boundary splitting.")


DNTC v9 patch loaded: lowercase admin body phrases preserved; sentence splitting rebuilt with balanced parentheses and historical soft starts.
DNTC v9.1 refinements loaded: statistical headings stripped, short currency totals kept, residual clear boundaries split.
DNTC v9.2 loaded: parenthetical sentence fragments are repaired after residual boundary splitting.


In [17]:
# ============================================================
# 8I. DNTC v10 patch: keep valid short Vietnamese sentences + split enumerated/long rows
# ============================================================
# v9 fixed lowercase admin false splits, but review of the new output showed two
# remaining issues:
#   1) many valid short location/history sentences were audited as
#      sentence_fragment_shape and excluded from final_sentences_only, e.g.
#      "Đây nguyên là đất của 2 huyện Nam-Đàn và Thanh-Chương.",
#      "Ở phía đông huyện 63 dặm.", "Đến Hồ-Hán-Thương mới đổi ra tên này."
#   2) some paragraph rows still exported as very long multi-sentence rows,
#      especially numbered notes "21. – ... 22.-..." and long parenthetical notes.
#
# This patch does not add Han-page preservation. It only improves Vietnamese
# sentence export after v9.

DNTC_V10_VALID_SHORT_SENT_RE = re.compile(
    r"^(?:Ở|Đông|Tây|Nam|Bắc|Năm|Hồi|Đời|Đến|Nay|Hiện|Đây|Theo|Từ|Sau|Lại|"
    r"Án|Đàn|Miếu|Đền|Chùa|Sông|Núi|Cửa|Phủ|Huyện|Tỉnh|Hợp|Cách|Dài|Rộng)\b",
    re.I,
)

DNTC_V10_BAD_FINAL_WORD_RE = re.compile(
    r"\b(?:của|và|ở|về|theo|phía|đến|từ|là|có|gồm|cho|đem|mỗi|đều|trong|ngoài|"
    r"giáp|cách|thuộc|nơi|chỗ|bằng|với|như|rằng|thì)\s*[,:;–—-]?$",
    re.I,
)

DNTC_V10_ENUM_MARK_RE = re.compile(
    r"\s+(?=(?:\d{1,2})\s*[.)]\s*[-–—]?\s*[A-ZÀ-ỸĐ])"
)

DNTC_V10_CLEAR_BOUNDARY_RE = re.compile(
    r"(?<=[.!?])\s+(?=(?:Đây\s+|Ở\s+|Năm\s+|Hồi\s+|Đời\s+|Đến\s+|Triều\s+|"
    r"Nay\s+|Hiện\s+|Sau\s+|Từ\s+|Lại\s+|Theo\s+|Hợp\s+|Đông\s+|Tây\s+|"
    r"Nam\s+|Bắc\s+|[A-ZÀ-ỸĐ][a-zà-ỹđ]+\s+|\d{1,2}\s*[.)]\s*[-–—]?\s*))"
)


_ORIG_apply_dntc_reflow_corrections_v10 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v10(text)
    if not s:
        return s

    # Old-scan/administrative residuals observed after v9.
    s = re.sub(r"\btheo\s+như\.\s+cũ\b", "theo như cũ", s, flags=re.I)
    s = re.sub(r"\bcho\s+thuộcvào\b", "cho thuộc vào", s, flags=re.I)
    s = re.sub(r"\bki[ée]m\s*['’]?\s*l[yý]\b", "kiêm-lý", s, flags=re.I)
    s = re.sub(r"\(0o\s+phủ\s+kiêm-D[ýy]\)", "(do phủ kiêm-lý)", s, flags=re.I)
    s = re.sub(r"\bD[oé]ng-Ngan\b", "Đông-Ngàn", s, flags=re.I)
    s = re.sub(r"\bThồ-Thành\b", "Thổ-Thành", s, flags=re.I)
    s = re.sub(r"\bbề(?=\s*[;,]|\s+\d|\s+phía)", "bể", s, flags=re.I)
    s = re.sub(r"\bo(?=\s+dặm\b)", "0", s, flags=re.I)
    s = re.sub(r"\bz(?=\s+dặm\b)", "2", s, flags=re.I)

    # Normalize numbered note spacing so the splitter can see item boundaries.
    s = re.sub(r"\b(\d{1,2})\.\s*[-–—]\s*", r"\1. – ", s)
    s = re.sub(r"\b(\d{1,2})\.-\s*", r"\1. – ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _v10_presplit_enumerators(text: str) -> str:
    text = apply_dntc_reflow_corrections(text)
    if not text:
        return text
    protected, repl = _v9_protect(text)
    protected = DNTC_V10_ENUM_MARK_RE.sub("\n", protected)
    return _v9_unprotect(protected, repl)


def _v10_secondary_split_long_piece(text: str) -> list:
    """Split very long pieces on clear sentence boundaries, even inside notes.

    v9 intentionally repaired parenthetical fragments, but some parenthetical
    notes contain several full Vietnamese sentences. If a piece is too long,
    splitting is better than exporting a 1000-3000 character "sentence".
    """
    text = apply_dntc_reflow_corrections(text)
    if not text:
        return []
    if len(text) < 520 and len(re.findall(r"[.!?]\s+", text)) <= 1:
        return [text]

    protected, repl = _v9_protect(text)
    chunks = DNTC_V10_CLEAR_BOUNDARY_RE.split(protected)
    out = []
    for c in chunks:
        c = _v9_unprotect(c.strip(), repl)
        c = apply_dntc_reflow_corrections(c)
        if c:
            out.append(c)
    return out or [text]


_ORIG_v9_split_candidates_v10 = _v9_split_candidates

def _v9_split_candidates(text: str) -> list:
    text = _v10_presplit_enumerators(text)
    out = []
    for part in [p.strip() for p in text.split("\n") if p.strip()]:
        for cand in _ORIG_v9_split_candidates_v10(part):
            out.extend(_v10_secondary_split_long_piece(cand))
    # Final cleanup: do not emit obvious duplicate whitespace or empty chunks.
    cleaned = []
    for p in out:
        p = apply_dntc_reflow_corrections(p)
        if p:
            cleaned.append(p)
    return cleaned


def _v10_valid_short_or_fragment_sentence(s: str, sq: dict = None) -> bool:
    s = apply_dntc_reflow_corrections(s)
    if not s:
        return False
    if not ends_strong_sentenceish(s):
        return False
    if DNTC_V10_BAD_FINAL_WORD_RE.search(s):
        return False
    if is_heading_text(s):
        return False
    if CLEAR_JUNK_RE.search(s) or is_ascii_short_junk_line(s):
        return False
    q = sq or text_metrics(s)
    if q.get("quality_score", 0) < 45:
        return False
    if q.get("vietnamese_ratio", 0) < 0.30 and q.get("word_count", 0) >= 4:
        return False
    if q.get("word_count", 0) < 3:
        return False
    if DNTC_V10_VALID_SHORT_SENT_RE.match(s):
        return True
    # Also keep normal short prose if it starts uppercase and is not a dangling fragment.
    return bool(q.get("word_count", 0) >= 6 and re.match(r"^[A-ZÀ-ỸĐ]", s))


def _v10_should_block_sentence(sent: str, sq: dict, reasons: list) -> bool:
    if not reasons:
        return False
    reasons = [r for r in reasons if r]
    joined = ";".join(reasons)
    # Always block truly bad cases.
    if "no_terminal_punctuation" in reasons:
        return True
    if "library_noise" in reasons:
        return True
    if "weird_char_ratio" in reasons and not _v10_valid_short_or_fragment_sentence(sent, sq):
        return True
    if sq.get("quality_score", 0) < MIN_KEEP_SENTENCE_QUALITY:
        return True
    # sentence_fragment_shape is noisy for this corpus; many valid short location
    # sentences start with "Ở...", "Đây...", "Đến...". Keep them when they are
    # terminal, Vietnamese-looking, and not dangling.
    if "sentence_fragment_shape" in reasons:
        return not _v10_valid_short_or_fragment_sentence(sent, sq)
    # Other warnings remain in audit but do not necessarily block final export.
    return False


def rebuild_sentences_from_paragraphs():
    global all_paragraphs, all_sentences, suspicious_sentences
    old_para_count = len(all_paragraphs)
    old_sent_count = len(all_sentences)
    all_paragraphs = stitch_paragraph_rows(all_paragraphs)
    all_sentences = []
    suspicious_sentences = []
    sent_counter = Counter()
    for para in all_paragraphs:
        if para.get("paragraph_type") == "heading":
            continue
        sentences = split_sentences_vietnamese(para.get("text", ""), para.get("paragraph_type", "body"))
        for si, sent in enumerate(sentences):
            sent = apply_dntc_reflow_corrections(sent)
            sq = text_metrics(sent)
            s_reasons = suspicious_reasons_for_text(sent, sq)
            sent_counter[str(para.get("work_id"))] += 1
            sent_id = f"{para.get('work_id')}_s{sent_counter[str(para.get('work_id'))]:07d}"
            srow = {
                "sent_id": sent_id,
                "paragraph_id": para.get("paragraph_id"),
                "work_id": para.get("work_id"),
                "pdf_path": para.get("pdf_path"),
                "page_idx": para.get("page_idx"),
                "page_number": para.get("page_number"),
                "page_span": para.get("page_span", para.get("page_number")),
                "sentence_index_in_paragraph": si,
                "text": sent,
                "source": para.get("source"),
                "paragraph_type": para.get("paragraph_type"),
                "bbox": para.get("bbox"),
                "quality_score": sq["quality_score"],
                "vietnamese_ratio": sq["vietnamese_ratio"],
                "weird_char_ratio": sq["weird_char_ratio"],
                "dictionary_hit_ratio": sq["dictionary_hit_ratio"],
                "suspicious_reasons": ";".join(s_reasons),
            }
            if s_reasons:
                suspicious_sentences.append(srow)
                if _v10_should_block_sentence(sent, sq, s_reasons):
                    continue
            all_sentences.append(srow)
    print(f"DNTC v10 post-process: paragraphs {old_para_count} -> {len(all_paragraphs)}, sentences {old_sent_count} -> {len(all_sentences)}")

print("DNTC v10 patch loaded: valid short sentences are kept; enumerated and very long multi-sentence rows are split.")


DNTC v10 patch loaded: valid short sentences are kept; enumerated and very long multi-sentence rows are split.


In [18]:

# ============================================================
# 8J. DNTC v11 safety patch: verified PAGE_STARTS, no silent content loss,
#     conservative page/line filtering, continuation rescue, and bad TINH fix.
# ============================================================
# This cell intentionally overrides selected functions defined above. It is
# placed before process_pdf() is executed, so Run All uses these versions.

PIPELINE_VERSION = "dntc_auto_v11_safety_metadata_quarantine"
PAGE_STARTS = {
    "01.pdf": 21,
    "05.pdf": 15,
    "07_08.pdf": 23,
    "09.pdf": 23,
    "10_11.pdf": 13,
    "12.pdf": 9,
    "13.pdf": 7,
    "14_15.pdf": 7,
    "16_17.pdf": 14,
    "q2_3_4.pdf": 13,
    "q6.pdf": 3,
}

# The project requirement is: uncertain content goes to audit/quarantine, not
# silent deletion. We therefore keep low-confidence pages and flag them later.
DROP_LOW_CONF_OCR_PAGES = False
MIN_SELECTED_PAGE_QUALITY = min(float(globals().get("MIN_SELECTED_PAGE_QUALITY", 43.0)), 30.0)
MIN_OCR_PAGE_QUALITY = min(float(globals().get("MIN_OCR_PAGE_QUALITY", 46.0)), 38.0)
MIN_KEEP_LINE_QUALITY = min(float(globals().get("MIN_KEEP_LINE_QUALITY", 38.0)), 32.0)
MIN_KEEP_SENTENCE_QUALITY = min(float(globals().get("MIN_KEEP_SENTENCE_QUALITY", 38.0)), 32.0)

# Save base functions once. Re-running the cell should not stack wrappers.
_base_estimate_content_start = globals().get("_base_estimate_content_start") or globals().get("estimate_content_start")
_base_classify_page = globals().get("_base_classify_page") or globals().get("classify_page")
_base_filter_line = globals().get("_base_filter_line") or globals().get("filter_line")
_base_apply_corrections = globals().get("_base_apply_corrections") or globals().get("apply_corrections")
_base_is_line_hard_noise = globals().get("_base_is_line_hard_noise") or globals().get("is_line_hard_noise")
_base_is_heading_text = globals().get("_base_is_heading_text") or globals().get("is_heading_text")
_base_split_sentences_vietnamese = globals().get("_base_split_sentences_vietnamese") or globals().get("split_sentences_vietnamese")
_base_select_page_source = globals().get("_base_select_page_source") or globals().get("select_page_source")

ADMIN_HEADING_PREFIX_RE = re.compile(
    r"^(?:TỈNH|TINH|PHỦ|PHU|HUYỆN|HUYEN|CHÂU|CHAU|TỔNG|TONG|XÃ|XA|TRẠM|TRAM|ĐẠO|DAO|THÀNH|THANH|NÚI|NUI|SÔNG|SONG|ĐỀN|DEN|CHÙA|CHUA|MIẾU|MIEU|LĂNG|LANG|PHONG\s*TỤC|HỌC\s*HIỆU|HỘ\s*KHẨU|ĐIỀN\s*PHÚ|NÚI\s*SÔNG|DỊCH\s*TRẠM)\b",
    re.I,
)
STRUCTURAL_PARENT_HEADING_RE = re.compile(
    r"^(?:PHONG\s*TỤC|HỌC\s*HIỆU|HỘ\s*KHẨU|ĐIỀN\s*PHÚ|NÚI\s*SÔNG|DỊCH\s*TRẠM|THÀNH\s*TRÌ|CỔ\s*TÍCH|ĐỀN\s*MIẾU|LĂNG\s*MỘ|QUAN\s*TẤN|CẦU\s*ĐƯỜNG|SẢN\s*VẬT|NHÂN\s*VẬT|PHẦN\s*DÃ|DỰNG\s*ĐẶT)\b",
    re.I,
)
DANGLING_PREV_END_RE = re.compile(
    r"(?:đến\s+trạm|tới\s+trạm|phía\s+(?:nam|bắc|đông|tây)\s+đến\s+trạm|thuộc|gồm|lãnh|là|ở|của|và|đến|tới|tại|trong|ngoài|phía|cách|làm\s+chỗ)\s*$",
    re.I,
)
MEANINGFUL_SHORT_RE = re.compile(
    r"(?:trạm|huyện|phủ|xã|thôn|làng|tổng|dặm|thước|tấc|phân|mẫu|sào|thước|đồng|quan|tiền|người|đinh|thuế|Hòa|Hoà|Lăng|Lẵng|Đàn|Linh|Phong|Phước|Phúc|Chương|Sơn|Giang|Nghệ|Tĩnh|Bình|Trị|Thiên|Huế)",
    re.I,
)
BAD_TINH_REPLACEMENTS = {
    "TỈNH thần": "tinh thần",
    "TỈNH xảo": "tinh xảo",
    "TỈNH tú": "tinh tú",
    "TỈNH thông": "tinh thông",
    "TỈNH kỳ": "tinh kỳ",
    "TỈNH khoan": "tinh khoan",
    "TỈNH quân": "tinh quân",
    "linh TỈNH": "linh tinh",
    "chu thiên TỈNH": "chu thiên tinh",
    "Linh TỈNH": "Linh tinh",
    "Chu thiên TỈNH": "Chu thiên tinh",
}
DANGEROUS_ROUTE_RE = re.compile(
    r"(?:đến\s+trạm\s+Ở|đến\s+trạm\s+TRẠM|phía\s+nam\s+đến\s+trạm\s+Ở|huyện\s+HUYỆN|phủ\s+PHỦ|xã\s+TỈNH|TỈNH\s+(?:thần|xảo|tú|thông|kỳ|khoan|quân))",
    re.I,
)
EXPLICIT_SHORT_NOISE_RE = re.compile(
    r"^(?:DS|H83|FEB|CZ|D13|A5|As|Vol\.?|B\s*\d{3,}|\d{1,3}|[ˆ^]+|[&ˆ£\-\s]+|Ö-®\s*Y|BILIARY\s+LICYRe.*)$",
    re.I,
)
WEIRD_OCR_CHARS_RE = re.compile(r"[ˆ¬¿¡ÐÑÖ§□�]")


def _pdf_key(pdf_path) -> str:
    try:
        return Path(str(pdf_path)).name
    except Exception:
        return str(pdf_path)


def estimate_content_start(doc, work_id, pdf_path) -> int:
    """Use verified metadata instead of heuristic auto-start."""
    key = _pdf_key(pdf_path)
    if key in PAGE_STARTS:
        return int(PAGE_STARTS[key])
    # Fallback for renamed work ids.
    wid_key = f"{work_id}.pdf"
    if wid_key in PAGE_STARTS:
        return int(PAGE_STARTS[wid_key])
    if _base_estimate_content_start:
        return int(_base_estimate_content_start(doc, work_id, pdf_path))
    return 1


def _has_significant_text(m: dict) -> bool:
    return bool(
        (m or {}).get("char_count", 0) > 300 or
        (m or {}).get("word_count", 0) > 50 or
        (m or {}).get("quality_score", 0) > 40
    )


def classify_page(page_number: int, text: str, text_m: dict, visual_m: dict, content_start_page: int | None) -> tuple:
    """Do not classify content pages as front/title once PAGE_STARTS says content began."""
    # Always drop true blank pages.
    if DROP_BLANK_PAGES and is_blank_page(text_m, visual_m):
        return "blank_page", ["blank_visual_or_no_text"]

    after_start = content_start_page is not None and page_number >= int(content_start_page)
    if after_start:
        # Pattern/library drops after content start are allowed only when almost no text.
        if DROP_PATTERNED_PAGES and is_patterned_page(text_m, visual_m, page_number) and text_m.get("word_count", 0) < 8:
            return "patterned_endpaper", ["patterned_visual_no_content_after_start"]
        if DROP_LIBRARY_PAGES and is_library_page(text, text_m) and text_m.get("word_count", 0) < 12:
            return "library_barcode_stamp_watermark", ["library_tokens_no_content_after_start"]
        # Never drop content-region pages as title/front matter or publisher page.
        return "content_candidate", ["verified_content_region"]

    # Before content start, use the base classifier.
    if _base_classify_page:
        return _base_classify_page(page_number, text, text_m, visual_m, content_start_page)
    return "content_candidate", []



def select_page_source(page, page_idx, work_id, pdf_path, page_class) -> tuple:
    """Select best available source but do not return an empty drop when text exists.
    Low-quality pages are kept and later quarantined/audited rather than silently lost.
    """
    tl_lines = extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path)
    tl_text = lines_to_text(tl_lines)
    tl_m = text_metrics(tl_text, lines=[r.get("raw_text", "") for r in tl_lines])

    should_ocr = ENABLE_TESSERACT and tesseract_available() and (
        tl_m["quality_score"] < MIN_TEXT_LAYER_GOOD_QUALITY or
        tl_m["letter_count"] < 40 or
        tl_m["weird_char_ratio"] > 0.06 or
        tl_m["repeated_char_ngram_ratio"] > 0.08
    )
    ocr_lines, ocr_m = [], {}
    if should_ocr and page_class not in {"blank_page", "patterned_endpaper", "library_barcode_stamp_watermark", "cover_title_front_matter", "front_matter_before_content"}:
        try:
            ocr_lines = extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=PAGE_OCR_DPI)
            ocr_text = lines_to_text(ocr_lines)
            ocr_m = text_metrics(ocr_text, lines=[r.get("raw_text", "") for r in ocr_lines], ocr_conf_values=[r.get("ocr_conf") for r in ocr_lines])
        except Exception as e:
            ocr_lines, ocr_m = [], {"quality_score": -999, "word_count": 0, "char_count": 0, "error": type(e).__name__}

    if ocr_lines and ocr_m.get("quality_score", -999) >= tl_m.get("quality_score", -999) + 7 and ocr_m.get("quality_score", 0) >= MIN_OCR_PAGE_QUALITY:
        return ocr_lines, "tesseract_page", ocr_m, tl_m, ocr_m, f"use_ocr_better_or_textlayer_poor tl={tl_m.get('quality_score',0):.1f} ocr={ocr_m.get('quality_score',0):.1f}"
    if tl_lines and (tl_m.get("word_count", 0) >= 5 or tl_m.get("char_count", 0) >= 80):
        reason = "keep_text_layer_or_quarantine_low_quality"
        if ocr_lines:
            reason += f" tl={tl_m.get('quality_score',0):.1f} ocr={ocr_m.get('quality_score',0):.1f}"
        return tl_lines, "pdf_text_layer", tl_m, tl_m, ocr_m, reason
    if ocr_lines:
        return ocr_lines, "tesseract_page_quarantine_low_quality", ocr_m, tl_m, ocr_m, f"ocr_available_no_good_text_layer tl={tl_m.get('quality_score',0):.1f} ocr={ocr_m.get('quality_score',0):.1f}"
    # Truly no text source.
    return [], "drop_no_text_source", tl_m, tl_m, ocr_m, "no_text_layer_or_ocr_lines"

def _repair_bad_tinh_phrases(text: str) -> str:
    s = str(text or "")
    for bad, good in BAD_TINH_REPLACEMENTS.items():
        s = s.replace(bad, good)
    # Case-insensitive cleanup for common false positives after mixed casing.
    s = re.sub(r"\bTỈNH\s+(thần|xảo|tú|thông|kỳ|khoan|quân|môn)\b", lambda m: "tinh " + m.group(1).lower(), s, flags=re.I)
    return s


def apply_corrections(text: str, meta: dict, correction_log: list) -> str:
    before = str(text or "")
    out = _base_apply_corrections(before, meta, correction_log) if _base_apply_corrections else before
    fixed = _repair_bad_tinh_phrases(out)
    if fixed != out:
        try:
            correction_log.append({**(meta or {}), "rule": "fix_bad_heading_tinh_false_positive", "before": out, "after": fixed})
        except Exception:
            pass
    return fixed


def is_heading_text(s: str) -> bool:
    s0 = norm_text(s)
    if not s0 or len(s0) > 120:
        return False
    # Explicit structural headings.
    if STRUCTURAL_PARENT_HEADING_RE.match(s0):
        return True
    # Administrative headings: require uppercase-ish or short standalone heading.
    if ADMIN_HEADING_PREFIX_RE.match(s0):
        letters = LETTERS_RE.findall(s0)
        upper_letters = sum(1 for ch in letters if ch.upper() == ch and ch.lower() != ch)
        upper_ratio = upper_letters / max(1, len(letters))
        if upper_ratio >= 0.45 or len(WORDS_RE.findall(s0)) <= 8:
            return True
    return bool(_base_is_heading_text(s0)) if _base_is_heading_text else False


def is_line_hard_noise(s: str) -> bool:
    s0 = norm_text(s)
    if not s0:
        return True
    if is_heading_text(s0):
        return False
    if EXPLICIT_SHORT_NOISE_RE.match(s0):
        return True
    # Preserve short geographic/admin/numeric content and continuation candidates.
    if len(s0) < 55 and (MEANINGFUL_SHORT_RE.search(s0) or re.search(r"\d+\s*(?:dặm|thước|tấc|mẫu|sào|đồng|quan|tiền|người|đinh)", s0, re.I)):
        return False
    if _base_is_line_hard_noise and _base_is_line_hard_noise(s0):
        # The base rule is sometimes too aggressive; do not drop meaningful short content.
        if len(s0) < 80 and (MEANINGFUL_SHORT_RE.search(s0) or has_vietnamese_diacritic(s0)):
            return False
        return True
    return False


_DNTC_LAST_LINE_CONTEXT = {}


def _make_rescued_line(raw_line: dict, fixed: str, reasons: list) -> dict:
    q = line_quality(fixed, raw_line.get("ocr_conf"))
    line_type = "heading" if is_heading_text(fixed) else "body"
    return {
        "work_id": raw_line.get("work_id"),
        "pdf_path": str(raw_line.get("pdf_path")),
        "page_idx": raw_line.get("page_idx"),
        "page_number": raw_line.get("page_number"),
        "line_global_id": raw_line.get("line_global_id"),
        "text": fixed,
        "raw_text": raw_line.get("raw_text", fixed),
        "source": raw_line.get("source"),
        "bbox": raw_line.get("bbox", []),
        "ocr_conf": raw_line.get("ocr_conf"),
        "line_type": line_type,
        "line_quality_score": q.get("quality_score"),
        "rescue_reasons": ";".join(reasons),
    }


def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    raw = norm_text(line.get("raw_text") or line.get("text") or "")
    key = (line.get("work_id"), line.get("page_number"))
    prev = _DNTC_LAST_LINE_CONTEXT.get(key, {})
    prev_text = prev.get("text", "")
    try:
        kept, is_kept, reasons = _base_filter_line(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
    except Exception as e:
        kept, is_kept, reasons = None, False, [f"base_filter_error:{type(e).__name__}"]

    # Rescue short real content and continuation lines.
    continuation = bool(prev_text and DANGLING_PREV_END_RE.search(prev_text))
    raw_has_content = bool(MEANINGFUL_SHORT_RE.search(raw) or re.search(r"\d+\s*(?:dặm|thước|tấc|mẫu|sào|đồng|quan|tiền|người|đinh|hộc|thăng|hạp)", raw, re.I))
    raw_is_heading = is_heading_text(raw)
    if (not is_kept or kept is None) and raw and not EXPLICIT_SHORT_NOISE_RE.match(raw):
        if raw_is_heading or continuation or raw_has_content:
            meta = {"work_id": line.get("work_id"), "pdf_path": str(line.get("pdf_path")), "page_number": line.get("page_number"), "line_id": line.get("line_global_id")}
            fixed = apply_corrections(raw, meta, correction_log)
            kept = _make_rescued_line(line, fixed, ["rescued_heading_or_continuation" if (raw_is_heading or continuation) else "rescued_meaningful_short_line"])
            is_kept = True
            reasons = list(reasons or []) + ["v11_rescue_heading_continuation_or_meaningful_short"]

    if is_kept and kept is not None:
        kept["text"] = _repair_bad_tinh_phrases(kept.get("text", ""))
        # Reclassify line type after final correction.
        kept["line_type"] = "heading" if is_heading_text(kept["text"]) else kept.get("line_type", "body")
        _DNTC_LAST_LINE_CONTEXT[key] = {"text": kept.get("text", ""), "kept": True, "line_type": kept.get("line_type")}
    else:
        _DNTC_LAST_LINE_CONTEXT[key] = {"text": raw, "kept": False, "line_type": "dropped"}
    return kept, bool(is_kept), reasons


_NUMBERED_ITEM_START_RE = re.compile(r"(?=(?:^|\s)(?:\d{1,3}|[IVXLC]+)\s*[\.\-–—]\s+)")
_SENT_BOUNDARY_RE = re.compile(r"(?<=[.!?])\s+(?=(?:[A-ZÀ-ỸĐ]|\d{1,3}\s*[\.\-–—]))")
_FOOTNOTE_START_RE = re.compile(r"(?=\s*\(\d{1,3}\)\s+[A-ZÀ-ỸĐ])")


def split_sentences_vietnamese(text: str, paragraph_type: str = "body") -> list:
    """Conservative but not lossy splitter. Keeps fragments flagged later rather than dropping."""
    s = apply_dntc_reflow_corrections(norm_text(text)) if "apply_dntc_reflow_corrections" in globals() else norm_text(text)
    s = _repair_bad_tinh_phrases(s)
    if not s:
        return []
    if paragraph_type == "heading" or is_heading_text(s):
        return [s]

    # Normalize numbered item markers for reliable splitting.
    s = re.sub(r"(\d{1,3})\s*[,،]\s*[–-]", r"\1. –", s)
    s = re.sub(r"(\d{1,3})\s*\.\s*[–-]", r"\1. –", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Protect common abbreviations and decimals.
    protected = []
    def _prot(m):
        protected.append(m.group(0))
        return f"§ABBR{len(protected)-1}§"
    work = re.sub(r"\b(?:v\.v\.|tr\.C\.N\.|T\.N\.|x\.\s*th\.)", _prot, s, flags=re.I)
    work = re.sub(r"(?<=\d)\.(?=\d)", "§DOT§", work)
    work = re.sub(r"(?<=\d),(?=\d)", "§COMMA§", work)

    # Split before numbered items when the item is not at the very start.
    work = re.sub(r"\s+(?=\d{1,3}\s*\.\s*[–-])", "\n", work)
    # Split footnote blocks if they start after body text.
    work = re.sub(r"\s+(?=\(\d{1,3}\)\s+[A-ZÀ-ỸĐ])", "\n", work)
    # Split normal sentence boundaries.
    work = _SENT_BOUNDARY_RE.sub("\n", work)
    # Split obvious transition openers after a full sentence accidentally comma-merged.
    work = re.sub(r"(?<=[.!?])\s+(?=(?:Năm|Đời|Hồi|Đến|Nay|Ở|Phía|Đông|Tây|Nam|Bắc)\b)", "\n", work)

    parts = []
    for chunk in work.split("\n"):
        chunk = chunk.replace("§DOT§", ".").replace("§COMMA§", ",").strip()
        for i, original in enumerate(protected):
            chunk = chunk.replace(f"§ABBR{i}§", original)
        if not chunk:
            continue
        # If still extremely long, split on semicolon before major openers.
        if len(chunk) > 650:
            subs = re.split(r"(?<=;)\s+(?=(?:Năm|Đời|Hồi|Đến|Nay|Phía|Ở|Cần\s+án|Lại|Theo)\b)", chunk)
            parts.extend([p.strip() for p in subs if p.strip()])
        else:
            parts.append(chunk)
    return parts

print("Loaded DNTC v11 safety patch. PAGE_STARTS:", PAGE_STARTS)


Loaded DNTC v11 safety patch. PAGE_STARTS: {'01.pdf': 21, '05.pdf': 15, '07_08.pdf': 23, '09.pdf': 23, '10_11.pdf': 13, '12.pdf': 9, '13.pdf': 7, '14_15.pdf': 7, '16_17.pdf': 14, 'q2_3_4.pdf': 13, 'q6.pdf': 3}


In [19]:

# ============================================================
# 8K. DNTC v12 patch: mixed-script OCR/layout reconstruction,
#     garbage fragment quarantine, micro-heading support.
# ============================================================
# This cell overrides selected v11 functions before process_pdf() runs.

PIPELINE_VERSION = "dntc_auto_v12_mixed_script_reconstruction"

CJK_CLASS = "\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff"
CJK_RE = re.compile(f"[{CJK_CLASS}]")
CJK_TOKEN_RE = re.compile(f"[{CJK_CLASS}]+")
LATIN_TOKEN_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]+(?:[-'][A-Za-zÀ-ỹĐđ]+)*")
NUMERIC_TOKEN_RE = re.compile(r"\d+(?:[.,]\d+)*(?:\$\d+|đ\d*)?")
PUNCT_TOKEN_RE = re.compile(r"[.,;:!?()\[\]{}\-–—/]+")

ENTRY_KEYWORD_RE = re.compile(
    r"(?:th[uủ]y[-\s]?quan|thuỷ[-\s]?quan|kiều|cầu|môn|thành|điện|đài|viện|cung|trì|phường|lầu|lang|miếu|đình|tạ|vu|sương|các|đường|án|miếu|đền)$",
    re.I,
)
MICRO_HEADING_LABEL_RE = re.compile(
    rf"(?P<label>(?:[A-ZÀ-ỸĐa-zà-ỹđ][A-Za-zÀ-ỹĐđ'\-]*\s+){{0,7}}(?:th[uủ]y[-\s]?quan|thuỷ[-\s]?quan|kiều|cầu|môn|thành|điện|đài|viện|cung|trì|phường|lầu|lang|miếu|đình|tạ|vu|sương|các|đường|án|đền))\s+(?P<cjk>[{CJK_CLASS}]{{2,14}})\s*(?P<colon>[:：])?",
    re.I,
)
BIG_MIXED_HEADING_RE = re.compile(
    rf"^(?P<label>[A-ZÀ-ỸĐ0-9][A-ZÀ-ỸĐ0-9\s\-–—]{{2,80}})\s+(?P<cjk>[{CJK_CLASS}]{{2,14}})\s*$"
)

# Garbage/noise fragments are token-level, not sentence-specific hard-coded corrections.
MIXED_GARBAGE_TOKEN_RE = re.compile(
    r"(?ix)(?:"
    r"\bbik\b|\bwt\b|\bkt\s+ah\b|\bor\s*\+\b|\bx\s+wt\b|"
    r"BILIARY\s+LICYRe[^,.;]*|"
    r"(?:\*\s*){1,}|[&^`#]+|[ˆ¬¿¡ÐÑÖ§□�]+|"
    r"\b\d+\s+th\s+gi\b|\b\d+\s+or\s*\+\s*\w*\b"
    r")"
)
MIXED_GARBAGE_LINE_RE = re.compile(
    r"(?ix)^(?:\s*(?:TỈNH\s+)?(?:bik|wt|x\s+wt|kt\s+ah|\d+\s+th\s+gi|\d+\s+or\s*\+\s*\w*|[&ˆ£`#^*\-\s]+|BILIARY\s+LICYRe.*|Ö-®\s*Y)\s*)+$"
)
FALSE_TINH_STRICT_RE = re.compile(
    r"\bTỈNH\s+(?:thần|xảo|tú|thông|kỳ|khoan|huyết|bồ|bik|quân|môn|bường|bik)\b",
    re.I,
)
TINH_GARBAGE_RE = re.compile(r"\bTỈNH\s+(?:bik|wt|x\s+wt|kt\s+ah|or\s*\+)(?:\s*\*)*", re.I)
DANGLING_SINGLE_CJK_AFTER_LABEL_RE = re.compile(
    rf"(?P<prefix>\b(?:có\s+\d+\s+cửa|đài|lầu|môn|kiều|viện|cung|trì|phường|th[uủ]y[-\s]?quan|thuỷ[-\s]?quan|nam[-\s]?khuyết[-\s]?đài|bắc[-\s]?khuyết|tây[-\s]?khuyết|đông[-\s]?khuyết|Ngũ\s+Phụng)[^,.;:]*?)\s+(?P<cjk>[{CJK_CLASS}])(?P<tail>\s*[,.;:]?)",
    re.I,
)
DANGLING_SINGLE_CJK_COLON_RE = re.compile(rf"(:\s*)[{CJK_CLASS}]\s*[,，]\s*")

MIXED_ENTRY_SPLIT_RE = MICRO_HEADING_LABEL_RE
MIXED_SCRIPT_QUARANTINE_RECORDS = globals().get("MIXED_SCRIPT_QUARANTINE_RECORDS", [])

_base_v12_apply_corrections = globals().get("_base_v12_apply_corrections") or globals().get("apply_corrections")
_base_v12_filter_line = globals().get("_base_v12_filter_line") or globals().get("filter_line")
_base_v12_is_line_hard_noise = globals().get("_base_v12_is_line_hard_noise") or globals().get("is_line_hard_noise")
_base_v12_is_heading_text = globals().get("_base_v12_is_heading_text") or globals().get("is_heading_text")
_base_v12_split_sentences = globals().get("_base_v12_split_sentences") or globals().get("split_sentences_vietnamese")


def cjk_count(s: str) -> int:
    return len(CJK_RE.findall(norm_text(s)))


def has_cjk(s: str) -> bool:
    return bool(CJK_RE.search(norm_text(s)))


def mixed_script_segments(s: str) -> list:
    """Segment a string into latin/vietnamese, cjk, numeric, punctuation, and noise runs."""
    s = norm_text(s)
    segs = []
    i = 0
    token_re = re.compile(rf"([{CJK_CLASS}]+|\d+(?:[.,]\d+)*(?:\$\d+|đ\d*)?|[A-Za-zÀ-ỹĐđ]+(?:[-'][A-Za-zÀ-ỹĐđ]+)*|[.,;:!?()\[\]{{}}\-–—/]+|\s+|.)")
    for m in token_re.finditer(s):
        tok = m.group(0)
        if tok.isspace():
            typ = "space"
        elif CJK_TOKEN_RE.fullmatch(tok):
            typ = "cjk"
        elif NUMERIC_TOKEN_RE.fullmatch(tok):
            typ = "numeric"
        elif LATIN_TOKEN_RE.fullmatch(tok):
            typ = "latin"
        elif PUNCT_TOKEN_RE.fullmatch(tok):
            typ = "punct"
        elif MIXED_GARBAGE_TOKEN_RE.search(tok):
            typ = "noise"
        else:
            typ = "other"
        segs.append({"type": typ, "text": tok, "start": m.start(), "end": m.end()})
    return segs


def _record_mixed_quarantine(reason: str, text: str, meta: dict | None = None, fragment: str = "", prev_text: str = "", next_text: str = ""):
    rec = {
        "reason": reason,
        "fragment": norm_text(fragment or text),
        "text": norm_text(text),
        "prev_text": norm_text(prev_text),
        "next_text": norm_text(next_text),
    }
    if meta:
        for k in ["work_id", "pdf_path", "page_number", "page_idx", "line_id", "line_global_id", "paragraph_id", "source_paragraph_id"]:
            if k in meta:
                rec[k] = meta.get(k)
    MIXED_SCRIPT_QUARANTINE_RECORDS.append(rec)


def _is_valid_short_content(s: str) -> bool:
    t = norm_text(s)
    if not t:
        return False
    if re.search(r"\d+\s*(?:dặm|thước|tấc|phân|mẫu|sào|đồng|quan|tiền|người|đinh|hộc|thăng|hạp|cấp)\b", t, re.I):
        return True
    if re.search(r"\b(?:Hòa|Hoà|Lẵng|Lăng|Đàn|Linh|Phong|Phước|Phúc|Chương|Sơn|Giang|Nghệ|Tĩnh|Bình|Trị|Thiên|Huế|Nam|Đông|Tây|Bắc)[-A-Za-zÀ-ỹĐđ]*\b", t):
        return True
    if _base_v12_is_heading_text and _base_v12_is_heading_text(t):
        return True
    if MICRO_HEADING_LABEL_RE.search(t) or BIG_MIXED_HEADING_RE.match(t):
        return True
    return False


def is_mixed_script_ocr_garbage_fragment(text: str, prev_text: str = "", next_text: str = "") -> bool:
    t = norm_text(text)
    if not t:
        return False
    if _is_valid_short_content(t):
        return False
    if MIXED_GARBAGE_LINE_RE.match(t):
        return True
    # Very short mixed-script/ASCII garbage near CJK/heading context.
    if len(t) <= 18 and MIXED_GARBAGE_TOKEN_RE.search(t):
        return True
    if len(t) <= 8 and re.search(r"[A-Za-z]", t) and not has_vietnamese_diacritic(t) and not re.search(r"\d+\s*(?:dặm|mẫu|quan|đồng)", t, re.I):
        if has_cjk(prev_text) or has_cjk(next_text) or is_heading_text(prev_text) or is_heading_text(next_text):
            return True
    return False


def _repair_tinh_uppercase_context(text: str, is_heading_context: bool | None = None) -> str:
    t = norm_text(text)
    if not t:
        return t
    if is_heading_context is None:
        letters = LETTERS_RE.findall(t) if "LETTERS_RE" in globals() else re.findall(r"[A-Za-zÀ-ỹĐđ]", t)
        upper_ratio = sum(1 for ch in letters if ch.upper() == ch and ch.lower() != ch) / max(1, len(letters)) if letters else 0
        is_heading_context = bool((t.startswith("TỈNH ") and upper_ratio > 0.45 and len(WORDS_RE.findall(t)) <= 8) or (_base_v12_is_heading_text and _base_v12_is_heading_text(t)))
    # Always repair known false positives, even if the line looks noisy heading-like.
    false_map = {
        "TỈNH thần": "tinh thần", "TỈNH xảo": "tinh xảo", "TỈNH tú": "tinh tú",
        "TỈNH thông": "tinh thông", "TỈNH kỳ": "tinh kỳ", "TỈNH khoan": "tinh khoan",
        "TỈNH huyết": "tinh huyết", "TỈNH bồ": "tinh bồ", "TỈNH quân": "tinh quân",
        "TỈNH môn": "tinh môn", "TỈNH bik": "tinh",
    }
    for a, b in false_map.items():
        t = re.sub(re.escape(a), b, t, flags=re.I)
    # TỈNH + garbage is never a valid administrative heading.
    t = TINH_GARBAGE_RE.sub("", t)
    if not is_heading_context:
        # In body text, uppercase TỈNH is usually a correction artifact. Keep the word but lowercase it.
        t = re.sub(r"\bTỈNH\b", "tỉnh", t)
    return norm_text(t)


def clean_mixed_script_text(text: str, meta: dict | None = None, *, context: str = "text") -> tuple[str, list]:
    """Remove/flag OCR garbage fragments around CJK without guessing missing Han characters."""
    original = norm_text(text)
    t = original
    reasons = []
    if not t:
        return "", reasons

    # Remove TỈNH + OCR garbage as a cluster before token-level cleanup.
    if TINH_GARBAGE_RE.search(t):
        for m in TINH_GARBAGE_RE.finditer(t):
            _record_mixed_quarantine("false_tinh_garbage_fragment", original, meta, fragment=m.group(0))
        reasons.append("false_tinh_garbage_fragment")
        t = TINH_GARBAGE_RE.sub(" ", t)

    # Remove garbage token clusters.
    def _garb(m):
        frag = m.group(0)
        if frag and frag.strip():
            _record_mixed_quarantine("mixed_script_ocr_garbage_fragment", original, meta, fragment=frag)
            reasons.append("mixed_script_ocr_garbage_fragment")
        return " "
    t = MIXED_GARBAGE_TOKEN_RE.sub(_garb, t)

    # Remove isolated single CJK after colon in phrases like "có 4 cửa: 門,".
    if DANGLING_SINGLE_CJK_COLON_RE.search(t):
        for m in DANGLING_SINGLE_CJK_COLON_RE.finditer(t):
            _record_mixed_quarantine("cjk_fragment_incomplete", original, meta, fragment=m.group(0))
        reasons.append("cjk_fragment_incomplete")
        t = DANGLING_SINGLE_CJK_COLON_RE.sub(r"\1", t)

    # Remove single dangling CJK after Vietnamese labels; preserve complete CJK labels of length >=2 elsewhere.
    def _dang(m):
        frag = m.group(0)
        _record_mixed_quarantine("cjk_fragment_incomplete", original, meta, fragment=frag)
        reasons.append("cjk_fragment_incomplete")
        return m.group("prefix") + (m.group("tail") or "")
    t = DANGLING_SINGLE_CJK_AFTER_LABEL_RE.sub(_dang, t)

    # Repair false TỈNH after garbage stripping.
    t2 = _repair_tinh_uppercase_context(t)
    if t2 != t:
        reasons.append("false_tinh_uppercase_repaired")
    t = t2

    # If a line/fragment is only garbage after cleanup, quarantine it.
    t = re.sub(r"\s+([,.;:])", r"\1", t)
    t = re.sub(r"([:])\s*([,.;])", r"\1", t)
    t = re.sub(r"\s+", " ", t).strip(" ,;:-–—")
    if original and not t:
        reasons.append("mixed_script_cleaned_empty")
    return norm_text(t), sorted(set(reasons))


def is_heading_text(s: str) -> bool:
    t = norm_text(s)
    if not t:
        return False
    if BIG_MIXED_HEADING_RE.match(t):
        return True
    if MICRO_HEADING_LABEL_RE.fullmatch(t.rstrip(":：")) or MICRO_HEADING_LABEL_RE.match(t) and len(t) <= 90 and t.endswith((":", "：")):
        return True
    if _base_v12_is_heading_text:
        return bool(_base_v12_is_heading_text(t))
    return False


def is_line_hard_noise(s: str) -> bool:
    t = norm_text(s)
    if is_mixed_script_ocr_garbage_fragment(t):
        return True
    return bool(_base_v12_is_line_hard_noise(t)) if _base_v12_is_line_hard_noise else False


def apply_corrections(text: str, meta: dict | None = None, log: list | None = None) -> str:
    before = norm_text(text)
    out = _base_v12_apply_corrections(before, meta, log) if _base_v12_apply_corrections else before
    out2, reasons = clean_mixed_script_text(out, meta, context="correction")
    if out2 != out and log is not None:
        log.append({**(meta or {}), "rule": "mixed_script_cleanup", "before": out, "after": out2, "reason": ";".join(reasons)})
    return out2


def filter_line(line: dict, reocr_state=None):
    raw = norm_text(line.get("text", ""))
    key = (line.get("work_id"), line.get("page_number"))
    prev_ctx = (globals().get("_DNTC_LAST_LINE_CONTEXT", {}) or {}).get(key, {})
    prev_text = norm_text(prev_ctx.get("text", ""))
    meta = {"work_id": line.get("work_id"), "pdf_path": str(line.get("pdf_path", "")), "page_number": line.get("page_number"), "line_id": line.get("line_global_id"), "line_global_id": line.get("line_global_id")}
    if is_mixed_script_ocr_garbage_fragment(raw, prev_text=prev_text):
        _record_mixed_quarantine("mixed_script_ocr_garbage_fragment", raw, meta, fragment=raw, prev_text=prev_text)
        if "_DNTC_LAST_LINE_CONTEXT" in globals():
            _DNTC_LAST_LINE_CONTEXT[key] = {"text": raw, "kept": False, "line_type": "dropped"}
        return None, False, ["mixed_script_ocr_garbage_fragment"]

    kept, is_kept, reasons = _base_v12_filter_line(line, reocr_state) if _base_v12_filter_line else (dict(line), True, [])
    reasons = list(reasons or [])
    if is_kept and kept is not None:
        cleaned, extra = clean_mixed_script_text(kept.get("text", raw), meta, context="line")
        reasons.extend(extra)
        if not cleaned and raw:
            _record_mixed_quarantine("mixed_script_cleaned_empty", raw, meta, fragment=raw, prev_text=prev_text)
            return None, False, sorted(set(reasons + ["mixed_script_cleaned_empty"]))
        kept["text"] = cleaned
        if extra:
            kept["suspicious_reasons"] = ";".join(sorted(set(str(kept.get("suspicious_reasons", "")).split(";") + extra)))
        kept["line_type"] = "heading" if is_heading_text(cleaned) else kept.get("line_type", "body")
    return kept, is_kept, sorted(set(reasons))


def split_sentences_vietnamese(text: str, paragraph_type: str = "body") -> list:
    s, clean_reasons = clean_mixed_script_text(text, None, context="sentence_split")
    if not s:
        return []
    if paragraph_type in {"heading", "micro_heading"} or is_heading_text(s):
        return [s.rstrip(":：")]
    # Split before micro-entry labels and large headings if they appear mid-paragraph.
    s = re.sub(rf"\s+(?=(?:[A-ZÀ-ỸĐ][A-Za-zÀ-ỹĐđ'\-]*\s+){{0,7}}(?:th[uủ]y[-\s]?quan|thuỷ[-\s]?quan|kiều|cầu|môn|thành|điện|đài|viện|cung|trì|phường|lầu|miếu|đình|đền)\s+[{CJK_CLASS}]{{2,14}}\s*[:：])", "\n", s)
    s = re.sub(r"\s+(?=(?:HOÀNG|HOANG|TỬ|TU|TỬ\s+CẤM|TU\s+CAM|ĐẠI|DAI)[A-ZÀ-ỸĐ\s\-]{2,}\s+[" + CJK_CLASS + r"]{2,14})", "\n", s)
    parts = []
    base_parts = []
    for chunk in s.split("\n"):
        chunk = chunk.strip()
        if not chunk:
            continue
        base_parts.extend(_base_v12_split_sentences(chunk, paragraph_type) if _base_v12_split_sentences else [chunk])
    for p in base_parts:
        p2, _ = clean_mixed_script_text(p, None, context="sentence_part")
        if p2:
            parts.append(p2)
    return parts

print("Loaded DNTC v12 mixed-script patch")


Loaded DNTC v12 mixed-script patch


In [20]:

# ============================================================
# 8L. DNTC v13 patch: PAGE_RANGES truth, reading-order reconstruction,
#     generic entry/micro-heading detection, context OCR-fragment quarantine.
# ============================================================
# This cell is deliberately mechanism-based. It does not hard-code individual
# sentences; it defines range filtering, layout ordering, entry label detection,
# and context-aware fragment quarantine for the whole corpus.

PIPELINE_VERSION = "dntc_auto_v13_page_ranges_entry_layout"

PAGE_RANGES = {
    "01.pdf": [(21, 127)],
    "05.pdf": [(15, 140)],
    "07_08.pdf": [(23, 90), (95, 206)],
    "09.pdf": [(23, 139)],
    "10_11.pdf": [(13, 129)],
    "12.pdf": [(9, 112)],
    "13.pdf": [(7, 120)],
    "14_15.pdf": [(7, 168)],
    "16_17.pdf": [(14, 128), (138, 293)],
    "q2_3_4.pdf": [(13, 499)],
    "q6.pdf": [(3, 525)],
}
PAGE_STARTS = {name: ranges[0][0] for name, ranges in PAGE_RANGES.items()}
PAGE_ENDS = {name: ranges[-1][1] for name, ranges in PAGE_RANGES.items()}

# Global audit sink filled during processing and post-processing.
excluded_pages_by_page_range = globals().get("excluded_pages_by_page_range", [])
LAYOUT_SUSPECT_RECORDS = globals().get("LAYOUT_SUSPECT_RECORDS", [])
CONTEXT_DROPPED_LINE_RECORDS = globals().get("CONTEXT_DROPPED_LINE_RECORDS", [])


def _range_key_from_pdf(pdf_path_or_name) -> str:
    try:
        p = Path(str(pdf_path_or_name))
        name = p.name
        if name in PAGE_RANGES:
            return name
        # Handle work_id/stem inputs.
        if p.suffix.lower() == ".pdf":
            return name
        stem = p.stem if p.suffix else str(pdf_path_or_name)
        cand = f"{stem}.pdf"
        return cand if cand in PAGE_RANGES else name
    except Exception:
        s = str(pdf_path_or_name)
        return s if s in PAGE_RANGES else (s + ".pdf" if s + ".pdf" in PAGE_RANGES else s)


def _range_key_from_work_id(work_id: str) -> str:
    s = str(work_id)
    if s in PAGE_RANGES:
        return s
    cand = s + ".pdf"
    return cand if cand in PAGE_RANGES else s


def in_page_ranges(pdf_path_or_name=None, page_number=None, work_id=None) -> bool:
    if page_number is None or pd.isna(page_number):
        return False
    key = _range_key_from_pdf(pdf_path_or_name) if pdf_path_or_name is not None else _range_key_from_work_id(work_id)
    ranges = PAGE_RANGES.get(key, [])
    try:
        pg = int(page_number)
    except Exception:
        return False
    return any(int(a) <= pg <= int(b) for a, b in ranges)


def page_range_label(pdf_path_or_name=None, work_id=None) -> str:
    key = _range_key_from_pdf(pdf_path_or_name) if pdf_path_or_name is not None else _range_key_from_work_id(work_id)
    return ";".join(f"{a}-{b}" for a, b in PAGE_RANGES.get(key, []))


def first_page_in_ranges(pdf_path_or_name=None, work_id=None) -> int:
    key = _range_key_from_pdf(pdf_path_or_name) if pdf_path_or_name is not None else _range_key_from_work_id(work_id)
    ranges = PAGE_RANGES.get(key)
    return int(ranges[0][0]) if ranges else 1


def row_in_page_ranges(row) -> bool:
    pdf_path = row.get("pdf_path", None) if hasattr(row, "get") else None
    work_id = row.get("work_id", None) if hasattr(row, "get") else None
    page_number = row.get("page_number", None) if hasattr(row, "get") else None
    return in_page_ranges(pdf_path, page_number, work_id)


# Override content-start estimator to use first verified range. This keeps old APIs stable.
def estimate_content_start(doc, work_id, pdf_path=None):
    return first_page_in_ranges(pdf_path, work_id)


# Generic entry/micro-heading detector.
ENTRY_KEYWORDS_V13 = (
    "kiều", "cầu", "thủy-quan", "thuỷ-quan", "thủy quan", "thuỷ quan",
    "môn", "đài", "điện", "cung", "miếu", "thành", "trì", "phủ", "huyện",
    "xã", "thôn", "trạm", "tấn", "quan", "núi", "sông", "chợ", "đền", "chùa",
    "lăng", "mộ", "viện", "phường", "lầu", "đình", "tạ", "vu", "sương", "các", "đường", "án"
)
ENTRY_KEYWORD_PATTERN_V13 = r"(?:" + "|".join(re.escape(x) for x in sorted(ENTRY_KEYWORDS_V13, key=len, reverse=True)) + r")"
ENTRY_LABEL_RE_V13 = re.compile(
    rf"(?P<label>(?:[A-ZÀ-ỸĐa-zà-ỹđ][A-Za-zÀ-ỹĐđ'\-]*\s+){{0,10}}{ENTRY_KEYWORD_PATTERN_V13})\s*(?P<cjk>[{CJK_CLASS}]{{1,16}})?\s*(?P<colon>[:：])?",
    re.I,
)
ENTRY_START_RE_V13 = re.compile(
    rf"^(?P<label>(?:[A-ZÀ-ỸĐa-zà-ỹđ][A-Za-zÀ-ỹĐđ'\-]*\s+){{0,10}}{ENTRY_KEYWORD_PATTERN_V13})\s*(?P<cjk>[{CJK_CLASS}]{{1,16}})?\s*(?P<colon>[:：])?(?=\s|$)",
    re.I,
)
ENTRY_BODY_OPENER_RE_V13 = re.compile(r"^(?:tên\s+cũ|đầu|dựng|xây|năm|ở|phía|cất|mở|đổi|lại|nay|nguyên|có)\b", re.I)
UPPER_HEADING_AFTER_DOT_RE_V13 = re.compile(
    rf"(?<=[.!?])\s+(?=(?:[A-ZÀ-ỸĐ][A-ZÀ-ỸĐ\s\-–—]{{2,80}}(?:\s+[{CJK_CLASS}]{{1,16}})?)(?:\s|$))"
)
MULTI_ENTRY_LABEL_RE_V13 = re.compile(
    rf"(?:{ENTRY_KEYWORD_PATTERN_V13})\s*(?:[{CJK_CLASS}]{{1,16}})?\s*[:：]",
    re.I,
)


def has_cjk(text: str) -> bool:
    return bool(CJK_RE.search(str(text or "")))


def cjk_count(text: str) -> int:
    return len(CJK_RE.findall(str(text or "")))


def is_entry_label_text(text: str) -> bool:
    t = norm_text(text)
    if not t:
        return False
    _big_heading_fn = globals().get("_is_big_mixed_heading_text")
    if callable(_big_heading_fn) and _big_heading_fn(t):
        return True
    if globals().get("_base_is_heading_text_v13") and _base_is_heading_text_v13(t):
        return True
    if len(t) <= 140:
        m = ENTRY_START_RE_V13.match(t)
        if m:
            rest = norm_text(t[m.end():])
            # A standalone label, a label with CJK, or a label ending in colon is a micro-heading.
            if m.group("colon") or m.group("cjk") or not rest or len(rest.split()) <= 4:
                return True
    return False


def split_entry_label_and_body(text: str):
    """Return (label, body, reason) if a line/paragraph starts with a generic entry label."""
    t = norm_text(text)
    m = ENTRY_START_RE_V13.match(t)
    if not m:
        return None, t, ""
    label = norm_text((m.group("label") or "") + (" " + m.group("cjk") if m.group("cjk") else ""))
    rest = norm_text(t[m.end():])
    if rest.startswith(":") or rest.startswith("："):
        rest = norm_text(rest[1:])
    # If no colon, split only when the remainder looks like body prose.
    if not m.group("colon") and rest and not ENTRY_BODY_OPENER_RE_V13.match(rest):
        return None, t, ""
    if label and (m.group("colon") or m.group("cjk") or len(label.split()) <= 8):
        return label, rest, "entry_label_reconstruction"
    return None, t, ""


# Reading order reconstruction.
def _safe_bbox(row):
    try:
        b = row.get("bbox", [])
        if isinstance(b, str):
            b = json.loads(b)
        if len(b) == 4:
            return [float(x) for x in b]
    except Exception:
        pass
    return None


def _line_h(row):
    b = _safe_bbox(row)
    return max(1.0, b[3] - b[1]) if b else 10.0


def _line_yc(row):
    b = _safe_bbox(row)
    return (b[1] + b[3]) / 2.0 if b else 0.0


def _line_x0(row):
    b = _safe_bbox(row)
    return b[0] if b else 0.0


def _line_x1(row):
    b = _safe_bbox(row)
    return b[2] if b else 0.0


def merge_same_baseline_fragments(lines: list) -> list:
    if not lines:
        return []
    rows = [dict(r) for r in lines]
    hs = [_line_h(r) for r in rows]
    med_h = float(np.median(hs)) if hs else 10.0
    tol = max(3.0, med_h * 0.42)
    rows.sort(key=lambda r: (_line_yc(r), _line_x0(r)))
    baselines = []
    for r in rows:
        yc = _line_yc(r)
        placed = False
        for group in baselines:
            if abs(np.median([_line_yc(x) for x in group]) - yc) <= tol:
                group.append(r)
                placed = True
                break
        if not placed:
            baselines.append([r])
    out = []
    for group in baselines:
        group.sort(key=_line_x0)
        cur = None
        for r in group:
            if cur is None:
                cur = dict(r)
                continue
            gap = _line_x0(r) - _line_x1(cur)
            cur_text = norm_text(cur.get("raw_text") or cur.get("text", ""))
            r_text = norm_text(r.get("raw_text") or r.get("text", ""))
            # Merge only close fragments on same baseline. Large x gaps become layout suspects.
            if gap <= max(28.0, med_h * 2.2):
                joined = norm_text(cur_text + " " + r_text)
                cur["raw_text"] = joined if "raw_text" in cur else cur.get("raw_text", joined)
                cur["text"] = joined if "text" in cur else cur.get("text", joined)
                cb = _safe_bbox(cur); rb = _safe_bbox(r)
                if cb and rb:
                    cur["bbox"] = [min(cb[0], rb[0]), min(cb[1], rb[1]), max(cb[2], rb[2]), max(cb[3], rb[3])]
                cur["layout_merge_reason"] = ";".join(sorted(set((str(cur.get("layout_merge_reason", "")).split(";") if cur.get("layout_merge_reason") else []) + ["same_baseline_fragment_merge"])))
            else:
                if gap > max(80.0, med_h * 6.0):
                    LAYOUT_SUSPECT_RECORDS.append({
                        "work_id": r.get("work_id", cur.get("work_id", "")),
                        "page_number": r.get("page_number", cur.get("page_number", "")),
                        "reason": "large_same_baseline_x_gap",
                        "prev_text": cur_text,
                        "current_text": r_text,
                        "x_gap": round(float(gap), 2),
                    })
                out.append(cur)
                cur = dict(r)
        if cur is not None:
            out.append(cur)
    return out


def order_lines_by_layout(lines: list, page_width=None, page_height=None) -> list:
    if not lines:
        return []
    rows = merge_same_baseline_fragments(lines)
    bbs = [_safe_bbox(r) for r in rows if _safe_bbox(r)]
    if not bbs:
        return sorted(rows, key=lambda r: (r.get("page_number", 0), 0))
    max_x = max(b[2] for b in bbs)
    min_x = min(b[0] for b in bbs)
    width = float(page_width or max_x or 1.0)
    centers = [(_line_x0(r) + _line_x1(r)) / 2.0 for r in rows]
    line_widths = [max(1.0, _line_x1(r) - _line_x0(r)) for r in rows]
    left = [r for r in rows if ((_line_x0(r) + _line_x1(r)) / 2.0) < width * 0.48]
    right = [r for r in rows if ((_line_x0(r) + _line_x1(r)) / 2.0) > width * 0.52]
    two_col = len(left) >= 5 and len(right) >= 5 and np.median(line_widths) < width * 0.68
    if two_col:
        for r in rows:
            cx = (_line_x0(r) + _line_x1(r)) / 2.0
            r["layout_column"] = 0 if cx < width * 0.5 else 1
        rows = sorted(rows, key=lambda r: (int(r.get("layout_column", 0)), _line_yc(r), _line_x0(r)))
    else:
        rows = sorted(rows, key=lambda r: (_line_yc(r), _line_x0(r)))
    # Mark footnote zone if possible. Actual paragraph reflow will split it.
    if page_height:
        for r in rows:
            b = _safe_bbox(r)
            if b and b[1] > page_height * 0.82:
                r["layout_zone"] = "footnote_zone"
    return rows


# Wrap extraction functions so downstream pipeline receives layout-ordered lines.
_base_extract_pdf_text_layer_lines_v13 = globals().get("_base_extract_pdf_text_layer_lines_v13") or globals().get("extract_pdf_text_layer_lines")
_base_extract_tesseract_lines_v13 = globals().get("_base_extract_tesseract_lines_v13") or globals().get("extract_tesseract_lines")


def extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path) -> list:
    lines = _base_extract_pdf_text_layer_lines_v13(page, page_idx, work_id, pdf_path)
    return order_lines_by_layout(lines, page_width=float(page.rect.width), page_height=float(page.rect.height))


def extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=PAGE_OCR_DPI) -> list:
    lines = _base_extract_tesseract_lines_v13(page, page_idx, work_id, pdf_path, dpi=dpi)
    return order_lines_by_layout(lines, page_width=float(page.rect.width), page_height=float(page.rect.height))


# Override heading and line classification to treat entry labels as structural boundaries.
_base_is_heading_text_v13 = globals().get("_base_is_heading_text_v13") or globals().get("is_heading_text")

def is_heading_text(text: str) -> bool:
    t = norm_text(text)
    if not t:
        return False
    if is_entry_label_text(t):
        return True
    return bool(_base_is_heading_text_v13(t))


_base_filter_line_v13 = globals().get("_base_v12_filter_line") or globals().get("_base_filter_line_v13") or globals().get("filter_line")

def _call_base_filter_line_v13(raw_line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None):
    """Call the filter_line implementation produced by earlier patch cells.

    Older cells in this notebook use different signatures:
      - v3-v11: filter_line(line, page_class, correction_log, doc=None, reocr_state=None)
      - v12:     filter_line(line, reocr_state=None)
    The v13 wrapper must support both, otherwise Run All fails with
    `unexpected keyword argument 'doc'` on Kaggle.
    """
    candidates = [
        lambda: _base_filter_line_v13(raw_line, page_class, correction_log, doc=doc, reocr_state=reocr_state),
        lambda: _base_filter_line_v13(raw_line, page_class, correction_log, reocr_state=reocr_state),
        lambda: _base_filter_line_v13(raw_line, page_class, correction_log),
        lambda: _base_filter_line_v13(raw_line, reocr_state=reocr_state),
        lambda: _base_filter_line_v13(raw_line, reocr_state),
        lambda: _base_filter_line_v13(raw_line),
    ]
    last_type_error = None
    for fn in candidates:
        try:
            return fn()
        except TypeError as e:
            msg = str(e)
            # Only swallow signature mismatch errors. Real TypeErrors inside the
            # underlying function should still surface.
            if any(k in msg for k in ["unexpected keyword", "positional", "required positional", "takes ", "multiple values"]):
                last_type_error = e
                continue
            raise
    raise last_type_error or TypeError("No compatible base filter_line signature")

def filter_line(raw_line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None):
    """v13 final line filter.

    This intentionally bypasses the v12 filter wrapper when possible because v12
    changed signature to filter_line(line, reocr_state=None) while the main
    process_pdf still calls filter_line(line, page_class, correction_log, ...).
    We call the v11/base filter with a compatible signature, then apply the v12
    mixed-script cleanup and the v13 entry/fragment logic here.
    """
    kept, is_kept, reasons = _call_base_filter_line_v13(raw_line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
    reasons = list(reasons or [])
    if is_kept and kept is not None:
        txt = norm_text(kept.get("text", "") or kept.get("raw_text", ""))
        meta = {
            "work_id": kept.get("work_id") or raw_line.get("work_id"),
            "pdf_path": str(kept.get("pdf_path") or raw_line.get("pdf_path", "")),
            "page_number": kept.get("page_number") or raw_line.get("page_number"),
            "line_id": kept.get("line_global_id") or raw_line.get("line_global_id"),
            "line_global_id": kept.get("line_global_id") or raw_line.get("line_global_id"),
        }

        # v12 mixed-script cleanup, kept here so v13 is self-contained even if
        # earlier wrappers have incompatible signatures.
        if "clean_mixed_script_text" in globals():
            cleaned, extra = clean_mixed_script_text(txt, meta, context="line")
            extra = list(extra or [])
            reasons.extend(extra)
            if not cleaned and txt:
                row = dict(raw_line)
                row.update({"text": txt, "reason": "mixed_script_cleaned_empty"})
                MIXED_SCRIPT_QUARANTINE_RECORDS.append(row)
                return None, False, sorted(set(reasons + ["mixed_script_cleaned_empty"]))
            txt = norm_text(cleaned)
            kept["text"] = txt
            if extra:
                old_flags = [x for x in str(kept.get("suspicious_reasons", "")).split(";") if x]
                kept["suspicious_reasons"] = ";".join(sorted(set(old_flags + extra)))

        label, body, reason = split_entry_label_and_body(txt)
        if label and not body:
            kept["line_type"] = "heading"
            kept["entry_label_reason"] = reason
        elif label and body:
            # Preserve full text for later paragraph splitter, but mark this line as a structural boundary.
            kept["line_type"] = "heading"
            kept["entry_label_reason"] = reason
            kept["entry_inline_body"] = body
        if MIXED_GARBAGE_LINE_RE.match(txt) and not is_entry_label_text(txt):
            row = dict(raw_line)
            row.update({"text": txt, "reason": "mixed_script_ocr_garbage_fragment"})
            MIXED_SCRIPT_QUARANTINE_RECORDS.append(row)
            return None, False, sorted(set(reasons + ["mixed_script_ocr_garbage_fragment"]))
    return kept, is_kept, reasons


_base_reflow_lines_to_paragraphs_v13 = globals().get("_base_reflow_lines_to_paragraphs_v13") or globals().get("reflow_lines_to_paragraphs")

def reflow_lines_to_paragraphs(lines: list, page_rect) -> list:
    if not lines:
        return []
    lines = order_lines_by_layout(lines, page_width=float(getattr(page_rect, "width", 0) or 0), page_height=float(getattr(page_rect, "height", 0) or 0))
    for r in lines:
        r["is_footnote"] = is_footnote_line(r, page_rect) or r.get("layout_zone") == "footnote_zone"
    med_h = median_line_height(lines)
    groups, cur = [], []

    def flush():
        nonlocal cur
        if cur:
            groups.append(cur)
            cur = []

    for r in lines:
        txt = norm_text(r.get("text") or r.get("raw_text", ""))
        is_struct = is_entry_label_text(txt) or r.get("line_type") == "heading"
        if is_struct:
            flush()
            groups.append([r])
            continue
        if not cur:
            cur = [r]
        elif cur[-1].get("line_type") == "heading" or is_entry_label_text(norm_text(cur[-1].get("text") or cur[-1].get("raw_text", ""))):
            flush(); cur = [r]
        elif should_new_paragraph(cur[-1], r, med_h, page_rect):
            flush(); cur = [r]
        else:
            cur.append(r)
    flush()

    out = []
    for i, g in enumerate(groups):
        # Inline entry label + body line becomes two paragraphs.
        if len(g) == 1:
            t0 = norm_text(g[0].get("text") or g[0].get("raw_text", ""))
            label, body, reason = split_entry_label_and_body(t0)
            if label and body:
                b = _safe_bbox(g[0]) or [0, 0, 0, 0]
                qh = text_metrics(label)
                out.append({
                    "paragraph_index_in_page": len(out), "paragraph_type": "micro_heading", "text": label, "bbox": b,
                    "line_count": 1, "source": g[0].get("source", ""), "quality_score": qh["quality_score"],
                    "vietnamese_ratio": qh["vietnamese_ratio"], "weird_char_ratio": qh["weird_char_ratio"],
                    "line_ids": g[0].get("line_global_id", ""), "entry_reconstruction_reason": reason,
                })
                qb = text_metrics(body)
                out.append({
                    "paragraph_index_in_page": len(out), "paragraph_type": "body", "text": body, "bbox": b,
                    "line_count": 1, "source": g[0].get("source", ""), "quality_score": qb["quality_score"],
                    "vietnamese_ratio": qb["vietnamese_ratio"], "weird_char_ratio": qb["weird_char_ratio"],
                    "line_ids": g[0].get("line_global_id", ""), "entry_reconstruction_reason": "body_after_" + reason,
                })
                continue
        text = join_paragraph_lines(g)
        if not text:
            continue
        bbs = [_safe_bbox(x) for x in g if _safe_bbox(x)]
        b = [min(x[0] for x in bbs), min(x[1] for x in bbs), max(x[2] for x in bbs), max(x[3] for x in bbs)] if bbs else [0,0,0,0]
        if all((x.get("line_type") == "heading" or is_entry_label_text(norm_text(x.get("text") or x.get("raw_text", "")))) for x in g) and len(text) <= 180:
            ptype = "micro_heading" if any(split_entry_label_and_body(norm_text(x.get("text") or x.get("raw_text", "")))[0] for x in g) else "heading"
        elif any(x.get("is_footnote") for x in g):
            ptype = "footnote"
        else:
            ptype = "body"
        q = text_metrics(text, lines=[x.get("text", "") for x in g])
        out.append({
            "paragraph_index_in_page": len(out), "paragraph_type": ptype, "text": text, "bbox": b,
            "line_count": len(g), "source": ";".join(sorted(set(x.get("source", "") for x in g))),
            "quality_score": q["quality_score"], "vietnamese_ratio": q["vietnamese_ratio"],
            "weird_char_ratio": q["weird_char_ratio"], "line_ids": ";".join(x.get("line_global_id", "") for x in g),
        })
    return out


# Generic context-aware OCR fragment detection after all lines are available.
def is_valid_short_content(text: str, prev_text: str = "", next_text: str = "") -> bool:
    t = norm_text(text)
    if not t:
        return False
    if is_entry_label_text(t) or is_heading_text(t):
        return True
    if re.search(r"\d+\s*(?:dặm|trượng|thước|tấc|phân|mẫu|sào|đồng|quan|người)\.?$", t, re.I):
        return True
    if DANGLING_PREV_END_RE.search(norm_text(prev_text)):
        return True
    if MEANINGFUL_SHORT_RE.search(t) and len(t) >= 5:
        return True
    if has_cjk(t) and len(t) <= 16 and (is_entry_label_text(prev_text) or is_entry_label_text(next_text)):
        return True
    return False


def is_isolated_ocr_fragment_near_context(text: str, prev_text: str = "", next_text: str = ""):
    t = norm_text(text)
    if not t:
        return False, "empty_fragment"
    if is_valid_short_content(t, prev_text, next_text):
        return False, ""
    compact = re.sub(r"\s+", "", t)
    near_struct = has_cjk(prev_text) or has_cjk(next_text) or is_heading_text(prev_text) or is_heading_text(next_text) or is_entry_label_text(prev_text) or is_entry_label_text(next_text)
    if MIXED_GARBAGE_LINE_RE.match(t) or MIXED_GARBAGE_TOKEN_RE.search(t):
        return True, "isolated_ocr_fragment_near_cjk_or_heading"
    if len(compact) <= 4:
        if re.fullmatch(r"[A-Za-zÀ-ỹĐđ0-9\W_]+", t) and near_struct:
            return True, "isolated_ocr_fragment_near_cjk_or_heading"
        letters = LETTERS_RE.findall(t)
        if len(letters) <= 2 and not re.search(r"\d+\s*(?:dặm|đ|quan|tiền)", t, re.I):
            return True, "isolated_short_ocr_fragment"
    if len(t) <= 8 and re.search(r"[ˆ¬¿¡ÐÑÖ§□�*#`^+]|\b(?:wt|bik|kt|ah|or)\b", t, re.I):
        return True, "isolated_ocr_fragment_near_cjk_or_heading"
    return False, ""


print("DNTC v13 patch loaded: PAGE_RANGES, layout ordering, entry reconstruction, context quarantine.")


DNTC v13 patch loaded: PAGE_RANGES, layout ordering, entry reconstruction, context quarantine.


In [21]:

# ============================================================
# 9. Process one PDF and all PDFs
# ============================================================
all_final_lines = []
all_paragraphs = []
all_sentences = []
page_quality_report = []
source_selection_report = []
line_filter_audit = []
dropped_pages = []
dropped_lines = []
correction_log = []
suspicious_lines = []
suspicious_sentences = []
excluded_pages_by_page_range = globals().get("excluded_pages_by_page_range", [])


def make_work_id(pdf_path: Path, used: set) -> str:
    base = re.sub(r"[^A-Za-z0-9_\-]+", "_", pdf_path.stem).strip("_") or "pdf"
    wid = base
    k = 2
    while wid in used:
        wid = f"{base}_{k}"
        k += 1
    used.add(wid)
    return wid


def suspicious_reasons_for_text(text: str, q: dict) -> list:
    reasons = []
    if CLEAR_JUNK_RE.search(norm_text(text)):
        reasons.append("known_junk_pattern")
    if q["quality_score"] < MIN_KEEP_SENTENCE_QUALITY:
        reasons.append(f"low_quality:{q['quality_score']:.1f}")
    if q["weird_char_ratio"] > 0.035:
        reasons.append("weird_char_ratio")
    if q["word_count"] >= 8 and q["vietnamese_ratio"] < MIN_VIET_RATIO_FOR_LONG_TEXT:
        reasons.append("low_vietnamese_ratio")
    if q["repeated_char_ngram_ratio"] > 0.05:
        reasons.append("repeated_ngram_ratio")
    if q["library_noise_score"] >= MAX_LIBRARY_NOISE_SCORE:
        reasons.append("library_noise")
    if q["unaccented_vi_hits"] >= 3 and q["accent_ratio"] < 0.015:
        reasons.append("possible_missing_diacritics")
    if globals().get("STRICT_SENTENCE_ONLY", True):
        try:
            if looks_like_sentence_fragment(text):
                reasons.append("sentence_fragment_shape")
            if not ends_hard_sentence(text) and q["word_count"] < 18:
                reasons.append("no_terminal_punctuation")
        except Exception:
            pass
    return reasons


def process_pdf(pdf_path: Path, work_id: str, remaining_page_budget=None):
    print(f"\nProcessing {work_id}: {pdf_path}")
    try:
        doc = fitz.open(str(pdf_path))
    except Exception as e:
        dropped_pages.append({"work_id": work_id, "pdf_path": str(pdf_path), "page_number": None, "reason": f"cannot_open:{type(e).__name__}"})
        print("Cannot open PDF:", e)
        return 0

    total_pages = len(doc)
    content_start_page = estimate_content_start(doc, work_id, pdf_path)
    print("pages:", total_pages, "auto_content_start_page:", content_start_page)
    page_limit = total_pages if MAX_PAGES_PER_PDF is None else min(total_pages, MAX_PAGES_PER_PDF)
    if remaining_page_budget is not None:
        page_limit = min(page_limit, remaining_page_budget)
    reocr_state = {"used": 0}
    processed_pages = 0

    for page_idx in tqdm(range(page_limit), desc=work_id):
        processed_pages += 1
        page = doc[page_idx]
        page_number = page_idx + 1
        if not in_page_ranges(pdf_path, page_number, work_id):
            # PAGE_RANGES is the source of truth. Do not OCR/export pages outside verified content ranges.
            try:
                visual_m = visual_page_metrics(page)
                tl_lines_for_range = extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path)
                tl_text_for_range = lines_to_text(tl_lines_for_range)
                range_m = text_metrics(tl_text_for_range, lines=[r.get("raw_text", "") for r in tl_lines_for_range])
            except Exception:
                visual_m, range_m = {}, text_metrics("")
            ex_row = {
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                "page_ranges": page_range_label(pdf_path, work_id), "page_class": "excluded_by_page_range",
                "drop_reason": "outside_PAGE_RANGES", "selected_source": "excluded_by_page_range",
                **{f"visual_{k}": v for k, v in visual_m.items()},
                **{f"class_text_{k}": v for k, v in range_m.items()},
                "kept_lines": 0, "dropped_lines": 0, "final_sentences": 0,
            }
            excluded_pages_by_page_range.append(ex_row)
            page_quality_report.append(ex_row)
            continue
        visual_m = visual_page_metrics(page)
        tl_lines_for_class = extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path)
        tl_text_for_class = lines_to_text(tl_lines_for_class)
        tl_m_for_class = text_metrics(tl_text_for_class, lines=[r.get("raw_text", "") for r in tl_lines_for_class])
        page_class, class_reasons = classify_page(page_number, tl_text_for_class, tl_m_for_class, visual_m, content_start_page)

        # If text layer is too poor to classify but visual has nonblank text-like page, do a quick OCR for classification.
        classification_text = tl_text_for_class
        classification_m = tl_m_for_class
        if page_class == "junk_ocr_page" and tesseract_available() and not is_blank_page(tl_m_for_class, visual_m):
            quick_lines = extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=FAST_FRONTMATTER_OCR_DPI)
            quick_text = lines_to_text(quick_lines)
            quick_m = text_metrics(quick_text, lines=[r.get("raw_text", "") for r in quick_lines], ocr_conf_values=[r.get("ocr_conf") for r in quick_lines])
            if quick_m["quality_score"] > classification_m["quality_score"]:
                classification_text, classification_m = quick_text, quick_m
                page_class, class_reasons = classify_page(page_number, classification_text, classification_m, visual_m, content_start_page)

        report = {
            "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
            "content_start_page": content_start_page, "page_class": page_class,
            "class_reasons": ";".join(class_reasons) if class_reasons else "",
            **{f"visual_{k}": v for k, v in visual_m.items()},
            **{f"class_text_{k}": v for k, v in classification_m.items()},
        }

        if page_class in {"blank_page", "patterned_endpaper", "library_barcode_stamp_watermark", "cover_title_front_matter", "front_matter_before_content", "publisher_backmatter_page"}:
            report.update({"selected_source": "dropped_by_page_classifier", "kept_lines": 0, "dropped_lines": 0, "final_sentences": 0})
            page_quality_report.append(report)
            dropped_pages.append({
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                "page_class": page_class, "drop_reason": ";".join(class_reasons) if class_reasons else page_class,
                **classification_m,
            })
            continue

        selected_lines, selected_source, selected_m, tl_m, ocr_m, source_reason = select_page_source(page, page_idx, work_id, pdf_path, page_class)
        source_selection_report.append({
            "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
            "page_class": page_class, "selected_source": selected_source, "source_reason": source_reason,
            **{f"text_layer_{k}": v for k, v in tl_m.items()},
            **{f"ocr_{k}": v for k, v in (ocr_m or {}).items()},
        })

        if not selected_lines or (DROP_LOW_CONF_OCR_PAGES and selected_m.get("quality_score", 0) < MIN_SELECTED_PAGE_QUALITY):
            report.update({"selected_source": selected_source, "kept_lines": 0, "dropped_lines": len(selected_lines), "final_sentences": 0})
            page_quality_report.append(report)
            dropped_pages.append({
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                "page_class": "junk_ocr_page" if selected_source == "drop_no_reliable_source" else page_class,
                "drop_reason": f"low_selected_source_quality:{selected_m.get('quality_score', 0):.1f};{source_reason}",
                **selected_m,
            })
            continue

        kept_lines = []
        dropped_count = 0
        seen_line_texts = Counter()
        for li, raw_line in enumerate(selected_lines):
            raw_line = dict(raw_line)
            raw_line["line_global_id"] = f"{work_id}_p{page_number:04d}_l{li+1:03d}"
            kept, is_kept, reasons = filter_line(raw_line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
            audit_row = {
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                "line_id": raw_line["line_global_id"], "source": raw_line.get("source"), "page_class": page_class,
                "raw_text": raw_line.get("raw_text", ""), "kept": bool(is_kept), "reasons": ";".join(reasons),
                "bbox": json.dumps(raw_line.get("bbox", []), ensure_ascii=False), "ocr_conf": raw_line.get("ocr_conf"),
            }
            if is_kept and kept is not None:
                # Drop duplicate exact line repeated on same page, except headings.
                key = normalize_for_match(kept["text"])
                seen_line_texts[key] += 1
                if seen_line_texts[key] > 2 and kept.get("line_type") != "heading":
                    is_kept = False
                    reasons = reasons + ["duplicate_repeated_line"]
                    audit_row.update({"kept": False, "reasons": ";".join(reasons), "final_text": kept["text"]})
                else:
                    kept["page_class"] = page_class
                    kept["selected_source"] = selected_source
                    kept["source_reason"] = source_reason
                    kept_lines.append(kept)
                    audit_row.update({"final_text": kept["text"], "line_quality_score": kept.get("line_quality_score")})
                    sr = suspicious_reasons_for_text(kept["text"], line_quality(kept["text"], kept.get("ocr_conf")))
                    if sr:
                        suspicious_lines.append({**audit_row, "suspicious_reasons": ";".join(sr)})
            if not is_kept:
                dropped_count += 1
                dropped_lines.append({**audit_row, "kept": False, "reasons": ";".join(reasons)})
            line_filter_audit.append(audit_row)

        paragraphs = reflow_lines_to_paragraphs(kept_lines, page.rect)
        page_sentence_count = 0
        for kline in kept_lines:
            row = dict(kline)
            row["bbox"] = json.dumps(row.get("bbox", []), ensure_ascii=False)
            all_final_lines.append(row)

        for pi, para in enumerate(paragraphs):
            paragraph_id = f"{work_id}_p{page_number:04d}_para{pi+1:03d}"
            prow = {
                "paragraph_id": paragraph_id, "work_id": work_id, "pdf_path": str(pdf_path),
                "page_idx": page_idx, "page_number": page_number, **para,
                "bbox": json.dumps(para.get("bbox", []), ensure_ascii=False),
            }
            all_paragraphs.append(prow)
            if para["paragraph_type"] == "heading":
                continue
            sentences = split_sentences_vietnamese(para["text"], para["paragraph_type"])
            for si, sent in enumerate(sentences):
                sq = text_metrics(sent)
                s_reasons = suspicious_reasons_for_text(sent, sq)
                sent_id = f"{work_id}_s{len(all_sentences)+1:07d}"
                srow = {
                    "sent_id": sent_id, "paragraph_id": paragraph_id, "work_id": work_id,
                    "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                    "sentence_index_in_paragraph": si, "text": sent, "source": para.get("source"),
                    "paragraph_type": para.get("paragraph_type"), "bbox": prow["bbox"],
                    "quality_score": sq["quality_score"], "vietnamese_ratio": sq["vietnamese_ratio"],
                    "weird_char_ratio": sq["weird_char_ratio"], "dictionary_hit_ratio": sq["dictionary_hit_ratio"],
                    "suspicious_reasons": ";".join(s_reasons),
                }
                if s_reasons:
                    suspicious_sentences.append(srow)
                    # Keep suspicious if it is still above hard floor; otherwise drop from final sentence-only.
                    if (sq["quality_score"] < MIN_KEEP_SENTENCE_QUALITY or
                        "sentence_fragment_shape" in s_reasons or "no_terminal_punctuation" in s_reasons):
                        continue
                all_sentences.append(srow)
                page_sentence_count += 1

        report.update({
            "selected_source": selected_source, "source_reason": source_reason,
            **{f"selected_{k}": v for k, v in selected_m.items()},
            "kept_lines": len(kept_lines), "dropped_lines": dropped_count,
            "paragraphs": len(paragraphs), "final_sentences": page_sentence_count,
            "line_reocr_used_pdf": reocr_state.get("used", 0),
        })
        page_quality_report.append(report)

    doc.close()
    return processed_pages

used_ids = set()
page_budget = FAST_TEST_MAX_PAGES_TOTAL if FAST_TEST_MODE else None
start_time = time.time()
processed_pages_total = 0

for pdf_i, pdf_path in enumerate(pdf_paths, 1):
    if page_budget is not None and page_budget <= 0:
        break
    work_id = make_work_id(pdf_path, used_ids)
    n = process_pdf(pdf_path, work_id, remaining_page_budget=page_budget)
    processed_pages_total += n
    if page_budget is not None:
        page_budget -= n

elapsed = time.time() - start_time
print("\nDone processing")
print("processed_pages:", processed_pages_total)
print("final_lines:", len(all_final_lines))
print("paragraphs:", len(all_paragraphs))
print("sentences:", len(all_sentences))
print("dropped_pages:", len(dropped_pages))
print("dropped_lines:", len(dropped_lines))
print("elapsed_min:", round(elapsed / 60, 2))




Processing 01: /kaggle/working/dntc_auto/raw_drive/01.pdf
pages: 136 auto_content_start_page: 21


01:   0%|          | 0/136 [00:00<?, ?it/s]

TypeError: filter_line() got multiple values for argument 'reocr_state'

In [ ]:

# ============================================================
# 9A. DNTC v13 post-process: enforce PAGE_RANGES, quarantine context fragments,
#     rebuild paragraphs from safe lines before sentence export.
# ============================================================

def _filter_rows_to_page_ranges(rows, table_name: str):
    kept, excluded = [], []
    for r in rows:
        rr = dict(r)
        if row_in_page_ranges(rr):
            kept.append(rr)
        else:
            rr["source_table"] = table_name
            rr["drop_reason"] = "outside_PAGE_RANGES_postprocess"
            excluded.append(rr)
    return kept, excluded

# Defense in depth: even if an old cell emitted rows outside ranges, remove them here.
all_final_lines, ex1 = _filter_rows_to_page_ranges(all_final_lines, "final_lines_pre_export")
all_paragraphs, ex2 = _filter_rows_to_page_ranges(all_paragraphs, "final_paragraphs_pre_export")
all_sentences, ex3 = _filter_rows_to_page_ranges(all_sentences, "final_sentences_pre_export")
if ex1 or ex2 or ex3:
    excluded_pages_by_page_range.extend(ex1 + ex2 + ex3)

# Context-aware line quarantine. This is intentionally after all pages are collected so prev/next are available.
filtered_lines = []
for (wid, pg), group in itertools.groupby(sorted(all_final_lines, key=lambda r: (str(r.get("work_id", "")), int(r.get("page_number", 0) or 0), _line_yc(r), _line_x0(r))), key=lambda r: (str(r.get("work_id", "")), int(r.get("page_number", 0) or 0))):
    glines = list(group)
    for i, r in enumerate(glines):
        prev_text = norm_text(glines[i-1].get("text", glines[i-1].get("raw_text", ""))) if i > 0 else ""
        cur_text = norm_text(r.get("text", r.get("raw_text", "")))
        next_text = norm_text(glines[i+1].get("text", glines[i+1].get("raw_text", ""))) if i + 1 < len(glines) else ""
        drop, reason = is_isolated_ocr_fragment_near_context(cur_text, prev_text, next_text)
        if drop:
            rec = dict(r)
            rec.update({
                "reason": reason,
                "reasons": ";".join([str(r.get("reasons", "")), reason]).strip(";"),
                "prev_text": prev_text,
                "current_text": cur_text,
                "next_text": next_text,
            })
            dropped_lines.append(rec)
            CONTEXT_DROPPED_LINE_RECORDS.append(rec)
            MIXED_SCRIPT_QUARANTINE_RECORDS.append(rec)
            continue
        filtered_lines.append(r)
all_final_lines = filtered_lines

# Rebuild paragraphs from the filtered, layout-ordered lines. This prevents paragraphs from retaining fragments dropped above.
rebuilt_paragraphs = []
for (wid, pg), group in itertools.groupby(sorted(all_final_lines, key=lambda r: (str(r.get("work_id", "")), int(r.get("page_number", 0) or 0), _line_yc(r), _line_x0(r))), key=lambda r: (str(r.get("work_id", "")), int(r.get("page_number", 0) or 0))):
    glines = list(group)
    if not glines:
        continue
    bbs = [_safe_bbox(x) for x in glines if _safe_bbox(x)]
    rect = fitz.Rect(0, 0, max([b[2] for b in bbs] + [100]), max([b[3] for b in bbs] + [100]))
    paras = reflow_lines_to_paragraphs(glines, rect)
    # derive stable source metadata from first line
    first = glines[0]
    for pi, para in enumerate(paras):
        paragraph_id = f"{wid}_p{int(pg):04d}_para{pi+1:03d}"
        prow = {
            "paragraph_id": paragraph_id,
            "work_id": wid,
            "pdf_path": first.get("pdf_path", ""),
            "page_idx": first.get("page_idx", int(pg)-1),
            "page_number": int(pg),
            **para,
            "bbox": json.dumps(para.get("bbox", []), ensure_ascii=False) if not isinstance(para.get("bbox"), str) else para.get("bbox"),
        }
        rebuilt_paragraphs.append(prow)
all_paragraphs = rebuilt_paragraphs
all_sentences = []
print(f"DNTC v13 postprocess: safe_lines={len(all_final_lines)}, rebuilt_paragraphs={len(all_paragraphs)}, context_quarantine={len(CONTEXT_DROPPED_LINE_RECORDS)}, range_excluded_rows={len(ex1)+len(ex2)+len(ex3)}")


In [ ]:

# ============================================================
# 9B. DNTC v5 post-process: stitch cross-page paragraphs and rebuild sentences
# ============================================================
# Processing is page-local for speed, but old books frequently continue a sentence
# from the bottom of one page to the top of the next. This pass stitches adjacent
# body paragraphs before exporting final_paragraphs/final_sentences_only.

CROSS_PAGE_DANGLING_END_RE = re.compile(
    r"\b(?:làm\s+chỗ|đề|để|mỗi|của|và|ở|là|theo|có|gồm|đem|cho|về|trong|ngoài|phía|cách)\s*$",
    re.I,
)
CROSS_PAGE_LOWER_START_RE = re.compile(r"^[a-zà-ỹđ]", re.I)


def should_stitch_paragraph_rows(prev: dict, cur: dict) -> bool:
    if not prev or not cur:
        return False
    if str(prev.get("work_id")) != str(cur.get("work_id")):
        return False
    if prev.get("paragraph_type") in {"heading", "micro_heading"} or cur.get("paragraph_type") in {"heading", "micro_heading"}:
        return False
    pt = apply_dntc_reflow_corrections(str(prev.get("text", "")))
    ct = apply_dntc_reflow_corrections(str(cur.get("text", "")))
    if not pt or not ct:
        return False
    # Only adjacent flow; across page or same page after geometry produced false split.
    try:
        pp = int(prev.get("page_number"))
        cp = int(cur.get("page_number"))
        if cp < pp or cp > pp + 1:
            return False
    except Exception:
        pass
    if CROSS_PAGE_DANGLING_END_RE.search(pt):
        return True
    if (not ends_hard_sentence(pt)) and (CROSS_PAGE_LOWER_START_RE.match(ct) or begins_continuation(ct)):
        return True
    return False


def stitch_paragraph_rows(rows: list) -> list:
    if not rows:
        return rows
    rows = sorted(rows, key=lambda r: (str(r.get("work_id", "")), int(r.get("page_number", 0) or 0), int(r.get("paragraph_index_in_page", 0) or 0)))
    stitched = []
    for r in rows:
        r = dict(r)
        r["text"] = apply_dntc_reflow_corrections(str(r.get("text", "")))
        if stitched and should_stitch_paragraph_rows(stitched[-1], r):
            prev = stitched[-1]
            prev["text"] = apply_dntc_reflow_corrections(str(prev.get("text", "")) + " " + str(r.get("text", "")))
            prev["line_count"] = int(prev.get("line_count", 0) or 0) + int(r.get("line_count", 0) or 0)
            prev["line_ids"] = ";".join(x for x in [str(prev.get("line_ids", "")), str(r.get("line_ids", ""))] if x and x != "nan")
            prev["stitched_from"] = ";".join(x for x in [str(prev.get("stitched_from", "")), str(r.get("paragraph_id", ""))] if x and x != "nan")
            prev["page_span"] = f"{prev.get('page_number')}-{r.get('page_number')}" if prev.get("page_number") != r.get("page_number") else str(prev.get("page_number"))
        else:
            r.setdefault("stitched_from", "")
            r.setdefault("page_span", str(r.get("page_number", "")))
            stitched.append(r)
    return stitched


def rebuild_sentences_from_paragraphs():
    global all_paragraphs, all_sentences, suspicious_sentences
    old_para_count = len(all_paragraphs)
    old_sent_count = len(all_sentences)
    all_paragraphs = stitch_paragraph_rows(all_paragraphs)
    all_sentences = []
    suspicious_sentences = []
    sent_counter = Counter()
    for para in all_paragraphs:
        if para.get("paragraph_type") in {"heading", "micro_heading"}:
            continue
        sentences = split_sentences_vietnamese(para.get("text", ""), para.get("paragraph_type", "body"))
        for si, sent in enumerate(sentences):
            sent = apply_dntc_reflow_corrections(sent)
            sq = text_metrics(sent)
            s_reasons = suspicious_reasons_for_text(sent, sq)
            # Hard fragment blocking remains in final, but keep audit rows.
            sent_counter[str(para.get("work_id"))] += 1
            sent_id = f"{para.get('work_id')}_s{sent_counter[str(para.get('work_id'))]:07d}"
            srow = {
                "sent_id": sent_id,
                "paragraph_id": para.get("paragraph_id"),
                "work_id": para.get("work_id"),
                "pdf_path": para.get("pdf_path"),
                "page_idx": para.get("page_idx"),
                "page_number": para.get("page_number"),
                "page_span": para.get("page_span", para.get("page_number")),
                "sentence_index_in_paragraph": si,
                "text": sent,
                "source": para.get("source"),
                "paragraph_type": para.get("paragraph_type"),
                "bbox": para.get("bbox"),
                "quality_score": sq["quality_score"],
                "vietnamese_ratio": sq["vietnamese_ratio"],
                "weird_char_ratio": sq["weird_char_ratio"],
                "dictionary_hit_ratio": sq["dictionary_hit_ratio"],
                "suspicious_reasons": ";".join(s_reasons),
            }
            if s_reasons:
                suspicious_sentences.append(srow)
                if (sq["quality_score"] < MIN_KEEP_SENTENCE_QUALITY or
                    "sentence_fragment_shape" in s_reasons or "no_terminal_punctuation" in s_reasons):
                    continue
            all_sentences.append(srow)
    print(f"DNTC v5 post-process: paragraphs {old_para_count} -> {len(all_paragraphs)}, sentences {old_sent_count} -> {len(all_sentences)}")

rebuild_sentences_from_paragraphs()


In [ ]:

# ============================================================
# 9C0. DNTC v12 paragraph mixed-script normalization:
#       split micro-headings, quarantine garbage fragments.
# ============================================================

def _new_para_from(row: dict, text: str, ptype: str, seq: int, reasons: list | None = None) -> dict:
    nr = dict(row)
    base_id = str(row.get("paragraph_id", "para"))
    nr["source_paragraph_id"] = row.get("source_paragraph_id", base_id)
    nr["paragraph_id"] = f"{base_id}_ms{seq:02d}"
    nr["text"] = norm_text(text)
    nr["paragraph_type"] = ptype
    nr["mixed_script_reasons"] = ";".join(sorted(set(reasons or [])))
    return nr


def _is_big_mixed_heading_text(t: str) -> bool:
    t = norm_text(t)
    if BIG_MIXED_HEADING_RE.match(t):
        return True
    if re.match(r"^(?:HOÀNG[-\s]?THÀNH|TỬ\s*CẤM[-\s]?THÀNH|ĐẠI[-\s]?NỘI|KINH[-\s]?THÀNH)\b", t, re.I):
        return True
    return False


def _find_next_micro_entry(text: str, start: int = 0):
    best = None
    for m in MIXED_ENTRY_SPLIT_RE.finditer(text, start):
        label = norm_text(m.group("label"))
        cjk = m.group("cjk") or ""
        if len(cjk) < 2:
            continue
        # Avoid false matches that eat a long sentence: the label should be fairly short and end with an entry keyword.
        if len(label) > 70 or not ENTRY_KEYWORD_RE.search(label):
            continue
        # Accept colon, or missing colon followed by common body openers.
        tail = text[m.end():m.end()+40]
        if m.group("colon") or re.match(r"\s*(?:tên\s+cũ|Đầu|Dựng|Xây|Cất|Năm|Ở|phía|là)\b", tail, re.I):
            best = m
            break
    return best


def split_paragraph_mixed_script(row: dict) -> list:
    original = norm_text(row.get("text", ""))
    meta = {"work_id": row.get("work_id"), "pdf_path": row.get("pdf_path"), "page_number": row.get("page_number"), "paragraph_id": row.get("paragraph_id")}
    text, clean_reasons = clean_mixed_script_text(original, meta, context="paragraph")
    if not text:
        _record_mixed_quarantine("paragraph_cleaned_empty", original, meta, fragment=original)
        return []
    ptype0 = str(row.get("paragraph_type", "body"))
    if ptype0 in {"heading", "micro_heading"}:
        ptype = "heading" if _is_big_mixed_heading_text(text) or ptype0 == "heading" else "micro_heading"
        return [_new_para_from(row, text.rstrip(":："), ptype, 1, clean_reasons)]

    out = []
    pos = 0
    seq = 1
    while pos < len(text):
        m = _find_next_micro_entry(text, pos)
        # Also split large mixed headings if they begin after a sentence boundary.
        big = None
        for bm in re.finditer(rf"(?:(?<=^)|(?<=[.!?]\s))(?P<h>(?:HOÀNG[-\s]?THÀNH|TỬ\s*CẤM[-\s]?THÀNH|ĐẠI[-\s]?NỘI|KINH[-\s]?THÀNH)[^.!?:;]{{0,60}}[{CJK_CLASS}]{{2,14}})", text[pos:], re.I):
            big_start = pos + bm.start("h")
            if m is None or big_start < m.start():
                big = (big_start, pos + bm.end("h"), bm.group("h"))
                break
        if big is not None:
            start, end, heading = big
            before = norm_text(text[pos:start])
            if before:
                out.append(_new_para_from(row, before, "body", seq, clean_reasons)); seq += 1
            out.append(_new_para_from(row, heading.rstrip(":："), "heading", seq, clean_reasons)); seq += 1
            pos = end
            continue
        if m is None:
            rest = norm_text(text[pos:])
            if rest:
                out.append(_new_para_from(row, rest, "body", seq, clean_reasons)); seq += 1
            break
        before = norm_text(text[pos:m.start()])
        if before:
            out.append(_new_para_from(row, before, "body", seq, clean_reasons)); seq += 1
        label = norm_text(f"{m.group('label')} {m.group('cjk')}").rstrip(":：")
        out.append(_new_para_from(row, label, "micro_heading", seq, clean_reasons)); seq += 1
        pos = m.end()
        # If the original had a colon, the body starts after it. If not, body starts immediately at pos.
    return out


_original_para_count_v12 = len(all_paragraphs)
normalized_paras = []
for row in all_paragraphs:
    try:
        normalized_paras.extend(split_paragraph_mixed_script(dict(row)))
    except Exception as e:
        rr = dict(row)
        rr["mixed_script_reasons"] = f"paragraph_mixed_script_exception:{type(e).__name__}"
        normalized_paras.append(rr)
all_paragraphs = normalized_paras
print(f"DNTC v12 paragraph normalization: { _original_para_count_v12 } -> {len(all_paragraphs)} paragraphs; quarantine={len(MIXED_SCRIPT_QUARANTINE_RECORDS)}")


In [ ]:

# ============================================================
# 9C. DNTC v12 rebuild: headings + micro_headings as sentence rows,
#     current_heading metadata, mixed-script flags synchronized.
# ============================================================

PARENT_HEADING_RE = STRUCTURAL_PARENT_HEADING_RE
CHILD_HEADING_RE = ADMIN_HEADING_PREFIX_RE


def heading_level(text: str, paragraph_type: str = "heading") -> str:
    t = norm_text(text)
    if paragraph_type == "micro_heading":
        return "child"
    if PARENT_HEADING_RE.match(t) or _is_big_mixed_heading_text(t):
        return "parent"
    if CHILD_HEADING_RE.match(t):
        return "child"
    letters = LETTERS_RE.findall(t)
    if letters:
        upper = sum(1 for ch in letters if ch.upper() == ch and ch.lower() != ch) / max(1, len(letters))
        if upper > 0.72 and len(WORDS_RE.findall(t)) <= 8:
            return "parent"
    return "child"


def _has_dangling_single_cjk(text: str) -> bool:
    t = norm_text(text)
    # Single CJK after a Latin label in body is suspicious; complete labels with >=2 CJK are allowed.
    if DANGLING_SINGLE_CJK_COLON_RE.search(t) or DANGLING_SINGLE_CJK_AFTER_LABEL_RE.search(t):
        return True
    # Lone CJK token at the end after Latin words, e.g. "lầu Ngũ Phụng 五".
    return bool(re.search(rf"[A-Za-zÀ-ỹĐđ][^.!?]{{0,40}}\s[{CJK_CLASS}][,.;:]?$", t) and not re.search(rf"[{CJK_CLASS}]{{2,}}", t))


def sentence_extra_flags(text: str, paragraph_type: str, q: dict) -> list:
    reasons = []
    t = norm_text(text)
    if DANGEROUS_ROUTE_RE.search(t):
        reasons.append("dangerous_route_or_heading_merge")
    if WEIRD_OCR_CHARS_RE.search(t):
        reasons.append("weird_ocr_char")
    if len(t) > 800:
        reasons.append("critical_overlong_sentence_gt800")
    elif len(t) > 400:
        reasons.append("overlong_sentence_gt400")
    if re.search(r"\b\d{1,3}\s*\.\s*[–-].+\b\d{1,3}\s*\.\s*[–-]", t):
        reasons.append("multiple_numbered_items_merged")
    if paragraph_type not in {"heading", "micro_heading"} and is_heading_text(t):
        reasons.append("body_looks_like_heading")
    if paragraph_type not in {"heading", "micro_heading"} and MICRO_HEADING_LABEL_RE.search(t):
        reasons.append("merged_entry_label")
    if len(MICRO_HEADING_LABEL_RE.findall(t)) >= 2:
        reasons.append("merged_entry_label")
    if MIXED_GARBAGE_TOKEN_RE.search(t) or MIXED_GARBAGE_LINE_RE.search(t):
        reasons.append("garbage_mixed_script_fragment")
    if FALSE_TINH_STRICT_RE.search(t) or re.search(r"\bTỈNH\s+(?:bik|wt|kt\s+ah|or\s*\+)", t, re.I):
        reasons.append("false_tinh_uppercase")
    if _has_dangling_single_cjk(t):
        reasons.append("cjk_fragment_incomplete")
    if re.search(r"\b(?:tên\s+Minh\s+mạng|th[uủ]y[-\s]?quan\s+mạng\s+thứ|hiện\s+Xây\s+năm)\b", t, re.I):
        reasons.append("layout_reconstruction_suspect")
    if re.search(r"\btây\s+cách\s+\d+\s+dặm,\s*từ\s+nam\s+đến\s+Tỉnh\s+này", t, re.I) or re.search(r"tây\s*-\s*HUYỆN", t):
        reasons.append("layout_order_suspect")
    if q.get("quality_score", 100) < MIN_KEEP_SENTENCE_QUALITY:
        reasons.append(f"low_quality:{q.get('quality_score', 0):.1f}")
    return sorted(set(reasons))


def _inherit_para_flags(para: dict) -> list:
    flags = []
    for key in ["mixed_script_reasons", "suspicious_reasons"]:
        val = str(para.get(key, "") or "")
        flags.extend([x for x in val.split(";") if x])
    return flags


def build_sentence_rows_from_paragraphs():
    global all_sentences, all_sentences_auto, suspicious_sentences
    para_rows = sorted(
        [dict(r) for r in all_paragraphs],
        key=lambda r: (str(r.get("work_id", "")), int(r.get("page_number", 0) or 0), int(r.get("paragraph_index", 0) or 0), str(r.get("paragraph_id", ""))),
    )
    current_parent = defaultdict(str)
    current_child = defaultdict(str)
    counters = defaultdict(int)
    auto_rows = []
    susp_rows = []

    for para in para_rows:
        work_id = str(para.get("work_id", ""))
        para_id = str(para.get("paragraph_id", ""))
        ptype = str(para.get("paragraph_type", "body") or "body")
        ptext_raw = norm_text(para.get("text", ""))
        meta = {"work_id": work_id, "pdf_path": para.get("pdf_path"), "page_number": para.get("page_number"), "paragraph_id": para_id}
        ptext, pclean_flags = clean_mixed_script_text(ptext_raw, meta, context="sentence_rebuild")
        ptext = _repair_tinh_uppercase_context(ptext, is_heading_context=ptype in {"heading", "micro_heading"})
        if not ptext:
            _record_mixed_quarantine("paragraph_empty_after_sentence_rebuild", ptext_raw, meta, fragment=ptext_raw)
            continue
        line_ids = para.get("source_line_ids", "")
        para_flags = sorted(set(_inherit_para_flags(para) + pclean_flags))

        if ptype in {"heading", "micro_heading"} or is_heading_text(ptext):
            ptype = "micro_heading" if ptype == "micro_heading" else "heading"
            lvl = heading_level(ptext, ptype)
            if lvl == "parent":
                current_parent[work_id] = ptext
                current_child[work_id] = ""
            else:
                current_child[work_id] = ptext
            counters[work_id] += 1
            sq = text_metrics(ptext)
            reasons = sorted(set(para_flags + sentence_extra_flags(ptext, ptype, sq)))
            row = {
                "sent_id": f"{work_id}_s{counters[work_id]:07d}",
                "paragraph_id": para_id,
                "source_paragraph_id": para.get("source_paragraph_id", para_id),
                "source_line_ids": line_ids,
                "work_id": work_id,
                "pdf_path": para.get("pdf_path"),
                "page_idx": para.get("page_idx"),
                "page_number": para.get("page_number"),
                "source_page": para.get("page_number"),
                "page_span": para.get("page_span", para.get("page_number")),
                "sentence_index_in_paragraph": 0,
                "text": ptext.rstrip(":："),
                "source": para.get("source"),
                "paragraph_type": ptype,
                "is_heading": True,
                "heading_level": lvl,
                "current_parent_heading": current_parent[work_id],
                "current_heading": current_child[work_id] or current_parent[work_id],
                "bbox": para.get("bbox"),
                "quality_score": sq.get("quality_score"),
                "vietnamese_ratio": sq.get("vietnamese_ratio"),
                "weird_char_ratio": sq.get("weird_char_ratio"),
                "dictionary_hit_ratio": sq.get("dictionary_hit_ratio"),
                "suspicious_reasons": ";".join(reasons),
            }
            auto_rows.append(row)
            if reasons:
                susp_rows.append(row)
            continue

        sents = split_sentences_vietnamese(ptext, ptype)
        if not sents:
            sents = [ptext]
            para_flags.append("paragraph_not_split_emit_whole")
        for si, sent in enumerate(sents):
            sent, sflags = clean_mixed_script_text(sent, meta, context="final_sentence")
            sent = _repair_tinh_uppercase_context(sent, is_heading_context=False)
            if not sent:
                continue
            sq = text_metrics(sent)
            reasons = sorted(set(para_flags + sflags + suspicious_reasons_for_text(sent, sq) + sentence_extra_flags(sent, ptype, sq)))
            counters[work_id] += 1
            row = {
                "sent_id": f"{work_id}_s{counters[work_id]:07d}",
                "paragraph_id": para_id,
                "source_paragraph_id": para.get("source_paragraph_id", para_id),
                "source_line_ids": line_ids,
                "work_id": work_id,
                "pdf_path": para.get("pdf_path"),
                "page_idx": para.get("page_idx"),
                "page_number": para.get("page_number"),
                "source_page": para.get("page_number"),
                "page_span": para.get("page_span", para.get("page_number")),
                "sentence_index_in_paragraph": si,
                "text": sent,
                "source": para.get("source"),
                "paragraph_type": ptype,
                "is_heading": False,
                "heading_level": "",
                "current_parent_heading": current_parent[work_id],
                "current_heading": current_child[work_id] or current_parent[work_id],
                "bbox": para.get("bbox"),
                "quality_score": sq.get("quality_score"),
                "vietnamese_ratio": sq.get("vietnamese_ratio"),
                "weird_char_ratio": sq.get("weird_char_ratio"),
                "dictionary_hit_ratio": sq.get("dictionary_hit_ratio"),
                "suspicious_reasons": ";".join(reasons),
            }
            auto_rows.append(row)
            if reasons:
                susp_rows.append(row)

    all_sentences_auto = auto_rows
    all_sentences = [r for r in auto_rows if not r.get("is_heading")]
    suspicious_sentences = susp_rows
    print(f"DNTC v12 rebuild: final_sentences_auto={len(all_sentences_auto)}, final_sentences_only={len(all_sentences)}, suspicious={len(suspicious_sentences)}")

build_sentence_rows_from_paragraphs()


In [ ]:
# ============================================================
# 9D. DNTC v15 patch: final-text + terminal-punctuation sentence export
# ============================================================
# Main change from v14:
#   - do NOT split on line/page breaks or on semantic openers alone;
#   - merge OCR/layout fragments first;
#   - then cut sentences only at real terminal punctuation: . ! ? ; and CJK variants;
#   - keep headings as standalone rows;
#   - never emit short orphan CJK/noise tails as independent sentences.
#
# In this corpus, a CSV row should be a sentence-like reading unit. If a paragraph
# has no reliable terminal punctuation, it is safer to keep it as one longer row
# than to invent boundaries from noisy line breaks.

import re
from collections import defaultdict

if "norm_text" not in globals():
    def norm_text(text):
        return re.sub(r"\s+", " ", str(text or "")).strip()

SENTENCE_EXPORT_MODE = globals().get("SENTENCE_EXPORT_MODE", "final_text_terminal_punctuation")
TERMINAL_PUNCT = globals().get("TERMINAL_PUNCT", ".!?;。！？；")
# v15.1: preserve micro-heading rows as structural anchors. Do not merge them into body rows.
V15_PRESERVE_MICRO_HEADING_ROWS = bool(globals().get("V15_PRESERVE_MICRO_HEADING_ROWS", True))
# Hard split is only a safety valve. It still searches backward for terminal punctuation.
FINAL_TEXT_SENTENCE_HARD_LIMIT = int(globals().get("FINAL_TEXT_SENTENCE_HARD_LIMIT", 1800))
FINAL_TEXT_SENTENCE_MIN_CHARS = int(globals().get("FINAL_TEXT_SENTENCE_MIN_CHARS", 18))

FINAL_TEXT_ENTRY_BODY_RE = re.compile(
    r"^(?:và\s+)?(?:niên\s+hiệu|đầu\s+hiệu|đầu\s+niên\s+hiệu|năm|xây|đắp|dựng|cất|trước\s+tên|tên\s+cũ|ở\s+|là\s+|gọi\s+|đổi\s+|cách\s+|chu\s*vi|có\s+)\b",
    re.I,
)

FINAL_TEXT_DANGLING_END_RE = re.compile(
    r"(?:\b(?:năm|thứ|và|của|có|làm|gọi|đổi|đặt|xây|dựng|cất|phía|cửa|điện|cung|viện|lầu|các|đường|trong|ngoài|ở|đến|từ|đề|thờ)\s*)$|"
    r"(?:[,;:\-–—\(\[]\s*)$",
    re.I,
)

FINAL_TEXT_FRAGMENT_START_RE = re.compile(
    r"^(?:và\b|của\b|phía\b|năm\b|thứ\b|đến\b|ở\b|đề\b|thờ\b|\d+\s*\(|[a-zà-ỹđ]|[\u3400-\u9FFF])",
    re.I,
)

FINAL_TEXT_PARENT_HEADING_RE = re.compile(
    r"^(?:ĐẠI\s+NAM|QUYỀN\b|KINH[-\s]?SƯ|TỬ[-\s]?CHÍ|THÀNH[-\s]?TRÌ|HOÀNG[-\s]?THÀNH|"
    r"TỬ\s+CẤM[-\s]?THÀNH|ĐÀN\s+MIẾU|THÁI[-\s]?MIẾU|THẾ[-\s]?MIẾU|TRIỆU[-\s]?MIẾU|"
    r"HƯNG[-\s]?MIẾU|PHỤNG[-\s]?TIÊN[-\s]?ĐIỆN|VĂN[-\s]?MIẾU|ĐÀN\b|MIẾU\b|CUNG\b|"
    r"ĐIỆN\b|LẦU\b|VIỆN\b|CỬA\b|CẦU\b)",
    re.I,
)


def normalize_final_text_for_sentence_export(text: str) -> str:
    t = norm_text(text)
    # OCR often uses apostrophe as a broken ư in this corpus.
    t = re.sub(r"\bKINH\s*[- ]?SU\s*['’]?\b", "KINH-SƯ", t, flags=re.I)

    # Broken spaces after cross-line joining.
    t = re.sub(r"\bkhố(?=thông\b)", "khố ", t, flags=re.I)
    t = re.sub(r"\bhộ(?=thành\b)", "hộ ", t, flags=re.I)
    t = re.sub(r"\bở(?=bờ\b)", "ở ", t, flags=re.I)
    t = re.sub(r"\bthờ(?=các|thần|Thánh|Hiếu|Hoàng)\b", "thờ ", t, flags=re.I)

    # Dot used as a broken hyphen inside Sino-Vietnamese proper names before CJK.
    # Example: Phước. Hoằng 福泓 -> Phước-Hoằng 福泓.
    t = re.sub(
        r"\b([A-ZÀ-Ỹ][A-Za-zÀ-ỹĐđ]{2,})\.\s+([A-ZÀ-Ỹ][A-Za-zÀ-ỹĐđ]{2,})(?=\s+[\u3400-\u9FFF])",
        r"\1-\2",
        t,
    )

    # OCR CÔ/Cồ after "thiên" is usually the tail of "thiên cổ" when followed by CJK.
    t = re.sub(r"\bthiên\s+C[ÔồO]\s+([\u3400-\u9FFF]{2,})", r"thiên cổ \1", t, flags=re.I)

    # Normalize punctuation spacing.
    t = re.sub(r"\s+([,.;:!?])", r"\1", t)
    t = re.sub(r"([\(\[（])\s+", r"\1", t)
    t = re.sub(r"\s+([\)\]）])", r"\1", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def is_final_text_parent_heading(text: str, paragraph_type: str = "body") -> bool:
    t = normalize_final_text_for_sentence_export(text)
    if not t:
        return False
    if paragraph_type == "heading" and len(t) <= 180:
        return True
    if FINAL_TEXT_PARENT_HEADING_RE.match(t) and len(t) <= 180:
        return True
    if paragraph_type == "micro_heading":
        return False
    return False


def is_mergeable_micro_entry(text: str, paragraph_type: str = "body") -> bool:
    t = normalize_final_text_for_sentence_export(text)
    if not t or len(t) > 240:
        return False
    if is_final_text_parent_heading(t, paragraph_type):
        return False
    if paragraph_type == "micro_heading":
        return True
    if "split_entry_label_and_body" in globals():
        label, body, _reason = split_entry_label_and_body(t)
        return bool(label and not body)
    return False


def _safe_int_page_v15(row: dict):
    try:
        return int(row.get("page_number", 0) or 0)
    except Exception:
        return 0


def _is_page_contiguous_for_merge_v15(prev: dict, cur: dict) -> bool:
    """Allow same-page merge, or one-page continuation only.
    Never merge across verified PAGE_RANGE gaps such as 07_08 page 90 -> 95.
    """
    pp = _safe_int_page_v15(prev)
    cp = _safe_int_page_v15(cur)
    if pp == cp:
        return True
    if cp == pp + 1:
        # Both pages still have to be inside PAGE_RANGES if helper exists.
        try:
            return row_in_page_ranges(prev) and row_in_page_ranges(cur)
        except Exception:
            return True
    return False


def should_merge_final_text_chunks(prev: dict, cur: dict) -> bool:
    pt = normalize_final_text_for_sentence_export(prev.get("text", ""))
    ct = normalize_final_text_for_sentence_export(cur.get("text", ""))
    if not pt or not ct:
        return False
    if str(prev.get("work_id")) != str(cur.get("work_id")):
        return False
    if not _is_page_contiguous_for_merge_v15(prev, cur):
        return False

    prev_ptype = str(prev.get("paragraph_type", "body") or "body")
    cur_ptype = str(cur.get("paragraph_type", "body") or "body")

    # Structural anchors must stay separate. This fixes the old issue where a
    # micro-heading paragraph could vanish from final_sentences_auto.csv.
    if cur_ptype in {"heading", "micro_heading"}:
        return False
    if prev_ptype in {"heading", "micro_heading"}:
        return False

    if is_final_text_parent_heading(ct, cur_ptype):
        return False

    # Body-to-body continuation only. Examples:
    # "Năm Minh Mạng thứ" + "6 (1835)..."
    if FINAL_TEXT_DANGLING_END_RE.search(pt) and FINAL_TEXT_FRAGMENT_START_RE.match(ct):
        return True

    # A line starting with "và <entry label>" can complete a preceding body fragment,
    # but not a heading/micro-heading.
    if re.match(r"^và\s+[A-ZÀ-ỸĐ][A-Za-zÀ-ỹĐđ\-]+", ct):
        return True

    # Short CJK tail fragments should attach to body, not become standalone rows,
    # but only if previous text has not ended and neither side is a structural anchor.
    if len(ct) <= 90 and re.search(r"[\u3400-\u9FFF]", ct) and not re.search(r"[.!?;。！？；]$", pt):
        return True
    return False


def _protect_sentence_internal_punctuation(text: str):
    protected = []

    def protect(m):
        protected.append(m.group(0))
        return f"§PROT{len(protected)-1}§"

    work = text
    # Common abbreviations / editorial markers that contain periods but are not sentence ends.
    work = re.sub(r"\b(?:v\.\s*v\.|x\.\s*th\.|tr\.\s*C\.\s*N\.|T\.\s*N\.|t\.\s*d\.|v\.\s*d\.)", protect, work, flags=re.I)
    # Decimal / thousands separators.
    work = re.sub(r"(?<=\d)[.,](?=\d)", protect, work)
    # Enumerators like "1." should not produce an empty/fragment sentence.
    work = re.sub(r"(?<!\d)(\b\d{1,2})\.\s+(?=[A-ZÀ-ỸĐ])", lambda m: m.group(1) + "§ENUMDOT§ ", work)
    return work, protected


def _restore_sentence_internal_punctuation(text: str, protected: list) -> str:
    out = text.replace("§ENUMDOT§", ".")
    for i, val in enumerate(protected):
        out = out.replace(f"§PROT{i}§", val)
    return out


def split_final_text_by_terminal_punctuation(text: str) -> list:
    """Split only at terminal punctuation, not at OCR line/page breaks.

    Allowed sentence ends are: . ! ? ; and CJK equivalents. A period is accepted
    only when it behaves like sentence punctuation, not when it is inside numbers,
    abbreviations, or broken proper-name OCR.
    """
    s = normalize_final_text_for_sentence_export(text)
    if not s:
        return []

    work, protected = _protect_sentence_internal_punctuation(s)
    chunks = []
    start = 0
    i = 0
    n = len(work)
    while i < n:
        ch = work[i]
        if ch in TERMINAL_PUNCT:
            # Include following closing quotes/brackets in the same sentence.
            j = i + 1
            while j < n and work[j] in "\"'”’)]}）】」』":
                j += 1
            # Split only at whitespace/end after terminal punctuation.
            k = j
            while k < n and work[k].isspace():
                k += 1
            at_end = k >= n
            current = work[start:j].strip()
            next_char = work[k:k+1]
            # Do not split after a dot followed by lowercase text: often broken hyphen/name OCR.
            dot_before_lower = ch == "." and next_char and re.match(r"[a-zà-ỹđ]", next_char)
            if len(current) >= FINAL_TEXT_SENTENCE_MIN_CHARS and (at_end or not dot_before_lower):
                chunks.append(current)
                start = k
                i = k
                continue
        i += 1

    tail = work[start:].strip()
    if tail:
        chunks.append(tail)

    # Safety: if a chunk is extremely long, split it at the last terminal punctuation before the hard limit.
    final_chunks = []
    for chunk in chunks or [work]:
        while len(chunk) > FINAL_TEXT_SENTENCE_HARD_LIMIT:
            cut = max(chunk.rfind(p, 0, FINAL_TEXT_SENTENCE_HARD_LIMIT) for p in TERMINAL_PUNCT)
            if cut < FINAL_TEXT_SENTENCE_MIN_CHARS:
                break
            final_chunks.append(chunk[:cut+1].strip())
            chunk = chunk[cut+1:].strip()
        if chunk:
            final_chunks.append(chunk)

    out = []
    for p in final_chunks:
        p = _restore_sentence_internal_punctuation(p, protected)
        p = normalize_final_text_for_sentence_export(p)
        if p:
            out.append(p)
    return out or [s]


def _sentence_row_from_chunk(base_para: dict, text: str, counter: int, si: int, is_heading: bool, current_parent: str, current_child: str, extra_flags=None) -> dict:
    work_id = str(base_para.get("work_id", ""))
    ptype = str(base_para.get("paragraph_type", "heading" if is_heading else "body") or "body")
    sq = text_metrics(text) if "text_metrics" in globals() else {"quality_score": None, "vietnamese_ratio": None, "weird_char_ratio": None, "dictionary_hit_ratio": None}
    flags = []
    if extra_flags:
        flags.extend(extra_flags)
    if "sentence_extra_flags" in globals():
        flags.extend(sentence_extra_flags(text, ptype, sq))
    flags = sorted(set([f for f in flags if f]))
    return {
        "sent_id": f"{work_id}_s{counter:07d}",
        "paragraph_id": base_para.get("paragraph_id", ""),
        "source_paragraph_id": base_para.get("source_paragraph_id", base_para.get("paragraph_id", "")),
        "source_line_ids": base_para.get("source_line_ids", base_para.get("line_ids", "")),
        "work_id": work_id,
        "pdf_path": base_para.get("pdf_path"),
        "page_idx": base_para.get("page_idx"),
        "page_number": base_para.get("page_number"),
        "source_page": base_para.get("page_number"),
        "page_span": base_para.get("page_span", base_para.get("page_number")),
        "sentence_index_in_paragraph": si,
        "text": text.rstrip(":：") if is_heading else text,
        "char_count": len(text.rstrip(":：") if is_heading else text),
        "source": base_para.get("source"),
        "paragraph_type": ptype,
        "is_heading": bool(is_heading),
        "heading_level": heading_level(text, ptype) if (is_heading and "heading_level" in globals()) else "",
        "current_parent_heading": current_parent,
        "current_heading": current_child or current_parent,
        "bbox": base_para.get("bbox"),
        "quality_score": sq.get("quality_score"),
        "vietnamese_ratio": sq.get("vietnamese_ratio"),
        "weird_char_ratio": sq.get("weird_char_ratio"),
        "dictionary_hit_ratio": sq.get("dictionary_hit_ratio"),
        "suspicious_reasons": ";".join(flags),
    }


def _inherit_para_flags_safe(para: dict) -> list:
    if "_inherit_para_flags" in globals():
        return _inherit_para_flags(para)
    return [x for x in str(para.get("suspicious_reasons", "")).split(";") if x]


def rebuild_sentences_from_final_text_terminal_punctuation():
    global all_sentences, all_sentences_auto, suspicious_sentences
    if SENTENCE_EXPORT_MODE != "final_text_terminal_punctuation":
        return
    para_rows = sorted(
        [dict(r) for r in all_paragraphs],
        key=lambda r: (
            str(r.get("work_id", "")),
            int(r.get("page_number", 0) or 0),
            int(r.get("paragraph_index", r.get("paragraph_index_in_page", 0)) or 0),
            str(r.get("paragraph_id", "")),
        ),
    )

    # Pass 1: merge adjacent OCR/layout fragments using final-text structure.
    merged = []
    for para in para_rows:
        p = dict(para)
        p["text"] = normalize_final_text_for_sentence_export(p.get("text", ""))
        if not p["text"]:
            continue
        if merged and should_merge_final_text_chunks(merged[-1], p):
            prev = merged[-1]
            prev["text"] = normalize_final_text_for_sentence_export(prev.get("text", "") + " " + p.get("text", ""))
            prev["paragraph_type"] = "body"
            prev["source_paragraph_id"] = ";".join(filter(None, [str(prev.get("source_paragraph_id", prev.get("paragraph_id", ""))), str(p.get("paragraph_id", ""))]))
            prev["source_line_ids"] = ";".join(filter(None, [str(prev.get("source_line_ids", prev.get("line_ids", ""))), str(p.get("source_line_ids", p.get("line_ids", "")))]))
            prev["page_span"] = f"{prev.get('page_span', prev.get('page_number'))}-{p.get('page_number')}" if str(prev.get("page_number")) != str(p.get("page_number")) else prev.get("page_span", prev.get("page_number"))
            old = [x for x in str(prev.get("suspicious_reasons", "")).split(";") if x]
            prev["suspicious_reasons"] = ";".join(sorted(set(old + ["final_text_fragment_merged_before_sentence_split"])))
        else:
            merged.append(p)

    # Pass 2: headings standalone; body split only by terminal punctuation.
    current_parent = defaultdict(str)
    current_child = defaultdict(str)
    counters = defaultdict(int)
    auto_rows, susp_rows = [], []
    for para in merged:
        work_id = str(para.get("work_id", ""))
        ptype = str(para.get("paragraph_type", "body") or "body")
        text = normalize_final_text_for_sentence_export(para.get("text", ""))
        if not text:
            continue
        inherited = _inherit_para_flags_safe(para)
        if is_final_text_parent_heading(text, ptype):
            current_parent[work_id] = text
            current_child[work_id] = ""
            counters[work_id] += 1
            row = _sentence_row_from_chunk(para, text, counters[work_id], 0, True, current_parent[work_id], current_child[work_id], inherited)
            auto_rows.append(row)
            if row.get("suspicious_reasons"):
                susp_rows.append(row)
            continue

        if ptype == "micro_heading" or is_mergeable_micro_entry(text, ptype):
            # v15.1: every micro-heading is a structural row. The following body
            # receives this value through current_heading instead of losing the
            # micro-heading by merging it into body text.
            current_child[work_id] = text.rstrip(":：")
            counters[work_id] += 1
            row = _sentence_row_from_chunk(para, current_child[work_id], counters[work_id], 0, True, current_parent[work_id], current_child[work_id], inherited + (["micro_heading_kept"] if ptype == "micro_heading" else ["entry_label_kept"]))
            auto_rows.append(row)
            if row.get("suspicious_reasons"):
                susp_rows.append(row)
            continue

        chunks = split_final_text_by_terminal_punctuation(text)
        for si, chunk in enumerate(chunks):
            counters[work_id] += 1
            row = _sentence_row_from_chunk(para, chunk, counters[work_id], si, False, current_parent[work_id], current_child[work_id], inherited)
            auto_rows.append(row)
            if row.get("suspicious_reasons"):
                susp_rows.append(row)

    all_sentences_auto = auto_rows
    all_sentences = [r for r in auto_rows if not r.get("is_heading")]
    suspicious_sentences = susp_rows
    print(
        f"DNTC v15 terminal-punctuation export: "
        f"final_sentences_auto={len(all_sentences_auto)}, "
        f"final_sentences_only={len(all_sentences)}, suspicious={len(suspicious_sentences)}"
    )

rebuild_sentences_from_final_text_terminal_punctuation()


In [ ]:

# ============================================================
# 9E. DNTC v16 R016 patch: spelling + false-period merge guard
# ============================================================
# Mechanism-based patch derived from spelling_merge_review_all.csv and
# rule_R016_spelling_sentence_merge.yaml. It does not hard-code individual
# sentence fixes; it implements general detectors for:
#   - false period splits inside names/CJK labels;
#   - context-bound OCR spelling corrections;
#   - false uppercase TỈNH in body text;
#   - short OCR noise fragments;
#   - sentence export after paragraph/page reflow.

import re
from pathlib import Path
from collections import defaultdict, Counter

PIPELINE_VERSION = "dntc_auto_v16_r016_spelling_merge_clean"
R016_RULE_ID = "R016"
R016_QUARANTINE_RECORDS = globals().get("R016_QUARANTINE_RECORDS", [])
R016_AUDIT_RECORDS = globals().get("R016_AUDIT_RECORDS", [])


def _find_optional_project_file_v16(filename: str):
    """Find optional review/rule files if user placed them next to notebook/input."""
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path("/mnt/data"), Path(".")]
    for root in roots:
        try:
            if not root.exists():
                continue
            direct = root / filename
            if direct.exists():
                return direct
            # Restrict recursive search to avoid slow scans.
            for p in list(root.glob(f"**/{filename}"))[:5]:
                if p.exists():
                    return p
        except Exception:
            continue
    return None

R016_REVIEW_CSV_PATH = _find_optional_project_file_v16("spelling_merge_review_all.csv")
R016_RULE_YAML_PATH = _find_optional_project_file_v16("rule_R016_spelling_sentence_merge.yaml")

try:
    if R016_REVIEW_CSV_PATH and R016_REVIEW_CSV_PATH.exists():
        _r016_review_df = pd.read_csv(R016_REVIEW_CSV_PATH)
        r016_review_summary_df = (_r016_review_df.groupby(["issue_type", "severity"]).size()
                                  .reset_index(name="count")
                                  .sort_values(["count"], ascending=False))
        r016_review_summary_df.to_csv(AUDIT_DIR / "r016_input_review_issue_summary.csv", index=False, encoding="utf-8-sig")
        print("Loaded R016 review CSV:", R016_REVIEW_CSV_PATH, "rows=", len(_r016_review_df))
    else:
        _r016_review_df = pd.DataFrame()
except Exception as e:
    print("Could not load R016 review CSV:", e)
    _r016_review_df = pd.DataFrame()

try:
    if R016_RULE_YAML_PATH and R016_RULE_YAML_PATH.exists():
        _r016_rule_text = R016_RULE_YAML_PATH.read_text(encoding="utf-8", errors="ignore")
        (AUDIT_DIR / "r016_loaded_rule_R016_spelling_sentence_merge.yaml.txt").write_text(_r016_rule_text, encoding="utf-8")
        print("Loaded R016 YAML rule:", R016_RULE_YAML_PATH)
    else:
        _r016_rule_text = ""
except Exception as e:
    print("Could not load R016 YAML rule:", e)
    _r016_rule_text = ""

if "norm_text" not in globals():
    def norm_text(text):
        return re.sub(r"\s+", " ", str(text or "")).strip()

R016_TERMINAL = ".!?;。！？；"
R016_CJK_RE = re.compile(r"[\u3400-\u9FFF]")
R016_LEX_TOKEN_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]+|[\u3400-\u9FFF]+|\d+")
R016_VIET_TOKEN_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]+")
R016_ADMIN_HEADING_RE = re.compile(
    r"^(?:ĐẠI\s+NAM|QUYỀN|TỈNH|PHỦ|HUYỆN|THÀNH|HOÀNG|TỬ\s*CẤM|ĐÀN|MIẾU|CHÙA|TRẠM|CẦU|SÔNG|NÚI|ĐIỆN|CUNG|LẦU|VIỆN|CỬA|HỌC\s+HIỆU|HỘ\s+KHẨU|ĐIỀN\s+PHÚ|PHONG\s+TỤC)\b",
    re.I,
)
R016_FALSE_TINH_COMMON_RE = re.compile(
    r"\bTỈNH\s+(?=(?:thần|xảo|tú|thông|kỳ|khoan|huyết|bồ|bik|quân|môn|hoa|tiết|tường|tế|mật|điền)\b)",
    re.I,
)
R016_NOISE_RE = re.compile(
    r"(?i)(?:\bbik\b|\bx\s+wt\b|\bkt\s+ah\b|\bor\s*\+|\*\s*\*|BILIARY\s+LICYRe|\bWSR\b|\bCC\b|\bFB\b|T\s*/\s*\^T|[\^`¬¡¿]{1,}|Ö-®\s*Y)"
)
R016_FALSE_PERIOD_PREFIX_RE = re.compile(
    r"\b(Gia|Minh|Thiệu|Tự|Thành|Đồng|Phước|Tân|Thế|Đông|Ngọc|Cảnh|Vĩnh|An|Hòa|Hoằng|Bảo|Long|Khánh|Triệu|Quảng)\.$",
    re.I,
)
R016_CONTEXT_DANGLING_END_RE = re.compile(
    r"\b(?:thứ|năm|niên\s+hiệu|ngày|tháng|cửa|cầu|kiều|điện|miếu|phủ|huyện|tỉnh|tổng|xã|trạm|môn|đài|viện|lầu|gọi|tên|đổi|đặt|xây|dựng|cất|đề|để)\.?$",
    re.I,
)
R016_COMPASS_AFTER_MEASURE_RE = re.compile(r"^(?:Đông|Tây|Nam|Bắc|Phía|Từ)\b", re.I)
R016_MEASURE_END_RE = re.compile(r"(?:dặm|trượng|thước|tấc|phân|mẫu|sào|thôn|người)\.$", re.I)
R016_ENUM_START_RE = re.compile(r"^(?:\(?\d{1,3}\)?\s*[\.\-–]|\(\d+\))")
R016_SINGLE_TOKEN_ALLOWED_RE = re.compile(r"^(?:\d+\s*(?:dặm|trượng|thước|tấc|phân|mẫu|sào|người)\.?|[A-ZÀ-ỸĐ][A-ZÀ-ỸĐ\-\s]{2,})$", re.I)

R016_PURPOSE_VERBS = "làm|cho|tiện|đi|biết|cai|tránh|thấy|dùng|cắm|xây|dựng|thống|trấn|tỏ|nêu|khảo|sửa|gọi|đặt|thờ|qua|ghi|chép"
R016_CHANGE_VERBS = "tên|làm|sang|lại|ra|thành|cho|gọi|dựng|xây|lệ|thuộc|đặt|hiện"
R016_MONTH_WORDS = "giêng|chạp|một|hai|ba|tư|năm|sáu|bảy|tám|chín|mười|[0-9]{1,2}"
R016_ADMIN_UNITS = "phía|huyện|phủ|tỉnh|tổng|xã|thôn|châu|sách|mường|trạm|gian|cửa|dặm|trượng|thước|tấc|phân|năm|ngày|mẫu|sào|người|quan|tiền|đồng|hộc|thăng|hạp|thược"


def r016_lexical_tokens(text: str):
    return R016_LEX_TOKEN_RE.findall(norm_text(text))


def r016_is_structural_heading(text: str, paragraph_type: str = "") -> bool:
    t = norm_text(text)
    if not t:
        return False
    ptype = str(paragraph_type or "")
    if ptype in {"heading", "micro_heading"}:
        return True
    if len(t) <= 180 and R016_ADMIN_HEADING_RE.match(t):
        return True
    # Mostly uppercase short lines are structural anchors, but do not classify long body text.
    letters = re.findall(r"[A-Za-zÀ-ỹĐđ]", t)
    if letters and len(t) <= 120:
        upper_ratio = sum(1 for ch in letters if ch.upper() == ch and ch.lower() != ch) / max(1, len(letters))
        if upper_ratio >= 0.72 and len(r016_lexical_tokens(t)) <= 12:
            return True
    # Generic entry label with CJK + optional colon.
    if "is_entry_label_text" in globals():
        try:
            if is_entry_label_text(t):
                return True
        except Exception:
            pass
    return False


def r016_is_measure_then_compass(prev: str, nxt: str) -> bool:
    return bool(R016_MEASURE_END_RE.search(norm_text(prev)) and R016_COMPASS_AFTER_MEASURE_RE.match(norm_text(nxt)))


def r016_is_real_short_content(text: str, paragraph_type: str = "", prev_text: str = "") -> bool:
    t = norm_text(text)
    if not t:
        return False
    if r016_is_structural_heading(t, paragraph_type):
        return True
    if re.search(r"\d+\s*(?:dặm|trượng|thước|tấc|phân|mẫu|sào|người|quan|tiền|đồng)\.?$", t, re.I):
        return True
    if re.search(r"(?:đến\s+trạm|thuộc|gồm|lãnh|ở|là|đến|tới)\s*$", norm_text(prev_text), re.I):
        return True
    # A short Hán-Việt place name with hyphen and a number/unit is likely meaningful.
    if re.search(r"[A-ZÀ-ỸĐ][A-Za-zÀ-ỹĐđ]+-[A-ZÀ-ỸĐA-Za-zÀ-ỹĐđ]+", t) and len(t) <= 50:
        return True
    return False


def r016_is_noise_fragment(text: str, paragraph_type: str = "", prev_text: str = "", next_text: str = "") -> bool:
    t = norm_text(text)
    if not t:
        return True
    if r016_is_real_short_content(t, paragraph_type, prev_text):
        return False
    toks = r016_lexical_tokens(t)
    letters = R016_VIET_TOKEN_RE.findall(t)
    weird = bool(R016_NOISE_RE.search(t))
    very_short = len(t) <= 4 or len(toks) <= 1
    near_cjk_or_heading = bool(R016_CJK_RE.search(prev_text + " " + next_text)) or r016_is_structural_heading(prev_text) or r016_is_structural_heading(next_text)
    if weird and (len(t) <= 80 or near_cjk_or_heading):
        return True
    if very_short and near_cjk_or_heading and not letters:
        return True
    if very_short and re.fullmatch(r"[A-Za-zÀ-ỹĐđ]{1,3}", t) and not R016_SINGLE_TOKEN_ALLOWED_RE.match(t):
        return True
    if len(t) <= 10 and re.search(r"[*^`¬¡¿+/#]", t) and not re.search(r"\d+\s*(?:dặm|trượng|thước|tấc|phân)", t, re.I):
        return True
    return False


def r016_log_audit(issue_type: str, severity: str, text: str, suggested="", note="", row=None):
    rec = {
        "issue_type": issue_type,
        "severity": severity,
        "original_text": norm_text(text),
        "suggested_correction": suggested,
        "rule_id": R016_RULE_ID,
        "note": note,
    }
    if isinstance(row, dict):
        for k in ["work_id", "page_number", "paragraph_id", "source_paragraph_id", "source_line_ids", "sent_id"]:
            if k in row:
                rec[k] = row.get(k)
    R016_AUDIT_RECORDS.append(rec)
    return rec


def _replace_geo_bien_r016(text: str) -> str:
    # Only fix biền/bề in geography/distance/sea contexts.
    geo_words = r"(?:biển|bể|hải|cửa|vịnh|vũng|sông|bờ|phía|đông|tây|nam|bắc|đến|ra|giáp|dặm)"
    def repl(m):
        start, end = m.span()
        window = text[max(0, start-45): min(len(text), end+45)]
        if re.search(geo_words, window, re.I):
            return "biển"
        return m.group(0)
    return re.sub(r"\b(?:biền|bề)\b", repl, text, flags=re.I)


def r016_apply_context_spelling(text: str, paragraph_type: str = "body"):
    t0 = norm_text(text)
    t = t0
    changed = []
    ptype = str(paragraph_type or "body")
    is_head = r016_is_structural_heading(t, ptype)

    # Heading/title OCR fallback.
    t2 = re.sub(r"\bKINH\s*[- ]?SU\s*['’]?\b", "KINH-SƯ", t, flags=re.I)
    t2 = re.sub(r"\bNH[ẤẬA]T\s+TH[O0ÔỐ]NG\s+CH[IÍỈ]\b", "NHẤT THỐNG CHÍ", t2, flags=re.I)
    if is_head:
        t2 = re.sub(r"\bDUNG\s+PAT\s+VA\s+DIEN\s+CACH\b", "DỰNG ĐẶT VÀ DIÊN CÁCH", t2, flags=re.I)
        t2 = re.sub(r"\bPHAN\s+DA\b", "PHÂN DÃ", t2, flags=re.I)
    if t2 != t:
        changed.append("r016_heading_title_fallback")
        t = t2

    # False uppercase TỈNH: body should use lowercase tỉnh/tinh unless heading.
    before = t
    t = R016_FALSE_TINH_COMMON_RE.sub("tinh ", t)
    if not is_head:
        # TỈNH followed by lowercase common text -> tinh; followed by proper place -> tỉnh.
        t = re.sub(r"\bTỈNH\s+(?=[a-zà-ỹđ])", "tinh ", t)
        t = re.sub(r"\bTỈNH\s+(?=[A-ZÀ-ỸĐ])", "tỉnh ", t)
    if t != before:
        changed.append("r016_false_tinh_uppercase_repair")

    # Context-bound spelling corrections.
    replacements = [
        (rf"\bđề\s+(?=({R016_PURPOSE_VERBS})\b)", "để ", "r016_de_to_de_purpose"),
        (rf"\bđồi\s+(?=({R016_CHANGE_VERBS})\b)", "đổi ", "r016_doi_to_doi_change"),
        (rf"\bthang\s+(?=({R016_MONTH_WORDS})\b)", "tháng ", "r016_thang_to_thang_date"),
        (r"\b(thứ|năm|ngày|tháng)(\d+)\b", r"\1 \2", "r016_missing_space_before_number"),
        (rf"\b(\d+)(?=({R016_ADMIN_UNITS})\b)", r"\1 ", "r016_missing_space_after_number"),
        (r"\btồng\b", "tổng", "r016_tong_ocr"),
        (r"\bthu€\b", "thuế", "r016_thue_mojibake"),
        (r"\bchi€m\b", "chiếm", "r016_chiem_mojibake"),
        (r"\bbi€n\b", "biển", "r016_bien_mojibake"),
        (r"\bki€m\b", "kiêm", "r016_kiem_mojibake"),
        (r"\bmi€u\b", "miếu", "r016_mieu_mojibake"),
    ]
    for pat, rep, rule in replacements:
        before = t
        t = re.sub(pat, rep, t, flags=re.I)
        if t != before:
            changed.append(rule)

    before = t
    t = _replace_geo_bien_r016(t)
    if t != before:
        changed.append("r016_bien_bien_geo_context")

    # Normalize common reign names where OCR broke accents/case. Hyphen is safe in this corpus style.
    before = t
    t = re.sub(r"\bMinh\s+mạng\b", "Minh-Mạng", t, flags=re.I)
    t = re.sub(r"\bGia\s+long\b", "Gia-Long", t, flags=re.I)
    t = re.sub(r"\bThiệu\s+trị\b", "Thiệu-Trị", t, flags=re.I)
    t = re.sub(r"\bTự\s+đức\b", "Tự-Đức", t, flags=re.I)
    t = re.sub(r"\bThành\s+thái\b", "Thành-Thái", t, flags=re.I)
    if t != before:
        changed.append("r016_reign_name_normalization")

    # Final spacing/punctuation cleanup.
    t = re.sub(r"\s+([,.;:!?])", r"\1", t)
    t = re.sub(r"([,.;:!?])(?=[A-Za-zÀ-ỹĐđ])", r"\1 ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t, sorted(set(changed))


def r016_should_merge_false_period(prev: str, nxt: str, prev_row=None, nxt_row=None) -> bool:
    p = norm_text(prev)
    n = norm_text(nxt)
    if not p or not n:
        return False
    n_ptype = str((nxt_row or {}).get("paragraph_type", "")) if isinstance(nxt_row, dict) else ""
    if r016_is_structural_heading(n, n_ptype):
        return False
    if r016_is_measure_then_compass(p, n):
        return False
    if R016_ENUM_START_RE.match(n):
        return False
    toks = r016_lexical_tokens(n)
    if not toks:
        return False
    if R016_FALSE_PERIOD_PREFIX_RE.search(p):
        return True
    if R016_CONTEXT_DANGLING_END_RE.search(p):
        return True
    if len(toks) <= 1 and p.endswith("."):
        return True
    if len(toks) <= 2 and R016_CJK_RE.search(n) and p.endswith("."):
        return True
    if p.endswith(".") and re.match(r"^[a-zà-ỹđ]", n):
        return True
    return False


def r016_merge_false_period(prev: str, nxt: str) -> str:
    p = norm_text(prev)
    n = norm_text(nxt)
    # Proper compound: Phước. Hoằng -> Phước-Hoằng. Also works for Gia. Long, Minh. Mạng.
    if R016_FALSE_PERIOD_PREFIX_RE.search(p) and re.match(r"^[A-ZÀ-ỸĐ][A-Za-zÀ-ỹĐđ]+", n):
        return re.sub(r"\.\s*$", "-", p) + n
    # CJK label tail: Trác việt thiên. cổ 卓越千古 -> Trác việt thiên cổ 卓越千古.
    if R016_CJK_RE.search(n):
        return re.sub(r"\.\s*$", " ", p) + n
    return re.sub(r"\.\s*$", " ", p) + n


def r016_sentence_flags(text: str, paragraph_type: str = "body") -> list:
    t = norm_text(text)
    flags = []
    if R016_NOISE_RE.search(t):
        flags.append("r016_ocr_noise_pattern")
    if R016_FALSE_TINH_COMMON_RE.search(t):
        flags.append("r016_false_tinh_uppercase")
    if re.search(r"\bKINH\s*[- ]?SU\b|\bKINH\s+SU\b", t, re.I):
        flags.append("r016_heading_kinh_su_remaining")
    if re.search(r"\b(?:Phước|Tân|Gia|Minh|Đông|Ngọc|Thế)\.\s+[A-ZÀ-ỸĐa-zà-ỹđ]", t):
        flags.append("r016_false_period_compound_remaining")
    toks = r016_lexical_tokens(t)
    if len(toks) <= 1 and not r016_is_structural_heading(t, paragraph_type) and not R016_SINGLE_TOKEN_ALLOWED_RE.match(t):
        flags.append("r016_one_token_sentence")
    if len(t) > 800:
        flags.append("critical_overlong_sentence_gt800")
    return flags


def r016_split_then_merge_sentences(text: str, paragraph_type: str = "body", base_row=None) -> list:
    # Start from v15 terminal splitter, then apply context spelling and false-period merges.
    if "split_final_text_by_terminal_punctuation" in globals():
        chunks = split_final_text_by_terminal_punctuation(text)
    else:
        chunks = re.split(r"(?<=[.!?;。！？；])\s+", norm_text(text))
    normalized = []
    for ch in chunks:
        ch2, _rules = r016_apply_context_spelling(ch, paragraph_type)
        if ch2:
            normalized.append(ch2)
    # Merge false-period tails.
    merged = []
    for ch in normalized:
        if merged and r016_should_merge_false_period(merged[-1], ch, base_row, base_row):
            old = merged[-1]
            new = r016_merge_false_period(old, ch)
            r016_log_audit("false_period_merge_applied", "medium", old + " || " + ch, new, "merged before final sentence IDs", base_row if isinstance(base_row, dict) else None)
            merged[-1] = new
        else:
            merged.append(ch)
    # Remove/quarantine short OCR fragments; keep meaningful short content.
    clean = []
    for i, ch in enumerate(merged):
        prev = clean[-1] if clean else ""
        nxt = merged[i+1] if i+1 < len(merged) else ""
        if r016_is_noise_fragment(ch, paragraph_type, prev, nxt):
            rec = dict(base_row or {})
            rec.update({
                "reason": "r016_ocr_noise_fragment_quarantined",
                "current_text": ch,
                "prev_text": prev,
                "next_text": nxt,
                "rule_id": R016_RULE_ID,
            })
            R016_QUARANTINE_RECORDS.append(rec)
            r016_log_audit("ocr_noise_fragment", "high", ch, "quarantine / remove from final output with audit", "short/noisy fragment after terminal split", base_row if isinstance(base_row, dict) else None)
            continue
        clean.append(ch)
    return clean


# Override v15 normalizer lightly so downstream export also sees R016 spelling fixes.
_base_normalize_final_text_for_sentence_export_v16 = globals().get("normalize_final_text_for_sentence_export")

def normalize_final_text_for_sentence_export(text: str) -> str:
    if callable(_base_normalize_final_text_for_sentence_export_v16):
        base = _base_normalize_final_text_for_sentence_export_v16(text)
    else:
        base = norm_text(text)
    fixed, _rules = r016_apply_context_spelling(base, "body")
    return fixed


def rebuild_sentences_r016_spelling_merge():
    global all_sentences, all_sentences_auto, suspicious_sentences
    para_rows = sorted(
        [dict(r) for r in all_paragraphs],
        key=lambda r: (
            str(r.get("work_id", "")),
            int(r.get("page_number", 0) or 0),
            int(r.get("paragraph_index", r.get("paragraph_index_in_page", 0)) or 0),
            str(r.get("paragraph_id", "")),
        ),
    )

    # Pass 1: body-to-body merge from v15, but spell-clean before merge.
    merged = []
    for para in para_rows:
        p = dict(para)
        ptype = str(p.get("paragraph_type", "body") or "body")
        p["text"], corr_rules = r016_apply_context_spelling(p.get("text", ""), ptype)
        if corr_rules:
            old_reasons = [x for x in str(p.get("suspicious_reasons", "")).split(";") if x]
            p["suspicious_reasons"] = ";".join(sorted(set(old_reasons + corr_rules)))
        if not p["text"]:
            continue
        # Do not merge structural anchors; use v15 body merge only.
        if merged and "should_merge_final_text_chunks" in globals() and should_merge_final_text_chunks(merged[-1], p):
            prev = merged[-1]
            prev["text"] = norm_text(prev.get("text", "") + " " + p.get("text", ""))
            prev["paragraph_type"] = "body"
            prev["source_paragraph_id"] = ";".join(filter(None, [str(prev.get("source_paragraph_id", prev.get("paragraph_id", ""))), str(p.get("paragraph_id", ""))]))
            prev["source_line_ids"] = ";".join(filter(None, [str(prev.get("source_line_ids", prev.get("line_ids", ""))), str(p.get("source_line_ids", p.get("line_ids", "")))]))
            old = [x for x in str(prev.get("suspicious_reasons", "")).split(";") if x]
            prev["suspicious_reasons"] = ";".join(sorted(set(old + ["r016_body_fragment_merged_before_sentence_split"])))
        else:
            merged.append(p)

    current_parent = defaultdict(str)
    current_child = defaultdict(str)
    counters = defaultdict(int)
    auto_rows, susp_rows = [], []

    for para in merged:
        work_id = str(para.get("work_id", ""))
        ptype = str(para.get("paragraph_type", "body") or "body")
        text = norm_text(para.get("text", ""))
        if not text:
            continue
        inherited = _inherit_para_flags_safe(para) if "_inherit_para_flags_safe" in globals() else [x for x in str(para.get("suspicious_reasons", "")).split(";") if x]
        # Heading/micro-heading as structural rows.
        if r016_is_structural_heading(text, ptype):
            if ptype == "micro_heading" or ("heading_level" in globals() and heading_level(text, ptype) == "child"):
                current_child[work_id] = text.rstrip(":：")
            else:
                current_parent[work_id] = text.rstrip(":：")
                current_child[work_id] = ""
            counters[work_id] += 1
            extra = sorted(set(inherited + ["r016_structural_heading_kept"]))
            row = _sentence_row_from_chunk(para, text.rstrip(":："), counters[work_id], 0, True, current_parent[work_id], current_child[work_id], extra)
            row["text"], corr_rules = r016_apply_context_spelling(row["text"], ptype)
            flags = [x for x in str(row.get("suspicious_reasons", "")).split(";") if x] + corr_rules + r016_sentence_flags(row["text"], ptype)
            row["suspicious_reasons"] = ";".join(sorted(set(flags)))
            auto_rows.append(row)
            if row.get("suspicious_reasons"):
                susp_rows.append(row)
            continue

        chunks = r016_split_then_merge_sentences(text, ptype, para)
        for si, chunk in enumerate(chunks):
            chunk, corr_rules = r016_apply_context_spelling(chunk, ptype)
            if not chunk:
                continue
            if r016_is_noise_fragment(chunk, ptype):
                rec = dict(para)
                rec.update({"reason": "r016_sentence_noise_fragment_quarantined", "current_text": chunk, "rule_id": R016_RULE_ID})
                R016_QUARANTINE_RECORDS.append(rec)
                continue
            counters[work_id] += 1
            extra = sorted(set(inherited + corr_rules + r016_sentence_flags(chunk, ptype)))
            row = _sentence_row_from_chunk(para, chunk, counters[work_id], si, False, current_parent[work_id], current_child[work_id], extra)
            auto_rows.append(row)
            if row.get("suspicious_reasons"):
                susp_rows.append(row)

    all_sentences_auto = auto_rows
    all_sentences = [r for r in auto_rows if not r.get("is_heading")]
    suspicious_sentences = susp_rows

    # Write pre-export R016 audit/quarantine. Export cells will package these files.
    try:
        pd.DataFrame(R016_AUDIT_RECORDS).to_csv(AUDIT_DIR / "r016_spelling_sentence_merge_audit.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame(R016_QUARANTINE_RECORDS).to_csv(QUARANTINE_DIR / "r016_quarantined_noise_fragments.csv", index=False, encoding="utf-8-sig")
    except Exception as e:
        print("R016 audit write warning:", e)

    print(
        f"DNTC v16 R016 export: final_sentences_auto={len(all_sentences_auto)}, "
        f"final_sentences_only={len(all_sentences)}, suspicious={len(suspicious_sentences)}, "
        f"r016_quarantine={len(R016_QUARANTINE_RECORDS)}"
    )

rebuild_sentences_r016_spelling_merge()


In [ ]:

# ============================================================
# 10. Export final files + audit + quarantine + zip (DNTC v11)
# ============================================================
FINAL_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PKG_DIR.mkdir(parents=True, exist_ok=True)
QUARANTINE_DIR = OUTPUT_DIR / "quarantine"
QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)

lines_df = pd.DataFrame(all_final_lines)
paras_df = pd.DataFrame(all_paragraphs)
auto_source = globals().get("all_sentences_auto", all_sentences)
sents_auto_df = pd.DataFrame(auto_source)
sents_df = sents_auto_df[sents_auto_df.get("is_heading", False).astype(str).str.lower().ne("true")].copy() if not sents_auto_df.empty and "is_heading" in sents_auto_df.columns else pd.DataFrame(all_sentences)
page_df = pd.DataFrame(page_quality_report)
source_df = pd.DataFrame(source_selection_report)
excluded_pages_range_df = pd.DataFrame(globals().get("excluded_pages_by_page_range", []))

# PAGE_RANGES final safety gate: no final CSV/text row may remain outside verified ranges.
def _filter_final_df_to_ranges(df: pd.DataFrame, source_table: str):
    if df.empty or "page_number" not in df.columns:
        return df, pd.DataFrame()
    mask = df.apply(lambda r: row_in_page_ranges(r), axis=1)
    outside = df[~mask].copy()
    if not outside.empty:
        outside["source_table"] = source_table
        outside["drop_reason"] = "outside_PAGE_RANGES_export_safety_gate"
    return df[mask].copy(), outside

outside_final_frames = []
lines_df, out = _filter_final_df_to_ranges(lines_df, "final_lines"); outside_final_frames.append(out)
paras_df, out = _filter_final_df_to_ranges(paras_df, "final_paragraphs"); outside_final_frames.append(out)
sents_auto_df, out = _filter_final_df_to_ranges(sents_auto_df, "final_sentences_auto"); outside_final_frames.append(out)
sents_df, out = _filter_final_df_to_ranges(sents_df, "final_sentences_only"); outside_final_frames.append(out)
outside_final_pages_df = pd.concat([x for x in outside_final_frames if not x.empty], ignore_index=True) if any(not x.empty for x in outside_final_frames) else pd.DataFrame()
if not outside_final_pages_df.empty:
    # Keep a trace in excluded_pages_by_page_range too.
    excluded_pages_range_df = pd.concat([excluded_pages_range_df, outside_final_pages_df], ignore_index=True) if not excluded_pages_range_df.empty else outside_final_pages_df.copy()
line_audit_df = pd.DataFrame(line_filter_audit)
dropped_pages_df = pd.DataFrame(dropped_pages)
dropped_lines_df = pd.DataFrame(dropped_lines)
correction_df = pd.DataFrame(correction_log)
susp_lines_df = pd.DataFrame(suspicious_lines)
susp_sents_df = pd.DataFrame(suspicious_sentences)

# Repair final text after all conversions.
for df in [lines_df, paras_df, sents_df, sents_auto_df, susp_sents_df]:
    if not df.empty and "text" in df.columns:
        df["text"] = df["text"].astype(str).map(_repair_bad_tinh_phrases).map(norm_text)
        if "suspicious_reasons" in df.columns:
            df["suspicious_reasons"] = df["suspicious_reasons"].fillna("").astype(str)

# Remove misleading bad heading_tinh correction-log entries after v11 fix; keep the v11 fix events.
if not correction_df.empty and {"rule", "after"}.issubset(correction_df.columns):
    bad_corr_mask = correction_df["rule"].astype(str).str.contains("heading_tinh", case=False, na=False) & correction_df["after"].astype(str).str.contains(r"TỈNH\s+(?:thần|xảo|tú|thông|kỳ|khoan|quân|môn)", regex=True, na=False)
    bad_corr_quarantine = correction_df[bad_corr_mask].copy()
    if not bad_corr_quarantine.empty:
        bad_corr_quarantine.to_csv(QUARANTINE_DIR / "bad_heading_tinh_correction_events_quarantined.csv", index=False, encoding="utf-8-sig")
    correction_df = correction_df[~bad_corr_mask].copy()

# Stable sort.
for df, cols in [
    (lines_df, ["work_id", "page_number", "line_global_id"]),
    (paras_df, ["work_id", "page_number", "paragraph_id"]),
    (sents_df, ["work_id", "page_number", "sent_id"]),
    (sents_auto_df, ["work_id", "page_number", "sent_id"]),
]:
    if not df.empty:
        use_cols = [c for c in cols if c in df.columns]
        if use_cols:
            df.sort_values(use_cols, inplace=True)
        df.reset_index(drop=True, inplace=True)

final_lines_csv = FINAL_DIR / "final_lines.csv"
final_paras_csv = FINAL_DIR / "final_paragraphs.csv"
final_sents_csv = FINAL_DIR / "final_sentences_only.csv"
final_sents_auto_csv = FINAL_DIR / "final_sentences_auto.csv"

page_report_csv = AUDIT_DIR / "page_quality_report.csv"
line_filter_csv = AUDIT_DIR / "line_filter_audit.csv"
correction_csv = AUDIT_DIR / "correction_log.csv"
susp_lines_csv = AUDIT_DIR / "suspicious_lines.csv"
susp_sents_csv = AUDIT_DIR / "suspicious_sentences.csv"
dropped_pages_csv = AUDIT_DIR / "dropped_pages.csv"
dropped_lines_csv = AUDIT_DIR / "dropped_lines.csv"
source_selection_csv = AUDIT_DIR / "source_selection_report.csv"
summary_by_work_csv = AUDIT_DIR / "summary_quality_by_work_id.csv"
summary_csv = AUDIT_DIR / "run_summary.csv"
excluded_pages_range_csv = AUDIT_DIR / "excluded_pages_by_page_range.csv"
outside_final_pages_csv = AUDIT_DIR / "final_pages_outside_page_ranges.csv"

# Build audit and quarantine tables.
page_gap_rows = []
marker_missing_rows = []
missing_heading_rows = []
missing_body_rows = []
dangerous_rows = []
false_tinh_rows = []
weird_line_rows = []
long_sentence_rows = []
layout_suspect_rows = []



def _sentence_para_id_coverage_set(df: pd.DataFrame) -> set:
    """Return paragraph IDs covered by sentence rows, splitting semicolon-separated merged source IDs."""
    ids = set()
    if df is None or df.empty:
        return ids
    for col in ["paragraph_id", "source_paragraph_id"]:
        if col in df.columns:
            for val in df[col].fillna("").astype(str):
                for part in re.split(r"[;|,]", val):
                    part = part.strip()
                    if part:
                        ids.add(part)
    return ids
# Page gaps based on kept lines/sentences/paragraphs.
kept_pages_by_work = defaultdict(set)
for df in [lines_df, paras_df, sents_auto_df]:
    if not df.empty and {"work_id", "page_number"}.issubset(df.columns):
        for wid, pg in df[["work_id", "page_number"]].dropna().itertuples(index=False):
            try:
                kept_pages_by_work[str(wid)].add(int(pg))
            except Exception:
                pass

for wid, pages in kept_pages_by_work.items():
    if not pages:
        continue
    start = min(pages)
    end = max(pages)
    for missing in range(start, end + 1):
        if missing in pages:
            continue
        dp = pd.DataFrame()
        if not dropped_pages_df.empty and {"work_id", "page_number"}.issubset(dropped_pages_df.columns):
            dp = dropped_pages_df[(dropped_pages_df["work_id"].astype(str) == wid) & (pd.to_numeric(dropped_pages_df["page_number"], errors="coerce") == missing)]
        pr = pd.DataFrame()
        if not page_df.empty and {"work_id", "page_number"}.issubset(page_df.columns):
            pr = page_df[(page_df["work_id"].astype(str) == wid) & (pd.to_numeric(page_df["page_number"], errors="coerce") == missing)]
        row = {"work_id": wid, "missing_page": missing, "prev_page": missing-1, "next_page": missing+1}
        if not dp.empty:
            d0 = dp.iloc[0].to_dict()
            row.update({"drop_reason": d0.get("drop_reason", d0.get("reason", "")), "page_class": d0.get("page_class", "")})
            for c in ["char_count", "word_count", "quality_score"]:
                if c in d0: row[c] = d0.get(c)
        if not pr.empty:
            p0 = pr.iloc[0].to_dict()
            row.update({"report_page_class": p0.get("page_class", ""), "class_text_char_count": p0.get("class_text_char_count", ""), "class_text_word_count": p0.get("class_text_word_count", ""), "class_text_quality_score": p0.get("class_text_quality_score", "")})
        page_gap_rows.append(row)
page_gap_df = pd.DataFrame(page_gap_rows)
page_gap_df.to_csv(AUDIT_DIR / "page_gap_audit.csv", index=False, encoding="utf-8-sig")

# Dropped content pages and quarantine.
significant_drop_rows = []
if not dropped_pages_df.empty:
    for _, r in dropped_pages_df.iterrows():
        char_count = pd.to_numeric(pd.Series([r.get("char_count", r.get("class_text_char_count", 0))]), errors="coerce").fillna(0).iloc[0]
        word_count = pd.to_numeric(pd.Series([r.get("word_count", r.get("class_text_word_count", 0))]), errors="coerce").fillna(0).iloc[0]
        quality = pd.to_numeric(pd.Series([r.get("quality_score", r.get("class_text_quality_score", 0))]), errors="coerce").fillna(0).iloc[0]
        if char_count > 300 or word_count > 50 or quality > 40:
            significant_drop_rows.append({**r.to_dict(), "char_count_eval": char_count, "word_count_eval": word_count, "quality_score_eval": quality})
dropped_content_df = pd.DataFrame(significant_drop_rows)
dropped_content_df.to_csv(AUDIT_DIR / "dropped_content_pages.csv", index=False, encoding="utf-8-sig")

# Dropped lines that look meaningful.
suspect_dropped_lines = []
if not dropped_lines_df.empty:
    for _, r in dropped_lines_df.iterrows():
        raw = norm_text(r.get("raw_text", r.get("text", "")))
        reasons = str(r.get("reasons", ""))
        if raw and (is_heading_text(raw) or MEANINGFUL_SHORT_RE.search(raw) or DANGLING_PREV_END_RE.search(raw) or "low_line_quality" in reasons):
            suspect_dropped_lines.append(r.to_dict())
suspect_dropped_lines_df = pd.DataFrame(suspect_dropped_lines)
suspect_dropped_lines_df.to_csv(QUARANTINE_DIR / "dropped_lines_suspect.csv", index=False, encoding="utf-8-sig")

# Paragraph-to-sentence coverage.
if not paras_df.empty:
    para_ids_in_auto = _sentence_para_id_coverage_set(sents_auto_df)
    for _, r in paras_df.iterrows():
        pid = str(r.get("paragraph_id", ""))
        ptype = str(r.get("paragraph_type", ""))
        if ptype in {"heading", "micro_heading"} and pid not in para_ids_in_auto:
            missing_heading_rows.append(r.to_dict())
        elif ptype in {"body", "footnote"} and pid not in para_ids_in_auto:
            missing_body_rows.append(r.to_dict())
missing_heading_df = pd.DataFrame(missing_heading_rows)
missing_body_df = pd.DataFrame(missing_body_rows)
missing_heading_df.to_csv(AUDIT_DIR / "missing_heading_paragraphs_in_sentences.csv", index=False, encoding="utf-8-sig")
missing_body_df.to_csv(AUDIT_DIR / "missing_body_paragraphs_in_sentences.csv", index=False, encoding="utf-8-sig")

# Sentence/content audits.
if not sents_auto_df.empty:
    for _, r in sents_auto_df.iterrows():
        txt = norm_text(r.get("text", ""))
        if DANGEROUS_ROUTE_RE.search(txt):
            dangerous_rows.append(r.to_dict())
        if re.search(r"\bTỈNH\s+(?:thần|xảo|tú|thông|kỳ|khoan|quân|môn)\b", txt, re.I):
            false_tinh_rows.append(r.to_dict())
        if len(txt) > 400:
            long_sentence_rows.append({**r.to_dict(), "char_len": len(txt), "critical": len(txt) > 800})
        if "layout_order_suspect" in str(r.get("suspicious_reasons", "")):
            layout_suspect_rows.append(r.to_dict())

dangerous_df = pd.DataFrame(dangerous_rows)
false_tinh_df = pd.DataFrame(false_tinh_rows)
long_sentence_df = pd.DataFrame(long_sentence_rows)
layout_suspect_df = pd.DataFrame(layout_suspect_rows)
dangerous_df.to_csv(AUDIT_DIR / "dangerous_merge_patterns.csv", index=False, encoding="utf-8-sig")
false_tinh_df.to_csv(AUDIT_DIR / "bad_tinh_false_positives.csv", index=False, encoding="utf-8-sig")
long_sentence_df.to_csv(AUDIT_DIR / "long_sentences.csv", index=False, encoding="utf-8-sig")
layout_suspect_df.to_csv(QUARANTINE_DIR / "layout_suspect_sentences.csv", index=False, encoding="utf-8-sig")

if not lines_df.empty and "text" in lines_df.columns:
    weird_mask = lines_df["text"].astype(str).str.contains(WEIRD_OCR_CHARS_RE, regex=True, na=False)
    weird_line_df = lines_df[weird_mask].copy()
else:
    weird_line_df = pd.DataFrame()
weird_line_df.to_csv(AUDIT_DIR / "weird_char_lines.csv", index=False, encoding="utf-8-sig")

# Write quarantine text files.
with open(QUARANTINE_DIR / "dropped_pages_suspect.txt", "w", encoding="utf-8") as f:
    if dropped_content_df.empty:
        f.write("No significant dropped pages.\n")
    else:
        for _, r in dropped_content_df.iterrows():
            f.write(f"{r.get('work_id')} page {r.get('page_number')} class={r.get('page_class')} reason={r.get('drop_reason', r.get('reason',''))} chars={r.get('char_count_eval')} words={r.get('word_count_eval')} q={r.get('quality_score_eval')}\n")
with open(QUARANTINE_DIR / "low_quality_pages.txt", "w", encoding="utf-8") as f:
    if page_df.empty:
        f.write("No page quality report.\n")
    else:
        lowq = page_df[pd.to_numeric(page_df.get("selected_quality_score", page_df.get("class_text_quality_score", pd.Series(dtype=float))), errors="coerce").fillna(100) < 45]
        if lowq.empty:
            f.write("No low-quality kept/selected pages under threshold.\n")
        else:
            for _, r in lowq.iterrows():
                f.write(f"{r.get('work_id')} page {r.get('page_number')} class={r.get('page_class')} source={r.get('selected_source')} q={r.get('selected_quality_score', r.get('class_text_quality_score'))}\n")

# Write main CSVs after audit enrichment.
lines_df.to_csv(final_lines_csv, index=False, encoding="utf-8-sig")
paras_df.to_csv(final_paras_csv, index=False, encoding="utf-8-sig")
sents_df.to_csv(final_sents_csv, index=False, encoding="utf-8-sig")
sents_auto_df.to_csv(final_sents_auto_csv, index=False, encoding="utf-8-sig")
page_df.to_csv(page_report_csv, index=False, encoding="utf-8-sig")
line_audit_df.to_csv(line_filter_csv, index=False, encoding="utf-8-sig")
correction_df.to_csv(correction_csv, index=False, encoding="utf-8-sig")
susp_lines_df.to_csv(susp_lines_csv, index=False, encoding="utf-8-sig")
susp_sents_df = sents_auto_df[sents_auto_df.get("suspicious_reasons", pd.Series(dtype=str)).fillna("").astype(str).str.len() > 0].copy() if not sents_auto_df.empty else pd.DataFrame()
susp_sents_df.to_csv(susp_sents_csv, index=False, encoding="utf-8-sig")
dropped_pages_df.to_csv(dropped_pages_csv, index=False, encoding="utf-8-sig")
dropped_lines_df.to_csv(dropped_lines_csv, index=False, encoding="utf-8-sig")
source_df.to_csv(source_selection_csv, index=False, encoding="utf-8-sig")
excluded_pages_range_df.to_csv(excluded_pages_range_csv, index=False, encoding="utf-8-sig")
outside_final_pages_df.to_csv(outside_final_pages_csv, index=False, encoding="utf-8-sig")

# Per-PDF final text: emit [Page N] for every kept page from lines/paragraphs/sentences.
TEXT_DIR.mkdir(parents=True, exist_ok=True)
for old in TEXT_DIR.glob("*_final.txt"):
    try:
        old.unlink()
    except Exception:
        pass

for wid, pages in kept_pages_by_work.items():
    parts = []
    for pg in sorted(pages):
        parts.append(f"\n\n[Page {pg}]\n")
        pg_paras = paras_df[(paras_df.get("work_id", pd.Series(dtype=str)).astype(str) == wid) & (pd.to_numeric(paras_df.get("page_number", pd.Series(dtype=float)), errors="coerce") == pg)] if not paras_df.empty else pd.DataFrame()
        if not pg_paras.empty:
            for _, r in pg_paras.iterrows():
                txt = norm_text(r.get("text", ""))
                if txt:
                    parts.append(txt)
        else:
            pg_lines = lines_df[(lines_df.get("work_id", pd.Series(dtype=str)).astype(str) == wid) & (pd.to_numeric(lines_df.get("page_number", pd.Series(dtype=float)), errors="coerce") == pg)] if not lines_df.empty else pd.DataFrame()
            for _, r in pg_lines.iterrows():
                txt = norm_text(r.get("text", ""))
                if txt:
                    parts.append(txt)
    (TEXT_DIR / f"{wid}_final.txt").write_text("\n".join(x for x in parts if norm_text(x)), encoding="utf-8")

# Marker audit after writing.
for wid, pages in kept_pages_by_work.items():
    text_path = TEXT_DIR / f"{wid}_final.txt"
    markers = set()
    if text_path.exists():
        markers = {int(x) for x in re.findall(r"\[Page\s+(\d+)\]", text_path.read_text(encoding="utf-8", errors="ignore"))}
    for pg in sorted(pages - markers):
        marker_missing_rows.append({"work_id": wid, "page_number": pg, "issue": "kept_page_missing_marker"})
marker_missing_df = pd.DataFrame(marker_missing_rows)
marker_missing_df.to_csv(AUDIT_DIR / "kept_pages_missing_marker.csv", index=False, encoding="utf-8-sig")

# Summary by work_id.
if not page_df.empty:
    pages_summary = page_df.groupby("work_id", dropna=False).agg(
        total_pages=("page_number", "count"),
        kept_pages=("kept_lines", lambda x: int(pd.to_numeric(x, errors="coerce").fillna(0).gt(0).sum())),
        dropped_pages=("selected_source", lambda x: int(x.astype(str).str.contains("dropped|drop", regex=True).sum())),
        avg_selected_quality=("selected_quality_score", "mean") if "selected_quality_score" in page_df.columns else ("class_text_quality_score", "mean"),
        final_sentences=("final_sentences", "sum") if "final_sentences" in page_df.columns else ("page_number", "count"),
        dropped_lines=("dropped_lines", "sum") if "dropped_lines" in page_df.columns else ("page_number", "count"),
    ).reset_index()
else:
    pages_summary = pd.DataFrame()

if not sents_auto_df.empty:
    sent_summary = sents_auto_df.groupby("work_id", dropna=False).agg(
        final_sentence_auto_rows=("sent_id", "count"),
        heading_sentence_rows=("is_heading", lambda x: int(x.astype(str).str.lower().eq("true").sum())),
        suspicious_sentence_rows=("suspicious_reasons", lambda x: int(x.astype(str).str.len().gt(0).sum())),
        avg_sentence_quality=("quality_score", "mean"),
    ).reset_index()
    summary_by_work = pages_summary.merge(sent_summary, on="work_id", how="outer") if not pages_summary.empty else sent_summary
else:
    summary_by_work = pages_summary

if not summary_by_work.empty:
    summary_by_work["quality_score_by_work_id"] = (
        pd.to_numeric(summary_by_work.get("avg_sentence_quality", 0), errors="coerce").fillna(0).clip(0, 100) * 0.55 +
        pd.to_numeric(summary_by_work.get("avg_selected_quality", 0), errors="coerce").fillna(0).clip(0, 100) * 0.45
    ).round(2)
summary_by_work.to_csv(summary_by_work_csv, index=False, encoding="utf-8-sig")

critical_count = int(
    len(marker_missing_df) + len(missing_heading_df) + len(missing_body_df) +
    len(dangerous_df) + len(false_tinh_df) + len(outside_final_pages_df) +
    int((long_sentence_df.get("critical", pd.Series(dtype=bool)).astype(str).str.lower() == "true").sum() if not long_sentence_df.empty and "critical" in long_sentence_df.columns else 0) +
    int((dropped_content_df.get("char_count_eval", pd.Series(dtype=float)) > 500).sum() if not dropped_content_df.empty and "char_count_eval" in dropped_content_df.columns else 0)
)

run_summary = pd.DataFrame([{
    "pdf_count": len(pdf_paths),
    "total_pages": int(len(page_df)) if not page_df.empty else processed_pages_total,
    "kept_pages": int(sum(len(v) for v in kept_pages_by_work.values())),
    "dropped_pages": int(len(dropped_pages_df)),
    "excluded_pages_by_page_range": int(len(excluded_pages_range_df)),
    "final_rows_outside_page_ranges": int(len(outside_final_pages_df)),
    "total_lines": int(len(lines_df)),
    "dropped_lines": int(len(dropped_lines_df)),
    "final_paragraphs": int(len(paras_df)),
    "final_sentences_only": int(len(sents_df)),
    "final_sentences_auto": int(len(sents_auto_df)),
    "heading_sentence_rows": int(sents_auto_df.get("is_heading", pd.Series(dtype=str)).astype(str).str.lower().eq("true").sum()) if not sents_auto_df.empty else 0,
    "suspicious_sentences": int(len(susp_sents_df)),
    "page_gap_rows": int(len(page_gap_df)),
    "dropped_content_pages": int(len(dropped_content_df)),
    "kept_pages_missing_marker": int(len(marker_missing_df)),
    "missing_heading_paragraphs_in_sentence_auto": int(len(missing_heading_df)),
    "missing_body_paragraphs_in_sentence_auto": int(len(missing_body_df)),
    "dangerous_merge_patterns": int(len(dangerous_df)),
    "bad_tinh_false_positives": int(len(false_tinh_df)),
    "weird_char_lines": int(len(weird_line_df)),
    "long_sentences": int(len(long_sentence_df)),
    "critical_issues": critical_count,
    "output_zip_path": str(PKG_DIR / "dntc_auto_output.zip"),
    "pipeline_version": PIPELINE_VERSION,
}])
run_summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")

# Zip output.
zip_path = PKG_DIR / "dntc_auto_output.zip"
if ZIP_OUTPUT:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root in [FINAL_DIR, AUDIT_DIR, QUARANTINE_DIR]:
            for f in root.rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to(OUTPUT_DIR))

print("\n=== DNTC v11 export summary ===")
print(run_summary.to_string(index=False))
print("\nOutput zip:", zip_path)
print("Quarantine dir:", QUARANTINE_DIR)
if critical_count:
    print("\nINTERMEDIATE ISSUES BEFORE FINAL V12/V13 ACCEPTANCE")
    for name, df in [
        ("kept_pages_missing_marker", marker_missing_df),
        ("missing_heading_paragraphs", missing_heading_df),
        ("missing_body_paragraphs", missing_body_df),
        ("dangerous_merge_patterns", dangerous_df),
        ("bad_tinh_false_positives", false_tinh_df),
        ("dropped_content_pages", dropped_content_df),
        ("critical_long_sentences", long_sentence_df[long_sentence_df.get("critical", pd.Series(dtype=bool)).astype(str).str.lower() == "true"] if not long_sentence_df.empty and "critical" in long_sentence_df.columns else pd.DataFrame()),
    ]:
        if not df.empty:
            print(f"- {name}: {len(df)}")
            cols = [c for c in ["work_id", "page_number", "paragraph_id", "sent_id", "text", "drop_reason", "issue"] if c in df.columns]
            print(df[cols].head(8).to_string(index=False))
else:
    print("\nINTERMEDIATE AUDIT PASSED")


In [ ]:

# ============================================================
# 10B. DNTC v12 mixed-script audit + quarantine + package refresh
# ============================================================

def _df_text_contains(df, pattern, flags=0):
    if df.empty or "text" not in df.columns:
        return pd.DataFrame()
    mask = df["text"].fillna("").astype(str).str.contains(pattern, regex=True, flags=flags, na=False)
    return df[mask].copy()

# Refresh dataframes from the just-written/final in-memory objects.
for df in [lines_df, paras_df, sents_df, sents_auto_df, susp_sents_df]:
    if not df.empty and "text" in df.columns:
        df["text"] = df["text"].astype(str).map(lambda x: clean_mixed_script_text(x, None, context="post_export_clean")[0]).map(lambda x: _repair_tinh_uppercase_context(x)).map(norm_text)

# Re-write final CSVs after final mixed-script cleanup.
lines_df.to_csv(final_lines_csv, index=False, encoding="utf-8-sig")
paras_df.to_csv(final_paras_csv, index=False, encoding="utf-8-sig")
sents_df.to_csv(final_sents_csv, index=False, encoding="utf-8-sig")
sents_auto_df.to_csv(final_sents_auto_csv, index=False, encoding="utf-8-sig")

# Mixed-script audits required by v12.
garbage_patterns = r"(?i)(?:\bTỈNH\s+bik\b|\bbik\b|\bx\s+wt\b|\bkt\s+ah\b|\bor\s*\+|\*\s*\*|BILIARY\s+LICYRe)"
false_tinh_pattern = r"(?i)\bTỈNH\s+(?:thần|xảo|tú|thông|kỳ|khoan|huyết|bồ|bik|quân|môn)\b"
dangling_cjk_pattern = rf"[A-Za-zÀ-ỹĐđ][^.!?]{{0,45}}\s[{CJK_CLASS}](?:[,.;:]|$)"
incomplete_label_pattern = rf"(?i)\b(?:có\s+\d+\s+cửa|đài|lầu|môn|kiều|viện|cung|trì|phường|th[uủ]y[-\s]?quan|nam[-\s]?khuyết[-\s]?đài)[^.!?]{{0,45}}\s[{CJK_CLASS}](?:[,.;:]|$)"
merged_entry_pattern = rf"(?i)(?:.+\s)(?:[A-ZÀ-ỸĐa-zà-ỹđ][A-Za-zÀ-ỹĐđ'\-]*\s+){{0,7}}(?:th[uủ]y[-\s]?quan|kiều|cầu|môn|thành|điện|đài|viện|cung|trì|phường|lầu|miếu|đình|đền)\s+[{CJK_CLASS}]{{2,14}}\s*[:：]"

mixed_audit_sources = []
for name, df in [("final_lines", lines_df), ("final_paragraphs", paras_df), ("final_sentences_auto", sents_auto_df)]:
    if not df.empty and "text" in df.columns:
        tmp = df.copy(); tmp["source_table"] = name; mixed_audit_sources.append(tmp)
all_text_df = pd.concat(mixed_audit_sources, ignore_index=True) if mixed_audit_sources else pd.DataFrame()

garbage_mixed_df = _df_text_contains(all_text_df, garbage_patterns, flags=re.I)
false_tinh_upper_df = _df_text_contains(all_text_df, false_tinh_pattern, flags=re.I)
dangling_cjk_df = _df_text_contains(sents_auto_df, dangling_cjk_pattern, flags=0)
incomplete_cjk_df = _df_text_contains(sents_auto_df, incomplete_label_pattern, flags=re.I)
merged_entry_df = _df_text_contains(sents_auto_df, merged_entry_pattern, flags=re.I)

# Missing heading/micro-heading sentence coverage.
if not paras_df.empty:
    para_ids = set(sents_auto_df.get("source_paragraph_id", pd.Series(dtype=str)).astype(str)) | set(sents_auto_df.get("paragraph_id", pd.Series(dtype=str)).astype(str)) if not sents_auto_df.empty else set()
    heading_mask = paras_df.get("paragraph_type", pd.Series(dtype=str)).astype(str).isin(["heading", "micro_heading"])
    missing_heading_sentence_v12_df = paras_df[heading_mask & ~paras_df.get("paragraph_id", pd.Series(dtype=str)).astype(str).isin(para_ids)].copy()
else:
    missing_heading_sentence_v12_df = pd.DataFrame()

# Quarantine records from mixed-script cleaner.
mixed_quarantine_df = pd.DataFrame(MIXED_SCRIPT_QUARANTINE_RECORDS)
if not mixed_quarantine_df.empty:
    mixed_quarantine_df = mixed_quarantine_df.drop_duplicates()

# Add prev/next context for dropped lines if possible.
if not dropped_lines_df.empty:
    dl = dropped_lines_df.copy()
    if {"work_id", "page_number"}.issubset(dl.columns):
        dl = dl.sort_values(["work_id", "page_number"] + (["line_id"] if "line_id" in dl.columns else []))
        if "text" in dl.columns:
            dl["current_text"] = dl["text"].astype(str)
            dl["prev_text"] = dl.groupby(["work_id", "page_number"])["current_text"].shift(1).fillna("")
            dl["next_text"] = dl.groupby(["work_id", "page_number"])["current_text"].shift(-1).fillna("")
            suspect_mask = dl.get("reasons", pd.Series(dtype=str)).astype(str).str.contains("mixed_script_ocr_garbage_fragment|rescued|low_line_quality|too_short", regex=True, na=False)
            dropped_lines_suspect_v12_df = dl[suspect_mask].copy()
        else:
            dropped_lines_suspect_v12_df = dl.copy()
else:
    dropped_lines_suspect_v12_df = pd.DataFrame()

# Write v12 audit/quarantine files.
garbage_mixed_df.to_csv(AUDIT_DIR / "garbage_mixed_script_fragments.csv", index=False, encoding="utf-8-sig")
false_tinh_upper_df.to_csv(AUDIT_DIR / "false_tinh_uppercase.csv", index=False, encoding="utf-8-sig")
dangling_cjk_df.to_csv(AUDIT_DIR / "dangling_cjk_fragment.csv", index=False, encoding="utf-8-sig")
incomplete_cjk_df.to_csv(AUDIT_DIR / "incomplete_cjk_after_label.csv", index=False, encoding="utf-8-sig")
merged_entry_df.to_csv(AUDIT_DIR / "merged_entry_label.csv", index=False, encoding="utf-8-sig")
missing_heading_sentence_v12_df.to_csv(AUDIT_DIR / "missing_heading_sentence_v12.csv", index=False, encoding="utf-8-sig")
mixed_quarantine_df.to_csv(QUARANTINE_DIR / "mixed_script_ocr_garbage_fragments.csv", index=False, encoding="utf-8-sig")
dropped_lines_suspect_v12_df.to_csv(QUARANTINE_DIR / "dropped_lines_suspect.csv", index=False, encoding="utf-8-sig")

# Append v12 counts into run_summary and refresh zip.
try:
    rs = pd.read_csv(summary_csv)
except Exception:
    rs = pd.DataFrame([{}])
for col, val in {
    "garbage_mixed_script_fragments": len(garbage_mixed_df),
    "false_tinh_uppercase": len(false_tinh_upper_df),
    "dangling_cjk_fragment": len(dangling_cjk_df),
    "incomplete_cjk_after_label": len(incomplete_cjk_df),
    "merged_entry_label": len(merged_entry_df),
    "missing_heading_sentence_v12": len(missing_heading_sentence_v12_df),
    "mixed_script_quarantine_records": len(mixed_quarantine_df),
    "pipeline_version": PIPELINE_VERSION,
}.items():
    rs[col] = val
# Critical if clean final still contains garbage/false TINH or missing heading rows.
rs["critical_issues_v12"] = int(len(garbage_mixed_df) + len(false_tinh_upper_df) + len(missing_heading_sentence_v12_df))
rs.to_csv(summary_csv, index=False, encoding="utf-8-sig")

# Refresh package so new audits are inside output zip.
zip_path = PKG_DIR / "dntc_auto_output.zip"
if ZIP_OUTPUT:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root in [FINAL_DIR, AUDIT_DIR, QUARANTINE_DIR]:
            for f in root.rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to(OUTPUT_DIR))

print("\n=== DNTC v12 mixed-script audit ===")
print(pd.DataFrame([{
    "garbage_mixed_script_fragments": len(garbage_mixed_df),
    "false_tinh_uppercase": len(false_tinh_upper_df),
    "dangling_cjk_fragment": len(dangling_cjk_df),
    "incomplete_cjk_after_label": len(incomplete_cjk_df),
    "merged_entry_label": len(merged_entry_df),
    "missing_heading_sentence_v12": len(missing_heading_sentence_v12_df),
    "mixed_script_quarantine_records": len(mixed_quarantine_df),
}]).to_string(index=False))
if int(rs["critical_issues_v12"].iloc[0]) > 0:
    print("CRITICAL ISSUES REMAIN")
else:
    print("AUDIT PASSED")
print("Output zip refreshed:", zip_path)

# DNTC v13 additions: persist context-based OCR fragment quarantine and layout suspects.
context_dropped_df = pd.DataFrame(globals().get("CONTEXT_DROPPED_LINE_RECORDS", []))
if not context_dropped_df.empty:
    context_dropped_df.to_csv(QUARANTINE_DIR / "dropped_lines_suspect_context_fragments.csv", index=False, encoding="utf-8-sig")
layout_suspect_df_v13 = pd.DataFrame(globals().get("LAYOUT_SUSPECT_RECORDS", []))
if not layout_suspect_df_v13.empty:
    layout_suspect_df_v13.to_csv(QUARANTINE_DIR / "layout_suspect_lines.csv", index=False, encoding="utf-8-sig")




# v15.1: refresh package one more time after v13 context quarantine files are written.
zip_path = PKG_DIR / "dntc_auto_output.zip"
if ZIP_OUTPUT:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root in [FINAL_DIR, AUDIT_DIR, QUARANTINE_DIR]:
            for f in root.rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to(OUTPUT_DIR))
    print("Output zip refreshed after v13 quarantine:", zip_path)


In [ ]:

# ============================================================
# 11. DNTC v13 acceptance tests and audit summary
# ============================================================
EXPECTED_PAGE_RANGES = {
    "01.pdf": [(21, 127)],
    "05.pdf": [(15, 140)],
    "07_08.pdf": [(23, 90), (95, 206)],
    "09.pdf": [(23, 139)],
    "10_11.pdf": [(13, 129)],
    "12.pdf": [(9, 112)],
    "13.pdf": [(7, 120)],
    "14_15.pdf": [(7, 168)],
    "16_17.pdf": [(14, 128), (138, 293)],
    "q2_3_4.pdf": [(13, 499)],
    "q6.pdf": [(3, 525)],
}

def _contains_pat(df, pattern, flags=0):
    return False if df.empty or "text" not in df.columns else bool(df["text"].astype(str).str.contains(pattern, case=False, regex=True, flags=flags, na=False).any())

def _rows_with_pat(df, pattern, flags=0):
    if df.empty or "text" not in df.columns:
        return pd.DataFrame()
    return df[df["text"].astype(str).str.contains(pattern, case=False, regex=True, flags=flags, na=False)].copy()

def _all_final_rows_in_ranges(df):
    if df.empty or "page_number" not in df.columns:
        return True
    return bool(df.apply(lambda r: row_in_page_ranges(r), axis=1).all())

def _text_markers_for_work(wid):
    p = TEXT_DIR / f"{wid}_final.txt"
    if not p.exists():
        return set()
    return {int(x) for x in re.findall(r"\[Page\s+(\d+)\]", p.read_text(encoding="utf-8", errors="ignore"))}

# Recompute strict acceptance audit tables after all v12/v13 cleanup cells.
final_outside_rows = []
for name, df in [("final_lines", lines_df), ("final_paragraphs", paras_df), ("final_sentences_auto", sents_auto_df), ("final_sentences_only", sents_df)]:
    if not df.empty and "page_number" in df.columns:
        bad = df[~df.apply(lambda r: row_in_page_ranges(r), axis=1)].copy()
        if not bad.empty:
            bad["source_table"] = name
            final_outside_rows.append(bad)
final_outside_pages_df2 = pd.concat(final_outside_rows, ignore_index=True) if final_outside_rows else pd.DataFrame()
final_outside_pages_df2.to_csv(AUDIT_DIR / "acceptance_final_pages_outside_page_ranges.csv", index=False, encoding="utf-8-sig")

# kept pages marker check.
kept_pages_by_work2 = defaultdict(set)
for df in [lines_df, paras_df, sents_auto_df]:
    if not df.empty and {"work_id", "page_number"}.issubset(df.columns):
        for wid, pg in df[["work_id", "page_number"]].dropna().itertuples(index=False):
            try:
                kept_pages_by_work2[str(wid)].add(int(pg))
            except Exception:
                pass
marker_missing2 = []
for wid, pages in kept_pages_by_work2.items():
    markers = _text_markers_for_work(wid)
    for pg in sorted(pages - markers):
        marker_missing2.append({"work_id": wid, "page_number": pg, "issue": "kept_page_missing_marker"})
marker_missing2_df = pd.DataFrame(marker_missing2)
marker_missing2_df.to_csv(AUDIT_DIR / "acceptance_kept_pages_missing_marker.csv", index=False, encoding="utf-8-sig")



def _sentence_para_id_coverage_set_v15(df: pd.DataFrame) -> set:
    ids = set()
    if df is None or df.empty:
        return ids
    for col in ["paragraph_id", "source_paragraph_id"]:
        if col in df.columns:
            for val in df[col].fillna("").astype(str):
                for part in re.split(r"[;|,]", val):
                    part = part.strip()
                    if part:
                        ids.add(part)
    return ids
# Heading/micro-heading coverage.
if not paras_df.empty:
    heading_mask2 = paras_df.get("paragraph_type", pd.Series(dtype=str)).astype(str).isin(["heading", "micro_heading"])
    sent_para_ids2 = _sentence_para_id_coverage_set_v15(sents_auto_df)
    missing_heading2_df = paras_df[heading_mask2 & ~paras_df.get("paragraph_id", pd.Series(dtype=str)).astype(str).isin(sent_para_ids2)].copy()
    body_mask2 = paras_df.get("paragraph_type", pd.Series(dtype=str)).astype(str).isin(["body", "footnote"])
    missing_body2_df = paras_df[body_mask2 & ~paras_df.get("paragraph_id", pd.Series(dtype=str)).astype(str).isin(sent_para_ids2)].copy()
else:
    missing_heading2_df = pd.DataFrame(); missing_body2_df = pd.DataFrame()
missing_heading2_df.to_csv(AUDIT_DIR / "acceptance_missing_heading_sentences.csv", index=False, encoding="utf-8-sig")
missing_body2_df.to_csv(AUDIT_DIR / "acceptance_missing_body_footnote_sentences.csv", index=False, encoding="utf-8-sig")

# Pattern audits.
bad_tinh2_df = _rows_with_pat(sents_auto_df, r"\bTỈNH\s+(thần|xảo|tú|thông|kỳ|khoan|huyết|bồ|bik)\b")
danger_route2_df = _rows_with_pat(sents_auto_df, r"đến\s+trạm\s+(Ở|TRẠM)")
short_frag2_df = _rows_with_pat(sents_auto_df, r"(?i)(?:\bTỈNH\s+bik\b|\bbik\b|\bx\s+wt\b|\bkt\s+ah\b|\bor\s*\+|\*\s*\*|BILIARY\s+LICYRe)")
long_critical2_df = sents_auto_df[pd.to_numeric(sents_auto_df.get("char_count", sents_auto_df.get("text", pd.Series(dtype=str)).astype(str).str.len()), errors="coerce").fillna(sents_auto_df.get("text", pd.Series(dtype=str)).astype(str).str.len()) > 800].copy() if not sents_auto_df.empty else pd.DataFrame()
# allow long only if explicitly flagged
if not long_critical2_df.empty and "suspicious_reasons" in long_critical2_df.columns:
    long_critical_unflagged2_df = long_critical2_df[~long_critical2_df["suspicious_reasons"].astype(str).str.contains("long_sentence", na=False)].copy()
else:
    long_critical_unflagged2_df = long_critical2_df
multi_heading2_df = _rows_with_pat(sents_auto_df, rf"(?i)(?:huyện\s+HUYỆN|phủ\s+PHỦ|hiện\s+kim\.\s+[A-ZÀ-ỸĐ][A-ZÀ-ỸĐ\s\-]+|(?:{ENTRY_KEYWORD_PATTERN_V13})\s*(?:[{CJK_CLASS}]{{1,16}})?\s*[:：].+(?:{ENTRY_KEYWORD_PATTERN_V13})\s*(?:[{CJK_CLASS}]{{1,16}})?\s*[:：])")

for name, df in [
    ("acceptance_bad_tinh_phrases.csv", bad_tinh2_df),
    ("acceptance_danger_route_merge.csv", danger_route2_df),
    ("acceptance_short_ocr_fragments.csv", short_frag2_df),
    ("acceptance_long_critical_unflagged.csv", long_critical_unflagged2_df),
    ("acceptance_multi_heading_entry_merge.csv", multi_heading2_df),
]:
    df.to_csv(AUDIT_DIR / name, index=False, encoding="utf-8-sig")

# Required excluded page audit exists and covers pages outside ranges that were processed.
excluded_exists = (AUDIT_DIR / "excluded_pages_by_page_range.csv").exists()
noise_quarantine_exists = (QUARANTINE_DIR / "dropped_lines_suspect.csv").exists() or (QUARANTINE_DIR / "dropped_lines_suspect_context_fragments.csv").exists()

checks = []
checks.append(("PAGE_RANGES_exact", PAGE_RANGES == EXPECTED_PAGE_RANGES))
checks.append(("no_final_pages_outside_PAGE_RANGES", final_outside_pages_df2.empty))
checks.append(("all_kept_pages_have_marker", marker_missing2_df.empty))
checks.append(("no_missing_heading_or_micro_heading_sentence", missing_heading2_df.empty))
checks.append(("no_silent_body_footnote_loss", missing_body2_df.empty))
checks.append(("no_den_tram_O_or_TRAM", danger_route2_df.empty))
checks.append(("no_bad_TINH_phrases", bad_tinh2_df.empty))
checks.append(("no_isolated_ocr_fragment_patterns", short_frag2_df.empty))
checks.append(("no_unflagged_sentence_over_800_chars", long_critical_unflagged2_df.empty))
checks.append(("no_multiple_heading_entry_labels_merged", multi_heading2_df.empty))
checks.append(("excluded_pages_by_page_range_audit_exists", excluded_exists))
checks.append(("dropped_noise_lines_have_quarantine", noise_quarantine_exists))

acceptance_df = pd.DataFrame([{"check": name, "passed": bool(ok)} for name, ok in checks])
acceptance_df.to_csv(AUDIT_DIR / "acceptance_tests.csv", index=False, encoding="utf-8-sig")
print("\n=== DNTC v13 acceptance tests ===")
print(acceptance_df.to_string(index=False))

failed = acceptance_df[~acceptance_df["passed"]]
if not failed.empty:
    print("\nCRITICAL ISSUES REMAIN")
    for name, df in [
        ("final_pages_outside_PAGE_RANGES", final_outside_pages_df2),
        ("kept_pages_missing_marker", marker_missing2_df),
        ("missing_heading_sentences", missing_heading2_df),
        ("missing_body_footnote_sentences", missing_body2_df),
        ("danger_route_merge", danger_route2_df),
        ("bad_TINH_phrases", bad_tinh2_df),
        ("short_ocr_fragments", short_frag2_df),
        ("long_critical_unflagged", long_critical_unflagged2_df),
        ("multi_heading_entry_merge", multi_heading2_df),
    ]:
        if not df.empty:
            print(f"- {name}: {len(df)}")
            cols = [c for c in ["work_id", "page_number", "paragraph_id", "source_paragraph_id", "sent_id", "reason", "suspicious_reasons", "text", "current_text"] if c in df.columns]
            print(df[cols].head(10).to_string(index=False))
else:
    print("\nAUDIT PASSED")


In [ ]:

# ============================================================
# 11B. DNTC v16 R016 acceptance tests + final package refresh
# ============================================================
# These tests are specifically tied to spelling_merge_review_all.csv and R016 YAML.

import re
from pathlib import Path

# Use final in-memory dataframes after all export/mixed-script cleanup cells.
def _r016_rows(df, pattern, flags=re.I):
    if df is None or df.empty or "text" not in df.columns:
        return pd.DataFrame()
    return df[df["text"].astype(str).str.contains(pattern, regex=True, flags=flags, na=False)].copy()

r016_acceptance_tables = {}
all_final_text_sources = []
for name, df in [("final_lines", lines_df), ("final_paragraphs", paras_df), ("final_sentences_auto", sents_auto_df), ("final_sentences_only", sents_df)]:
    if df is not None and not df.empty and "text" in df.columns:
        tmp = df.copy(); tmp["source_table"] = name; all_final_text_sources.append(tmp)
all_final_text_df = pd.concat(all_final_text_sources, ignore_index=True) if all_final_text_sources else pd.DataFrame()

r016_patterns = {
    "r016_remaining_kinh_su": r"\bKINH\s*[- ]?SU\s*['’]?\b|\bKINH\s+SU\b",
    "r016_remaining_false_tinh": r"\bTỈNH\s+(?:thần|xảo|tú|thông|kỳ|khoan|huyết|bồ|bik|quân|môn|hoa|tiết|điền)\b",
    "r016_remaining_noise_fragment": r"(?i)(?:\bbik\b|\bx\s+wt\b|\bkt\s+ah\b|\bor\s*\+|\*\s*\*|BILIARY\s+LICYRe|\bWSR\b|T\s*/\s*\^T)",
    "r016_remaining_false_period_compound": r"\b(?:Phước|Tân|Gia|Minh|Đông|Ngọc|Thế|Thiệu|Tự|Thành|Đồng)\.\s+[A-ZÀ-ỸĐa-zà-ỹđ]",
    "r016_remaining_de_purpose": rf"\bđề\s+(?:{R016_PURPOSE_VERBS})\b" if "R016_PURPOSE_VERBS" in globals() else r"\bđề\s+(?:làm|cho|tiện)\b",
    "r016_remaining_doi_change": rf"\bđồi\s+(?:{R016_CHANGE_VERBS})\b" if "R016_CHANGE_VERBS" in globals() else r"\bđồi\s+(?:tên|làm|sang)\b",
    "r016_remaining_missing_space_before_number": r"\b(?:thứ|năm|ngày|tháng)\d+\b",
}
for issue, pat in r016_patterns.items():
    df = _r016_rows(all_final_text_df, pat, flags=re.I)
    r016_acceptance_tables[issue] = df
    df.to_csv(AUDIT_DIR / f"acceptance_{issue}.csv", index=False, encoding="utf-8-sig")

# One-token sentence audit: allowed only when heading or meaningful numeric/measurement.
if sents_auto_df is not None and not sents_auto_df.empty and "text" in sents_auto_df.columns:
    tmp = sents_auto_df.copy()
    tmp["_tok_count"] = tmp["text"].astype(str).map(lambda x: len(r016_lexical_tokens(x)) if "r016_lexical_tokens" in globals() else len(str(x).split()))
    mask = (tmp["_tok_count"] <= 1) & (~tmp.get("is_heading", pd.Series(False, index=tmp.index)).astype(bool))
    if "R016_SINGLE_TOKEN_ALLOWED_RE" in globals():
        mask &= ~tmp["text"].astype(str).map(lambda x: bool(R016_SINGLE_TOKEN_ALLOWED_RE.match(str(x).strip())))
    one_token_df = tmp[mask].copy()
else:
    one_token_df = pd.DataFrame()
one_token_df.to_csv(AUDIT_DIR / "acceptance_r016_one_token_nonheading_sentences.csv", index=False, encoding="utf-8-sig")
r016_acceptance_tables["r016_one_token_nonheading_sentences"] = one_token_df

# Ensure quarantined noise exists if anything was removed.
r016_quarantine_path = QUARANTINE_DIR / "r016_quarantined_noise_fragments.csv"
if "R016_QUARANTINE_RECORDS" in globals():
    pd.DataFrame(R016_QUARANTINE_RECORDS).to_csv(r016_quarantine_path, index=False, encoding="utf-8-sig")

r016_critical = []
for issue, df in r016_acceptance_tables.items():
    if df is not None and not df.empty:
        severity = "critical" if issue in {
            "r016_remaining_noise_fragment",
            "r016_remaining_false_tinh",
            "r016_remaining_false_period_compound",
            "r016_one_token_nonheading_sentences",
        } else "high"
        r016_critical.append({"issue": issue, "severity": severity, "count": int(len(df))})

r016_acceptance_df = pd.DataFrame(r016_critical)
r016_acceptance_df.to_csv(AUDIT_DIR / "r016_acceptance_summary.csv", index=False, encoding="utf-8-sig")

# Refresh zip one final time so R016 audits/quarantine are included.
try:
    if ZIP_OUTPUT:
        import zipfile
        zip_path = PKG_DIR / "dntc_auto_output.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for root in [FINAL_DIR, AUDIT_DIR, QUARANTINE_DIR]:
                if root.exists():
                    for p in root.rglob("*"):
                        if p.is_file():
                            zf.write(p, p.relative_to(OUTPUT_DIR))
        print("R016 refreshed zip:", zip_path)
except Exception as e:
    print("R016 zip refresh warning:", e)

if r016_critical:
    print("R016 CRITICAL/HIGH ISSUES REMAIN")
    print(r016_acceptance_df.head(30).to_string(index=False))
else:
    print("R016 AUDIT PASSED")
